In [ ]:
import torch, torch_geometric, torch_sparse, torch_scatter, torch_cluster, pyg_lib

import os, time, math, copy, random, warnings, logging, json
from collections import defaultdict, Counter
from typing import Dict, List, Set, Tuple, Optional, Any

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import (
    MessagePassing, global_mean_pool, global_add_pool,
    GCNConv, GINConv, SAGEConv, GATConv,
)
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader

from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score, roc_curve,
)
from scipy.stats import spearmanr, chi2_contingency, wilcoxon

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

class InputGraph:
    """Graph representation for the explanation framework.

    Wraps a node feature matrix and edge index, and maintains the
    bookkeeping needed by the cooperative-game explanation:
      - which nodes are *players* (participants in the game)
      - which non-player nodes each player "owns" (masking a player
        also masks its owned nodes)
      - adjacency among players (for connectedness constraints)
      - a baseline feature vector (features assigned to masked nodes)

    For most graph datasets (including synthetic benchmarks), every
    node is a player and there are no owned non-player nodes.  The
    player/owned distinction supports molecular graphs where hydrogen
    atoms are non-player nodes owned by their bonded heavy atom.

    Parameters
    ----------
    x : torch.Tensor, shape (n_total, feat_dim)
        Node feature matrix for the full graph.
    edge_index : torch.Tensor, shape (2, 2|E|)
        Directed edge index (each undirected edge appears twice).
    player_idx : list of int, optional
        Indices of player nodes.  If None, all nodes are players.
    owned_nodes : dict[int, list[int]], optional
        Mapping from player index to list of non-player indices it owns.
        If None, no nodes are owned.
    baseline : torch.Tensor, optional
        Feature vector used for masked nodes.  If None, uses zeros.
    node_labels : list of str, optional
        Human-readable label for each player node.
    """

    def __init__(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        player_idx: Optional[List[int]] = None,
        owned_nodes: Optional[Dict[int, List[int]]] = None,
        baseline: Optional[torch.Tensor] = None,
        node_labels: Optional[List[str]] = None,
    ):
        self.x = x.clone()
        self.edge_index = edge_index.clone()
        self.n_total = x.shape[0]
        self.feat_dim = x.shape[1]

        if player_idx is not None:
            self.player_idx: List[int] = list(player_idx)
        else:
            self.player_idx = list(range(self.n_total))
        self.n_players = len(self.player_idx)

        if owned_nodes is not None:
            self._owned: Dict[int, List[int]] = dict(owned_nodes)
        else:
            self._owned = defaultdict(list)

        player_set = set(self.player_idx)
        self.non_player_idx: List[int] = [
            i for i in range(self.n_total) if i not in player_set
        ]

        if baseline is not None:
            self.baseline = baseline.clone()
        else:
            self.baseline = torch.zeros(self.feat_dim, dtype=torch.float32)

        self.player_adj: Dict[int, Set[int]] = defaultdict(set)
        ei = edge_index.numpy()
        for k in range(ei.shape[1]):
            a, b = int(ei[0, k]), int(ei[1, k])
            if a in player_set and b in player_set:
                self.player_adj[a].add(b)

        self._node_labels = node_labels

    # --- Backward-compatible aliases ---
    @property
    def heavy_idx(self) -> List[int]:
        return self.player_idx

    @property
    def n_heavy(self) -> int:
        return self.n_players

    @property
    def heavy_adj(self) -> Dict[int, Set[int]]:
        return self.player_adj

    @property
    def heavy_to_h(self) -> Dict[int, List[int]]:
        return self._owned

    @property
    def h_idx(self) -> List[int]:
        return self.non_player_idx

    # --- Convenience methods ---

    def pyg_data(self) -> Data:
        return Data(x=self.x.clone(), edge_index=self.edge_index.clone())

    def node_label(self, idx: int) -> str:
        if self._node_labels is not None:
            pos = self.player_idx.index(idx) if idx in self.player_idx else -1
            if 0 <= pos < len(self._node_labels):
                return self._node_labels[pos]
        return f"v{idx}"

    def node_labels(self) -> List[str]:
        return [f"{i}:{self.node_label(i)}" for i in self.player_idx]

    def heavy_atom_symbol(self, idx: int) -> str:
        return self.node_label(idx)

    def heavy_atom_labels(self) -> List[str]:
        return self.node_labels()

    @classmethod
    def from_pyg_data(
        cls,
        data: Data,
        player_idx: Optional[List[int]] = None,
        baseline: Optional[torch.Tensor] = None,
    ) -> "InputGraph":
        return cls(
            x=data.x,
            edge_index=data.edge_index,
            player_idx=player_idx,
            baseline=baseline,
        )


In [ ]:
###########################################################################
# S2  MODEL DEFINITION
###########################################################################

class MPNN(MessagePassing):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__(aggr='add')
        self.lin = nn.Linear(in_channels, out_channels)
        self.lin_update = nn.Linear(out_channels * 2, out_channels)

    def forward(self, x, edge_index):
        m = self.propagate(edge_index, x=x)
        x = self.lin_update(torch.cat([self.lin(x), m], dim=1))
        return F.leaky_relu(x, negative_slope=0.1)

    def message(self, x_j):
        return self.lin(x_j)


class DumplingGNN(nn.Module):
    """MPNN -> 3xGAT -> GraphSAGE -> global mean pool -> single logit."""

    def __init__(self, input_dim: int = 8, hidden_channels: int = 64,
                 dropout: float = 0.1):
        super().__init__()
        self.mpnn = MPNN(input_dim, hidden_channels)
        self.gat1 = GATConv(hidden_channels, hidden_channels,
                            heads=8, concat=True, dropout=dropout)
        self.gat2 = GATConv(hidden_channels * 8, hidden_channels,
                            heads=8, concat=True, dropout=dropout)
        self.gat3 = GATConv(hidden_channels * 8, hidden_channels,
                            heads=8, concat=True, dropout=dropout)
        self.sage = SAGEConv(hidden_channels * 8, hidden_channels)
        self.out = nn.Linear(hidden_channels, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, data: Data) -> torch.Tensor:
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = self.mpnn(x, edge_index); x = F.relu(x); x = self.dropout(x)
        x = self.gat1(x, edge_index); x = F.elu(x); x = self.dropout(x)
        x = self.gat2(x, edge_index); x = F.leaky_relu(x); x = self.dropout(x)
        x = self.gat3(x, edge_index); x = F.elu(x)
        x = self.sage(x, edge_index); x = F.relu(x)
        x = global_mean_pool(x, batch)
        return self.out(x).squeeze(-1)

    @torch.no_grad()
    def predict_proba(self, data: Data) -> torch.Tensor:
        self.eval()
        return torch.sigmoid(self.forward(data))


###########################################################################
# S3  VALUE FUNCTION  (masked-node evaluation)
###########################################################################

def evaluate_coalition(
    model: nn.Module,
    graph,            # InputGraph or any object with the required attributes
    active_players: Set[int],
    device: torch.device,
) -> float:
    """Compute f_GNN(S) -- the raw logit when only player nodes in
    *active_players* (and their owned non-player nodes) retain original
    features, and everything else is masked to baseline.

    This is the characteristic function w(S) = f_GNN(S) from the paper
    (Eq. 7), before subtracting f_GNN(emptyset).

    The graph topology (edges) is always preserved; only node features
    are replaced.
    """
    active_total: Set[int] = set()
    for p in active_players:
        active_total.add(p)
        active_total.update(graph.heavy_to_h.get(p, []))

    x_masked = graph.x.clone()
    for i in range(graph.n_total):
        if i not in active_total:
            x_masked[i] = graph.baseline

    data = Data(x=x_masked, edge_index=graph.edge_index)
    batch = Batch.from_data_list([data]).to(device)

    with torch.no_grad():
        model.eval()
        z = model(batch)
        # Handle varying output shapes:
        #   scalar or [1]       → single logit (e.g. DumplingGNN)
        #   [1, 1]              → single logit wrapped in 2D
        #   [1, C] where C >= 2 → multi-class logits
        if z.dim() == 0:
            return float(z.item())
        if z.dim() == 1:
            if z.shape[0] == 1:
                return float(z[0].item())
            # Batch size > 1 shouldn't happen here; take first
            return float(z[0].item())
        # z.dim() >= 2: shape [batch, C]
        if z.shape[-1] == 1:
            return float(z[0, 0].item())
        # Multi-class: return positive-class logit (index 1) for binary,
        # or max logit for multi-class (preserves ranking)
        if z.shape[-1] == 2:
            return float(z[0, 1].item())
        return float(z[0].max().item())


class ValueFunction:
    """Caching wrapper around evaluate_coalition.

    w(T) = f_GNN(T) - f_GNN(emptyset)          (Eq. 7)

    The cache stores raw logit f_GNN(T); callers subtract f_GNN(emptyset).
    """

    def __init__(self, model: nn.Module, graph, device: torch.device):
        self.model = model
        self.graph = graph
        self.device = device
        self._cache: Dict[FrozenSet[int], float] = {}
        self.n_calls = 0
        self.n_evals = 0

    @property
    def mol_graph(self):
        """Backward-compatible alias."""
        return self.graph

    def __call__(self, active_players: Set[int]) -> float:
        key = frozenset(active_players)
        self.n_calls += 1
        if key not in self._cache:
            self._cache[key] = evaluate_coalition(
                self.model, self.graph, active_players, self.device)
            self.n_evals += 1
        return self._cache[key]

    @property
    def baseline_score(self) -> float:
        return self(set())

    @property
    def full_score(self) -> float:
        return self(set(self.graph.heavy_idx))

    def stats(self) -> Dict[str, Any]:
        bs = self.baseline_score
        fs = self.full_score
        return {
            "baseline_score": bs, "full_score": fs,
            "total_change": fs - bs,
            "n_calls": self.n_calls, "n_evals": self.n_evals,
            "cache_size": len(self._cache),
        }


###########################################################################
# S4  ORDER-2 SHAPLEY-TAYLOR PERMUTATION-PATH ESTIMATOR
###########################################################################

def compute_shapley_taylor_matrix(
    vfunc: ValueFunction,
    graph,                          # InputGraph or compatible
    n_permutations: int = 100,
    seed: Optional[int] = 42,
) -> np.ndarray:
    """Compute the interaction matrix M via the permutation-path estimator
    (Section 4.1).

    Returns M in R^{n x n} where n = number of player nodes.
      M[i,i] = phi_i   (main effect of node i)
      M[i,j] = phi_{ij} (pairwise interaction, i != j)
    """
    rng = np.random.RandomState(seed) if seed is not None else np.random.RandomState()

    players = graph.heavy_idx
    n = len(players)

    phi = np.zeros(n, dtype=np.float64)
    phi_ij = np.zeros((n, n), dtype=np.float64)
    phi_ij_counts = np.zeros((n, n), dtype=np.float64)

    v_empty = vfunc(set())

    for m_perm in range(n_permutations):
        perm = rng.permutation(n)

        prefix_coalitions: List[Set[int]] = [set()]
        prefix_vals: List[float] = [0.0]

        coalition: Set[int] = set()
        for k in range(n):
            gi = players[perm[k]]
            coalition = coalition | {gi}
            prefix_coalitions.append(set(coalition))
            prefix_vals.append(vfunc(coalition) - v_empty)

        # Degree-1
        for k in range(n):
            phi[perm[k]] += prefix_vals[k + 1] - prefix_vals[k]

        # Degree-2
        for k_paper in range(2, n + 1):
            prev_pos = perm[k_paper - 2]
            curr_pos = perm[k_paper - 1]
            prev_global = players[prev_pos]
            curr_global = players[curr_pos]

            T_km1 = prefix_coalitions[k_paper - 1]
            T_minus_prev = T_km1 - {prev_global}
            T_swap = T_minus_prev | {curr_global}

            v_minus_prev = vfunc(T_minus_prev) - v_empty
            v_swap = vfunc(T_swap) - v_empty

            d_k = (prefix_vals[k_paper] - prefix_vals[k_paper - 1]
                   - v_swap + v_minus_prev)

            phi_ij[prev_pos, curr_pos] += d_k
            phi_ij[curr_pos, prev_pos] += d_k
            phi_ij_counts[prev_pos, curr_pos] += 1
            phi_ij_counts[curr_pos, prev_pos] += 1

        if (m_perm + 1) % max(1, n_permutations // 5) == 0:
            logger.info(
                f"ST permutation {m_perm + 1}/{n_permutations}  "
                f"(model evals so far: {vfunc.n_evals})")

    phi /= n_permutations
    mask = phi_ij_counts > 0
    phi_ij[mask] /= phi_ij_counts[mask]

    M = phi_ij.copy()
    for i in range(n):
        M[i, i] = phi[i]
    return M


###########################################################################
# S5  GRAPH CONNECTIVITY UTILITIES
###########################################################################

def is_connected_subgraph(
    nodes: Set[int], adj: Dict[int, Set[int]],
) -> bool:
    if len(nodes) <= 1:
        return True
    start = next(iter(nodes))
    visited = {start}
    stack = [start]
    while stack:
        v = stack.pop()
        for nb in adj.get(v, set()):
            if nb in nodes and nb not in visited:
                visited.add(nb); stack.append(nb)
    return visited == nodes


def split_into_connected_components(
    nodes: Set[int], adj: Dict[int, Set[int]],
) -> List[Set[int]]:
    remaining = set(nodes)
    components: List[Set[int]] = []
    while remaining:
        start = next(iter(remaining))
        comp = {start}; stack = [start]
        while stack:
            v = stack.pop()
            for nb in adj.get(v, set()):
                if nb in remaining and nb not in comp:
                    comp.add(nb); stack.append(nb)
        components.append(comp)
        remaining -= comp
    return components


def enforce_connected_partition(
    partition: List[Set[int]], adj: Dict[int, Set[int]],
) -> List[Set[int]]:
    repaired: List[Set[int]] = []
    for group in partition:
        if len(group) <= 1 or is_connected_subgraph(group, adj):
            repaired.append(group)
        else:
            repaired.extend(split_into_connected_components(group, adj))
    return repaired


###########################################################################
# S6  GROUP-LEVEL AGGREGATION FROM INTERACTION MATRIX
###########################################################################

def aggregate_group_values(
    M: np.ndarray,
    partition: List[List[int]],
    player_idx: List[int],
) -> Dict[str, Any]:
    """Aggregate node-level Shapley-Taylor values to group level (Eq. 12).

    Value(S_i)  = Sum_{a in S_i} phi_a + Sum_{a<b in S_i} phi_{ab}
    I(S_i,S_j)  = Sum_{a in S_i} Sum_{b in S_j} phi_{ab}
    """
    global_to_pos = {g: p for p, g in enumerate(player_idx)}
    K = len(partition)

    group_values: List[float] = []
    for group in partition:
        positions = [global_to_pos[g] for g in group if g in global_to_pos]
        val = sum(M[p, p] for p in positions)
        for ii in range(len(positions)):
            for jj in range(ii + 1, len(positions)):
                val += M[positions[ii], positions[jj]]
        group_values.append(float(val))

    cross = np.zeros((K, K), dtype=np.float64)
    for a in range(K):
        pos_a = [global_to_pos[g] for g in partition[a] if g in global_to_pos]
        for b in range(a + 1, K):
            pos_b = [global_to_pos[g] for g in partition[b] if g in global_to_pos]
            s = sum(M[pa, pb] for pa in pos_a for pb in pos_b)
            cross[a, b] = s; cross[b, a] = s

    return {
        "group_values": group_values,
        "cross_scores": cross.tolist(),
        "total_within": sum(group_values),
        "total_cross": float(np.triu(cross, 1).sum()),
    }


###########################################################################
# S7  AGGLOMERATIVE INITIAL PARTITIONING  (Section 4.2)
###########################################################################

def agglomerative_partition(graph, M: np.ndarray) -> List[Set[int]]:
    """Agglomerative merging driven by interaction matrix M.

    Merges highest-affinity adjacent group pairs until k_0 = ceil(sqrt(n))
    groups remain.
    """
    players = graph.heavy_idx
    n = len(players)
    adj = graph.heavy_adj
    k_target = max(1, math.ceil(math.sqrt(n)))
    global_to_pos = {g: p for p, g in enumerate(players)}

    groups: List[Set[int]] = [{h} for h in players]

    def _adj(gi, gj):
        for a in gi:
            for b in gj:
                if b in adj.get(a, set()):
                    return True
        return False

    def _aff(gi, gj):
        return sum(abs(M[global_to_pos[a], global_to_pos[b]])
                   for a in gi for b in gj)

    while len(groups) > k_target:
        best_s, best_p = -1.0, None
        for i in range(len(groups)):
            for j in range(i + 1, len(groups)):
                if not _adj(groups[i], groups[j]):
                    continue
                af = _aff(groups[i], groups[j])
                if af > best_s:
                    best_s = af; best_p = (i, j)
        if best_p is None:
            break
        i, j = best_p
        groups[i] = groups[i] | groups[j]
        groups.pop(j)

    logger.info(f"Agglomerative partition: {n} nodes -> {len(groups)} groups "
                f"(target k_0 = {k_target})")
    return groups


###########################################################################
# S8  PARTITION SCORER  (Section 4.4)
###########################################################################

class PartitionScorer:
    """Scoring function: Coverage - gamma_H*Entropy - gamma_S*SizePenalty."""

    def __init__(self, group_values: List[float], cross_scores: List[List[float]],
                 group_sizes: List[int], n_nodes: int,
                 gamma_H: float = 0.4, gamma_S: float = 0.3):
        self.gamma_H = gamma_H
        self.gamma_S = gamma_S
        self.mu_ideal = math.sqrt(n_nodes)

        self._components: List[Dict[str, Any]] = []
        for i, v in enumerate(group_values):
            if v > 0:
                self._components.append(
                    {"type": "group", "index": i, "value": v,
                     "size": group_sizes[i]})
        K = len(group_values)
        for i in range(K):
            for j in range(i + 1, K):
                v = cross_scores[i][j]
                if v > 0:
                    self._components.append(
                        {"type": "pair", "index": (i, j), "value": v})
        self._components.sort(key=lambda c: c["value"], reverse=True)
        self._ell = len(self._components)

    def score(self) -> Tuple[float, int, List[Dict]]:
        if self._ell == 0:
            return 0.0, 0, []

        best_score = -float("inf"); best_m = 0; best_Lm = []

        for m in range(1, self._ell + 1):
            L_m = self._components[:m]

            coverage = sum((1.0 / j) * self._components[j - 1]["value"]
                           for j in range(1, m + 1))

            total = sum(c["value"] for c in L_m)
            H = 0.0
            if total > 0:
                for c in L_m:
                    p = c["value"] / total
                    if p > 0:
                        H -= p * math.log2(p)

            group_comps = [c for c in L_m if c["type"] == "group"]
            size_pen = 0.0
            if group_comps and self.mu_ideal > 1e-8:
                log_mu = math.log(self.mu_ideal)
                for c in group_comps:
                    if c["size"] > 0:
                        size_pen += (math.log(c["size"]) - log_mu) ** 2
                size_pen /= len(group_comps)

            s_m = coverage - self.gamma_H * H - self.gamma_S * size_pen
            if s_m > best_score:
                best_score = s_m; best_m = m; best_Lm = list(L_m)

        return max(0.0, best_score), best_m, best_Lm


###########################################################################
# S9  SIMULATED ANNEALING PARTITION SEARCH  (Section 4.3)
###########################################################################

class PartitionSearcher:
    """SA over connected partitions of the player-node graph."""

    def __init__(self, graph, M: np.ndarray,
                 gamma_H: float = 0.4, gamma_S: float = 0.3):
        self.graph = graph
        self.M = M
        self.gamma_H = gamma_H
        self.gamma_S = gamma_S

        self.players = graph.heavy_idx
        self.n = len(self.players)
        self.adj = graph.heavy_adj

        # Backward-compat aliases
        self.heavy = self.players
        self.mol_graph = self.graph

        self._score_cache: Dict[FrozenSet[FrozenSet[int]], float] = {}
        self.best_partition: Optional[List[Set[int]]] = None
        self.best_score = -float("inf")
        self.log: Dict[str, List] = {
            "iteration": [], "temperature": [], "current_score": [],
            "best_score": [], "accepted": [], "n_groups": [],
        }

    def _evaluate(self, partition: List[Set[int]]) -> float:
        key = frozenset(frozenset(g) for g in partition)
        if key in self._score_cache:
            return self._score_cache[key]
        part_lists = [sorted(g) for g in partition]
        agg = aggregate_group_values(self.M, part_lists, self.players)
        sizes = [len(g) for g in partition]
        scorer = PartitionScorer(
            group_values=agg["group_values"],
            cross_scores=agg["cross_scores"],
            group_sizes=sizes, n_nodes=self.n,
            gamma_H=self.gamma_H, gamma_S=self.gamma_S)
        s, _, _ = scorer.score()
        self._score_cache[key] = s
        return s

    def _n2g(self, partition):
        return {a: gi for gi, g in enumerate(partition) for a in g}

    def _boundary(self, partition):
        n2g = self._n2g(partition)
        boundary = []
        for node, gi in n2g.items():
            for nb in self.adj.get(node, set()):
                gj = n2g.get(nb)
                if gj is not None and gj != gi:
                    boundary.append((node, gi, gj)); break
        return boundary

    def _move_node(self, partition, rng):
        boundary = self._boundary(partition)
        if not boundary: return None
        node, from_g, to_g = boundary[rng.randint(len(boundary))]
        new_part = []
        for i, g in enumerate(partition):
            if i == from_g:
                rem = g - {node}
                if rem: new_part.append(rem)
            elif i == to_g:
                new_part.append(g | {node})
            else:
                new_part.append(set(g))
        return new_part

    def _merge(self, partition, rng):
        n2g = self._n2g(partition)
        pairs = set()
        for node, gi in n2g.items():
            for nb in self.adj.get(node, set()):
                gj = n2g.get(nb)
                if gj is not None and gi < gj:
                    pairs.add((gi, gj))
        if not pairs: return None
        pairs_list = list(pairs)
        gi, gj = pairs_list[rng.randint(len(pairs_list))]
        new_part = []
        for i, g in enumerate(partition):
            if i == gi: continue
            elif i == gj: new_part.append(partition[gi] | partition[gj])
            else: new_part.append(set(g))
        return new_part

    def _split(self, partition, rng):
        splittable = [(i, g) for i, g in enumerate(partition) if len(g) > 1]
        if not splittable: return None
        idx = rng.randint(len(splittable))
        gi, group = splittable[idx]
        terminals = [n for n in group
                     if sum(1 for nb in self.adj.get(n, set()) if nb in group) <= 1]
        node = terminals[rng.randint(len(terminals))] if terminals else list(group)[rng.randint(len(group))]
        new_part = []
        for i, g in enumerate(partition):
            if i == gi:
                rem = g - {node}
                if rem: new_part.append(rem)
                new_part.append({node})
            else:
                new_part.append(set(g))
        return new_part

    def _propose(self, partition, rng):
        if rng.rand() < 0.5:
            result = self._move_node(partition, rng)
        else:
            result = self._merge(partition, rng) if rng.rand() < 0.5 else self._split(partition, rng)
        if result is not None:
            return enforce_connected_partition(result, self.adj)
        return list(partition)

    def search(self, init_partition=None, max_iter=100,
               T0=1.0, alpha=0.99, seed=42, verbose=True):
        rng = np.random.RandomState(seed); random.seed(seed)

        if init_partition is not None:
            current = [set(g) for g in init_partition]
        else:
            current = agglomerative_partition(self.graph, self.M)

        current = enforce_connected_partition(current, self.adj)
        cur_score = self._evaluate(current)
        self.best_partition = copy.deepcopy(current)
        self.best_score = cur_score
        T = T0

        if verbose:
            logger.info(f"SA start: {len(current)} groups, score={cur_score:.4f}")

        for it in range(max_iter):
            candidate = self._propose(current, rng)
            cand_score = self._evaluate(candidate)
            delta = cand_score - cur_score

            if delta > 0 or (T > 0 and rng.rand() < math.exp(delta / max(T, 1e-12))):
                current = candidate; cur_score = cand_score; accepted = True
                if cur_score > self.best_score:
                    self.best_partition = copy.deepcopy(current)
                    self.best_score = cur_score
                    if verbose and it % 10 == 0:
                        logger.info(f"  iter {it}: new best {self.best_score:.4f} "
                                    f"({len(self.best_partition)} groups)")
            else:
                accepted = False

            T *= alpha
            self.log["iteration"].append(it)
            self.log["temperature"].append(T)
            self.log["current_score"].append(cur_score)
            self.log["best_score"].append(self.best_score)
            self.log["accepted"].append(accepted)
            self.log["n_groups"].append(len(current))

        if verbose:
            logger.info(f"SA done: best={self.best_score:.4f}, "
                        f"{len(self.best_partition)} groups, "
                        f"{len(self._score_cache)} partitions evaluated")
        return self.best_partition, self.best_score

    def plot_progress(self):
        if not self.log["iteration"]:
            print("No search data."); return
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        iters = self.log["iteration"]
        axes[0,0].plot(iters, self.log["current_score"], alpha=.5, label="current")
        axes[0,0].plot(iters, self.log["best_score"], lw=2, label="best")
        axes[0,0].set_ylabel("Score"); axes[0,0].legend(); axes[0,0].grid(True, alpha=.3)
        axes[0,1].plot(iters, self.log["temperature"])
        axes[0,1].set_ylabel("temperature"); axes[0,1].set_yscale("log"); axes[0,1].grid(True, alpha=.3)
        win = 10
        if len(iters) > win:
            rates = [np.mean(self.log["accepted"][max(0,i-win):i])
                     for i in range(win, len(iters))]
            axes[1,0].plot(range(win, len(iters)), rates)
        axes[1,0].set_ylabel("accept rate"); axes[1,0].grid(True, alpha=.3)
        axes[1,1].plot(iters, self.log["n_groups"])
        axes[1,1].set_ylabel("# groups"); axes[1,1].grid(True, alpha=.3)
        for ax in axes.flat: ax.set_xlabel("iteration")
        plt.suptitle("SA Progress"); plt.tight_layout(); plt.show()


###########################################################################
# S10  HIGH-LEVEL API
###########################################################################

def explain_graph(
    model: nn.Module,
    graph: InputGraph,
    device: torch.device,
    n_permutations: int = 100,
    sa_iterations: int = 60,
    sa_T0: float = 1.0,
    sa_alpha: float = 0.99,
    gamma_H: float = 0.4,
    gamma_S: float = 0.3,
    seed: int = 42,
    verbose: bool = True,
) -> Dict[str, Any]:
    """End-to-end explanation of a single graph.

    1. Build ValueFunction
    2. Compute interaction matrix M
    3. Agglomerative initial partition
    4. SA search
    5. Aggregate group values
    """
    t0 = time.time()

    if verbose:
        logger.info(f"Graph: n_players={graph.n_heavy}  n_total={graph.n_total}")

    vfunc = ValueFunction(model, graph, device)
    bs = vfunc.baseline_score; fs = vfunc.full_score; delta_z = fs - bs
    if verbose:
        logger.info(f"Baseline z={bs:.4f}  Full z={fs:.4f}  dz={delta_z:.4f}")

    if verbose:
        logger.info(f"Computing interaction matrix M via {n_permutations} permutations...")
    M = compute_shapley_taylor_matrix(vfunc, graph,
                                       n_permutations=n_permutations, seed=seed)

    n = graph.n_heavy
    reconstructed = M.diagonal().sum() + sum(
        M[i, j] for i in range(n) for j in range(i + 1, n))
    if verbose:
        logger.info(f"Efficiency check: Sum phi = {reconstructed:.4f}  "
                    f"vs w(V) = {delta_z:.4f}  "
                    f"(error = {abs(reconstructed - delta_z):.4f})")

    init_part = agglomerative_partition(graph, M)
    init_part = enforce_connected_partition(init_part, graph.heavy_adj)

    searcher = PartitionSearcher(graph, M, gamma_H=gamma_H, gamma_S=gamma_S)
    init_score = searcher._evaluate(init_part)

    best_part, best_score = searcher.search(
        init_partition=init_part, max_iter=sa_iterations,
        T0=sa_T0, alpha=sa_alpha, seed=seed, verbose=verbose)

    best_lists = [sorted(g) for g in best_part]
    best_agg = aggregate_group_values(M, best_lists, graph.heavy_idx)

    elapsed = time.time() - t0
    if verbose:
        logger.info(f"Explanation complete in {elapsed:.1f}s")
        logger.info(f"  model evals: {vfunc.n_evals}  "
                    f"(cache hits: {vfunc.n_calls - vfunc.n_evals})")

    return {
        "graph": graph,
        "vfunc_stats": vfunc.stats(),
        "M": M,
        "init_partition": [sorted(g) for g in init_part],
        "init_score": init_score,
        "best_partition": best_lists,
        "best_score": best_score,
        "sa_log": searcher.log,
        "group_agg": best_agg,
        "elapsed_seconds": elapsed,
    }


def print_explanation(result: Dict[str, Any]):
    """Pretty-print output of explain_graph."""
    graph = result["graph"]
    stats = result["vfunc_stats"]
    delta_z = stats["total_change"]

    print("=" * 70)
    print(f"EXPLANATION")
    print(f"  player nodes: {graph.n_heavy}   total nodes: {graph.n_total}")
    print(f"  baseline z = {stats['baseline_score']:.4f}")
    print(f"  full z     = {stats['full_score']:.4f}")
    print(f"  dz         = {delta_z:.4f}")
    print(f"  p(pos)     = {1/(1+math.exp(-stats['full_score'])):.4f}")
    print(f"  model evals: {stats['n_evals']}")
    print()

    agg = result["group_agg"]; bp = result["best_partition"]
    abs_dz = abs(delta_z) if abs(delta_z) > 1e-8 else 1.0
    print(f"Best partition ({len(bp)} groups, Score={result['best_score']:.4f}):")
    for i, group in enumerate(bp):
        labels = [graph.heavy_atom_symbol(a) for a in group]
        v = agg["group_values"][i]
        print(f"  G{i}: {{{', '.join(labels)}}}  "
              f"Value={v:+.4f}  ({v/abs_dz:+.0%} of dz)")

    cross = agg["cross_scores"]; K = len(bp)
    interactions = sorted(
        [(a, b, cross[a][b]) for a in range(K) for b in range(a+1, K)
         if abs(cross[a][b]) > 1e-6],
        key=lambda x: abs(x[2]), reverse=True)
    if interactions:
        print(f"\nTop group interactions:")
        for a, b, v in interactions[:5]:
            kind = "synergistic" if v > 0 else "antagonistic"
            print(f"  G{a} <-> G{b}: {v:+.4f} ({v/abs_dz:+.0%} of dz)  ({kind})")

    total_v = sum(agg["group_values"])
    total_c = sum(cross[a][b] for a in range(K) for b in range(a+1, K))
    recon = total_v + total_c
    err = abs(recon - delta_z) / abs_dz if abs_dz > 1e-8 else 0
    print(f"\n  Sum Values + Sum Cross = {recon:.4f}  "
          f"(dz = {delta_z:.4f}, error = {err:.1%})")
    print(f"\nElapsed: {result['elapsed_seconds']:.1f}s")
    print("=" * 70)


In [ ]:
"""
Synthetic Benchmark Experiment
=============================================
Synthetic data generation, model training, and explanation evaluation.
explanation verification.  No chemistry dependencies.

This experiment generates synthetic graphs directly from graph templates
(no SMILES, no RDKit).  When used in Colab, the framework file
(framework_generalized.py) must be executed first to provide the
explanation functions (ValueFunction, PartitionSearcher, etc.).
"""

# ═══════════════════════════════════════════════════════════════════════════
# §0  CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════

class CFG:
    SEED              = 42
    DEVICE            = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # dataset
    N_TOTAL_PER_TASK  = 8000       # QUICK — production: 8000
    LABEL_NOISE_RATE  = 0.03
    DIST_THRESH       = 3

    # training
    BATCH_SIZE        = 64
    EPOCHS            = 300         # QUICK — production: 300
    LR                = 1e-3
    HIDDEN            = 64
    DROPOUT           = 0.1
    PATIENCE          = 30         # QUICK — production: 30
    INPUT_DIM         = 8

    # explanation
    N_EXPLAIN         = 100         # QUICK — production: 100
    N_PERM            = 100         # QUICK — production: 100
    SA_ITERATIONS     = 100         # QUICK — production: 100
    MAX_NODES_EXPLAIN = 30

    # deletion faithfulness
    DEL_ENABLED       = True
    DEL_N_RANDOM      = 20          # QUICK — production: 20
    DEL_BOOTSTRAP_N   = 1000        # QUICK — production: 1000
    INSERTION_ENABLED = False       # QUICK — production: True

    # baselines
    GRAD_BASELINE     = True
    BASELINE_N_EXPLAIN = 100        # QUICK — production: 100

    # artifact diagnostics
    ARTIFACT_CHECK    = True       # QUICK — production: True

    # stability
    STABILITY_SEEDS   = [42, 123, 777, 2024, 9999]  # QUICK — production: [42, 123, 777, 2024, 9999]
    STABILITY_N       = 20          # QUICK — production: 20

    # counterfactual experiment (DIST task)
    CF_N              = 50         # QUICK — production: 50
    CF_MAX_INSERT     = 5

    # scaffold balance
    SCAFFOLD_BALANCE  = True

    # output
    OUT_DIR           = "outputs"
    DATA_DIR          = "outputs"

os.makedirs(CFG.OUT_DIR, exist_ok=True)
os.makedirs(CFG.DATA_DIR, exist_ok=True)

def set_seed(seed=CFG.SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed()
print(f"Device: {CFG.DEVICE}")

def save_config():
    cfg = {k: v for k, v in vars(CFG).items()
           if not k.startswith("_") and k != "DEVICE"}
    cfg["DEVICE"] = str(CFG.DEVICE)
    path = os.path.join(CFG.DATA_DIR, "config.json")
    with open(path, "w") as f:
        json.dump(cfg, f, indent=2, default=str)
    print(f"  Config saved to {path}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# §1  GRAPH TEMPLATE LIBRARY
# ═══════════════════════════════════════════════════════════════════════════

class GraphTemplate:
    """Immutable graph template: a small graph used as a building block.

    Stores node count and an edge list.  Templates are combined by the
    GraphBuilder to produce full synthetic graphs.
    """
    def __init__(self, name: str, num_nodes: int, edges: List[Tuple[int, int]]):
        self.name = name
        self.num_nodes = num_nodes
        self.edges = list(edges)   # list of (u, v) undirected

    def __repr__(self):
        return f"GraphTemplate({self.name}, n={self.num_nodes}, |E|={len(self.edges)})"


# ── Motif templates ───────────────────────────────────────────────────────
# Motif A: 3-node path (linear, no clique)  — matches paper §5.1
MOTIF_A = GraphTemplate("motif_A_path", 3,
                        [(0, 1), (1, 2)])

# Motif B: 3-node triangle (clique substructure) — matches paper §5.1
MOTIF_B = GraphTemplate("motif_B_triangle", 3,
                        [(0, 1), (1, 2), (0, 2)])


# ── Scaffold templates ────────────────────────────────────────────────────
def _make_cycle(n: int, name: str = "") -> GraphTemplate:
    edges = [(i, (i + 1) % n) for i in range(n)]
    return GraphTemplate(name or f"cycle_{n}", n, edges)

def _make_path(n: int, name: str = "") -> GraphTemplate:
    edges = [(i, i + 1) for i in range(n - 1)]
    return GraphTemplate(name or f"path_{n}", n, edges)

def _make_star(n_leaves: int, name: str = "") -> GraphTemplate:
    """Central node 0 connected to n_leaves peripheral nodes."""
    edges = [(0, i + 1) for i in range(n_leaves)]
    return GraphTemplate(name or f"star_{n_leaves + 1}", n_leaves + 1, edges)

def _make_wheel(n_rim: int, name: str = "") -> GraphTemplate:
    """Cycle of n_rim nodes + central hub connected to all rim nodes."""
    n = n_rim + 1
    hub = n_rim
    edges = [(i, (i + 1) % n_rim) for i in range(n_rim)]
    edges += [(hub, i) for i in range(n_rim)]
    return GraphTemplate(name or f"wheel_{n}", n, edges)

def _make_ladder(rungs: int, name: str = "") -> GraphTemplate:
    """Two parallel paths of length rungs, connected by rung edges."""
    n = 2 * rungs
    edges = []
    for i in range(rungs - 1):
        edges.append((i, i + 1))
        edges.append((rungs + i, rungs + i + 1))
    for i in range(rungs):
        edges.append((i, rungs + i))
    return GraphTemplate(name or f"ladder_{n}", n, edges)

def _make_binary_tree(depth: int, name: str = "") -> GraphTemplate:
    """Complete binary tree of given depth (root at node 0)."""
    n = (1 << (depth + 1)) - 1
    edges = []
    for i in range(n):
        left = 2 * i + 1
        right = 2 * i + 2
        if left < n:
            edges.append((i, left))
        if right < n:
            edges.append((i, right))
    return GraphTemplate(name or f"btree_d{depth}", n, edges)

def _make_grid(rows: int, cols: int, name: str = "") -> GraphTemplate:
    """2D grid graph."""
    n = rows * cols
    edges = []
    for r in range(rows):
        for c in range(cols):
            idx = r * cols + c
            if c + 1 < cols:
                edges.append((idx, idx + 1))
            if r + 1 < rows:
                edges.append((idx, idx + cols))
    return GraphTemplate(name or f"grid_{rows}x{cols}", n, edges)

def _make_dumbbell(left_n: int, right_n: int, bridge_len: int,
                   name: str = "") -> GraphTemplate:
    """Two cycles connected by a path (bridge).

    Nodes 0..left_n-1: left cycle
    Nodes left_n..left_n+bridge_len-1: bridge chain
    Nodes left_n+bridge_len..left_n+bridge_len+right_n-1: right cycle

    The bridge connects node (left_n-1) to node (left_n) and continues
    to node (left_n+bridge_len-1) connecting to node (left_n+bridge_len).
    """
    offset_bridge = left_n
    offset_right = left_n + bridge_len
    total = left_n + bridge_len + right_n
    edges = []
    # Left cycle
    for i in range(left_n):
        edges.append((i, (i + 1) % left_n))
    # Bridge chain
    prev = left_n - 1
    for i in range(bridge_len):
        cur = offset_bridge + i
        edges.append((prev, cur))
        prev = cur
    # Connect bridge end to right cycle
    edges.append((prev, offset_right))
    # Right cycle
    for i in range(right_n):
        edges.append((offset_right + i, offset_right + (i + 1) % right_n))
    return GraphTemplate(name or f"dumbbell_{left_n}_{right_n}_b{bridge_len}",
                         total, edges)


# ── Scaffold libraries ────────────────────────────────────────────────────
SCAFFOLDS_GENERAL: List[GraphTemplate] = [
    # Cycles
    _make_cycle(5), _make_cycle(6), _make_cycle(7), _make_cycle(8),
    # Paths
    _make_path(4), _make_path(5), _make_path(6),
    # Stars
    _make_star(3), _make_star(4), _make_star(5),
    # Wheels
    _make_wheel(4), _make_wheel(5),
    # Ladders
    _make_ladder(3), _make_ladder(4),
    # Binary trees
    _make_binary_tree(2), _make_binary_tree(3),
    # Grids
    _make_grid(2, 3), _make_grid(3, 3),
    # Dumbbells (for general use too)
    _make_dumbbell(4, 4, 1, "dumbbell_4_4_b1"),
    _make_dumbbell(4, 5, 1, "dumbbell_4_5_b1"),
    _make_dumbbell(5, 5, 1, "dumbbell_5_5_b1"),
    _make_dumbbell(4, 4, 2, "dumbbell_4_4_b2"),
    _make_dumbbell(5, 5, 2, "dumbbell_5_5_b2"),
]

# DIST scaffolds: dumbbells with varying bridge length to control distance.
# Short bridge -> motifs can be close; long bridge -> motifs must be far.
SCAFFOLDS_DIST: List[GraphTemplate] = [
    # Short bridges (enable close motifs, d <= 3 possible)
    _make_dumbbell(4, 4, 0, "dist_4_4_b0"),
    _make_dumbbell(4, 5, 0, "dist_4_5_b0"),
    _make_dumbbell(5, 5, 0, "dist_5_5_b0"),
    _make_dumbbell(4, 4, 1, "dist_4_4_b1"),
    _make_dumbbell(5, 4, 1, "dist_5_4_b1"),
    _make_dumbbell(5, 5, 1, "dist_5_5_b1"),
    # Medium bridges (close depends on attachment site)
    _make_dumbbell(4, 4, 2, "dist_4_4_b2"),
    _make_dumbbell(5, 5, 2, "dist_5_5_b2"),
    _make_dumbbell(4, 5, 2, "dist_4_5_b2"),
    # Longer bridges (tend to produce far motifs)
    _make_dumbbell(4, 4, 3, "dist_4_4_b3"),
    _make_dumbbell(5, 5, 3, "dist_5_5_b3"),
    _make_dumbbell(4, 4, 4, "dist_4_4_b4"),
    _make_dumbbell(5, 5, 4, "dist_5_5_b4"),
]

# Noise templates: small structures attached to add diversity
NOISE_TEMPLATES: List[GraphTemplate] = [
    GraphTemplate("noise_single", 1, []),
    GraphTemplate("noise_edge", 2, [(0, 1)]),
    GraphTemplate("noise_triangle", 3, [(0, 1), (1, 2), (0, 2)]),
    GraphTemplate("noise_path3", 3, [(0, 1), (1, 2)]),
    GraphTemplate("noise_star3", 4, [(0, 1), (0, 2), (0, 3)]),
]


# ═══════════════════════════════════════════════════════════════════════════
# §2  GRAPH BUILDER
# ═══════════════════════════════════════════════════════════════════════════

class GraphBuilder:
    """Incrementally build a graph by attaching templates.

    Maintains a growing node list, edge list, and metadata about which
    nodes belong to which structural role (scaffold, motif_A, motif_B,
    noise).

    All node indices are global (0-indexed, contiguous).
    """

    def __init__(self):
        self.num_nodes: int = 0
        self.edges: List[Tuple[int, int]] = []
        self.node_roles: List[str] = []    # per-node role label
        self.motif_A_nodes: List[int] = []
        self.motif_B_nodes: List[int] = []
        self.scaffold_nodes: List[int] = []
        self.noise_nodes: List[int] = []
        self.scaffold_id: str = ""

    def add_scaffold(self, template: GraphTemplate) -> List[int]:
        """Add a scaffold template.  Returns global indices of new nodes."""
        offset = self.num_nodes
        new_nodes = list(range(offset, offset + template.num_nodes))
        for u, v in template.edges:
            self.edges.append((offset + u, offset + v))
        self.node_roles.extend(["scaffold"] * template.num_nodes)
        self.scaffold_nodes.extend(new_nodes)
        self.num_nodes += template.num_nodes
        self.scaffold_id = template.name
        return new_nodes

    def attach_motif(self, template: GraphTemplate, attach_to: int,
                     motif_attach_node: int = 0,
                     role: str = "motif_A") -> List[int]:
        """Attach a motif template to an existing node.

        Parameters
        ----------
        template : GraphTemplate
            The motif to attach.
        attach_to : int
            Global index of the existing node to connect to.
        motif_attach_node : int
            Which node within the motif template becomes the attachment point.
        role : str
            "motif_A" or "motif_B".

        Returns
        -------
        List of global indices of the new motif nodes.
        """
        offset = self.num_nodes
        new_nodes = list(range(offset, offset + template.num_nodes))
        for u, v in template.edges:
            self.edges.append((offset + u, offset + v))
        # Connect motif to existing graph
        self.edges.append((attach_to, offset + motif_attach_node))
        self.node_roles.extend([role] * template.num_nodes)

        if role == "motif_A":
            self.motif_A_nodes.extend(new_nodes)
        elif role == "motif_B":
            self.motif_B_nodes.extend(new_nodes)
        self.num_nodes += template.num_nodes
        return new_nodes

    def attach_noise(self, template: GraphTemplate, attach_to: int) -> List[int]:
        """Attach a noise template to an existing node."""
        offset = self.num_nodes
        new_nodes = list(range(offset, offset + template.num_nodes))
        for u, v in template.edges:
            self.edges.append((offset + u, offset + v))
        if template.num_nodes > 0:
            self.edges.append((attach_to, offset))
        self.node_roles.extend(["noise"] * template.num_nodes)
        self.noise_nodes.extend(new_nodes)
        self.num_nodes += template.num_nodes
        return new_nodes

    def get_adjacency(self) -> Dict[int, Set[int]]:
        """Build adjacency dict from edge list."""
        adj: Dict[int, Set[int]] = defaultdict(set)
        for u, v in self.edges:
            adj[u].add(v)
            adj[v].add(u)
        return adj

    def shortest_path_distance(self, sources: List[int],
                                targets: List[int]) -> int:
        """BFS shortest path between any source and any target.

        Returns -1 if no path exists or either set is empty.
        """
        if not sources or not targets:
            return -1
        adj = self.get_adjacency()
        target_set = set(targets)
        visited = set(sources)
        frontier = list(sources)
        dist = 0
        while frontier:
            if visited & target_set:
                return dist
            next_frontier = []
            for node in frontier:
                for nb in adj.get(node, set()):
                    if nb not in visited:
                        visited.add(nb)
                        next_frontier.append(nb)
            frontier = next_frontier
            dist += 1
        return -1

    def to_edge_index(self) -> torch.Tensor:
        """Convert edge list to PyG edge_index (2 x 2|E|), undirected."""
        if not self.edges:
            return torch.zeros((2, 0), dtype=torch.long)
        edge_list = []
        for u, v in self.edges:
            edge_list.append([u, v])
            edge_list.append([v, u])
        return torch.tensor(edge_list, dtype=torch.long).t().contiguous()

    def metadata(self) -> Dict[str, Any]:
        return {
            "scaffold_id": self.scaffold_id,
            "n_nodes": self.num_nodes,
            "motif_A_nodes": list(self.motif_A_nodes),
            "motif_B_nodes": list(self.motif_B_nodes),
            "scaffold_nodes": list(self.scaffold_nodes),
            "noise_nodes": list(self.noise_nodes),
            "has_A": len(self.motif_A_nodes) > 0,
            "has_B": len(self.motif_B_nodes) > 0,
        }


# ═══════════════════════════════════════════════════════════════════════════
# §3  NODE FEATURE COMPUTATION
# ═══════════════════════════════════════════════════════════════════════════

def compute_structural_features(
    num_nodes: int,
    edge_index: torch.Tensor,
    feat_dim: int = 8,
) -> torch.Tensor:
    """Compute structural node features from graph topology.

    Features are derived purely from local graph structure so that the
    GNN must learn motif patterns through message passing.  No feature
    directly encodes motif membership.

    Features (all per-node):
      0: degree
      1: inverse degree  1/(deg+1)
      2: sum of neighbor degrees
      3: number of triangles the node participates in
      4: local clustering coefficient
      5: is leaf (degree == 1)
      6: max neighbor degree
      7: min neighbor degree

    Parameters
    ----------
    num_nodes : int
    edge_index : torch.Tensor, shape (2, 2|E|)
    feat_dim : int
        Must be <= 8.  Extra dimensions are zero-padded.

    Returns
    -------
    x : torch.Tensor, shape (num_nodes, feat_dim)
    """
    adj: Dict[int, Set[int]] = defaultdict(set)
    ei = edge_index.numpy()
    for k in range(ei.shape[1]):
        u, v = int(ei[0, k]), int(ei[1, k])
        adj[u].add(v)

    feats = np.zeros((num_nodes, max(feat_dim, 8)), dtype=np.float32)
    for v in range(num_nodes):
        neighbors = sorted(adj[v])
        deg = len(neighbors)

        feats[v, 0] = deg
        feats[v, 1] = 1.0 / (deg + 1)
        feats[v, 2] = sum(len(adj[n]) for n in neighbors)

        # Triangles
        tri = 0
        for i, n1 in enumerate(neighbors):
            for n2 in neighbors[i + 1:]:
                if n2 in adj[n1]:
                    tri += 1
        feats[v, 3] = tri

        # Local clustering coefficient
        if deg >= 2:
            feats[v, 4] = tri / (deg * (deg - 1) / 2)

        feats[v, 5] = 1.0 if deg == 1 else 0.0
        feats[v, 6] = max((len(adj[n]) for n in neighbors), default=0)
        feats[v, 7] = min((len(adj[n]) for n in neighbors), default=0)

    x = torch.tensor(feats[:, :feat_dim], dtype=torch.float32)

    # Per-feature standardization within the graph:
    # Each feature is normalized to mean=0, std=1 across nodes.
    # This prevents degree (range 1-20) from dominating clustering (range 0-1).
    if num_nodes > 1:
        mu = x.mean(dim=0, keepdim=True)
        std = x.std(dim=0, keepdim=True).clamp(min=1e-6)
        x = (x - mu) / std

    return x


# ═══════════════════════════════════════════════════════════════════════════
# §4  SYNTHETIC GRAPH GENERATOR
# ═══════════════════════════════════════════════════════════════════════════

def _pick_attachment_sites(
    scaffold_nodes: List[int],
    adj: Dict[int, Set[int]],
    n_sites: int,
    rng: np.random.RandomState,
    prefer_peripheral: bool = True,
) -> List[int]:
    """Pick n_sites distinct attachment nodes from scaffold_nodes.

    If prefer_peripheral, favour nodes with degree <= 2 (boundary nodes).
    """
    scaffold_set = set(scaffold_nodes)
    if prefer_peripheral:
        peripheral = [n for n in scaffold_nodes
                      if len(adj.get(n, set()) & scaffold_set) <= 2]
        if len(peripheral) >= n_sites:
            return rng.choice(peripheral, size=n_sites, replace=False).tolist()
    return rng.choice(scaffold_nodes, size=min(n_sites, len(scaffold_nodes)),
                      replace=False).tolist()


def generate_graph(
    scaffold: GraphTemplate,
    add_A: bool,
    add_B: bool,
    rng: np.random.RandomState,
    n_noise_range: Tuple[int, int] = (0, 2),
    feat_dim: int = 8,
    max_nodes: int = 30,
) -> Optional[Tuple[Data, Dict[str, Any]]]:
    """Generate a single synthetic graph.

    Parameters
    ----------
    scaffold : GraphTemplate
        Base scaffold structure.
    add_A, add_B : bool
        Whether to attach motif A and/or motif B.
    rng : np.random.RandomState
    n_noise_range : (int, int)
        Min and max number of noise attachments.
    feat_dim : int
        Node feature dimension.
    max_nodes : int
        Reject graphs exceeding this size.

    Returns
    -------
    (PyG Data, metadata dict) or None if generation fails.
    """
    builder = GraphBuilder()
    scaffold_nodes = builder.add_scaffold(scaffold)
    adj = builder.get_adjacency()

    # Determine how many attachment sites we need
    n_motifs = int(add_A) + int(add_B)
    n_needed = n_motifs + rng.randint(max(n_noise_range[0], 0),
                                       n_noise_range[1] + 1)

    # Adaptive noise: more noise for fewer motifs (equalises graph size)
    extra_noise = 2 - n_motifs
    n_noise = rng.randint(max(n_noise_range[0] + extra_noise, 0),
                          n_noise_range[1] + extra_noise + 1)

    # Pick attachment sites
    all_sites = _pick_attachment_sites(
        scaffold_nodes, adj, n_motifs + n_noise + 2, rng)

    site_idx = 0

    # Attach motifs
    if add_A and site_idx < len(all_sites):
        builder.attach_motif(MOTIF_A, all_sites[site_idx], role="motif_A")
        site_idx += 1

    if add_B and site_idx < len(all_sites):
        builder.attach_motif(MOTIF_B, all_sites[site_idx], role="motif_B")
        site_idx += 1

    # Attach noise
    for _ in range(n_noise):
        if site_idx >= len(all_sites):
            # Pick a random existing node
            attach_node = rng.randint(builder.num_nodes)
        else:
            attach_node = all_sites[site_idx]
            site_idx += 1
        noise_tmpl = NOISE_TEMPLATES[rng.randint(len(NOISE_TEMPLATES))]
        builder.attach_noise(noise_tmpl, attach_node)

    # Size check
    if builder.num_nodes > max_nodes or builder.num_nodes < 3:
        return None

    # Build PyG Data
    edge_index = builder.to_edge_index()
    x = compute_structural_features(builder.num_nodes, edge_index, feat_dim)

    meta = builder.metadata()
    if add_A and add_B:
        meta["dist_AB"] = builder.shortest_path_distance(
            builder.motif_A_nodes, builder.motif_B_nodes)
    else:
        meta["dist_AB"] = -1

    data = Data(x=x, edge_index=edge_index)
    # Store motif node sets as tensors in the Data object
    data.motif_A_nodes = torch.tensor(builder.motif_A_nodes, dtype=torch.long)
    data.motif_B_nodes = torch.tensor(builder.motif_B_nodes, dtype=torch.long)

    return data, meta


def _gen_class(
    add_A: bool, add_B: bool, target: int, label: int,
    cname: str, rng: np.random.RandomState,
    scaffolds: List[GraphTemplate],
    seen_hashes: Set[int],
    feat_dim: int = 8,
    max_nodes: int = 30,
    id_offset: int = 0,
) -> Tuple[List[Data], List[Dict]]:
    """Generate a class of synthetic graphs, deduplicating by edge hash.

    Parameters
    ----------
    id_offset : int
        Starting graph_id.  Must be passed from the caller to ensure
        globally unique IDs across multiple _gen_class calls.
    """
    data_list, records = [], []
    attempts = 0
    while len(data_list) < target and attempts < target * 80:
        attempts += 1
        scaff = scaffolds[rng.randint(len(scaffolds))]
        result = generate_graph(scaff, add_A, add_B, rng,
                                feat_dim=feat_dim, max_nodes=max_nodes)
        if result is None:
            continue
        data, meta = result
        # Simple dedup via hash of sorted edge pairs
        edge_hash = hash(tuple(sorted(
            (int(data.edge_index[0, i]), int(data.edge_index[1, i]))
            for i in range(data.edge_index.shape[1])
        )))
        if edge_hash in seen_hashes:
            continue
        seen_hashes.add(edge_hash)

        gid = id_offset + len(data_list)
        data.y = torch.tensor([label], dtype=torch.float32)
        data.graph_id = gid
        data_list.append(data)

        records.append({
            "graph_id": gid,
            "label": label,
            "has_A": int(add_A),
            "has_B": int(add_B),
            "struct_class": cname,
            "scaffold_id": meta["scaffold_id"],
            "n_nodes": meta["n_nodes"],
            "dist_AB": meta["dist_AB"],
        })

    print(f"  {cname}: {len(data_list)}/{target} (attempts={attempts})")
    return data_list, records


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# §5  DATASET GENERATORS (AND / OR / DIST)
# ═══════════════════════════════════════════════════════════════════════════

def add_label_noise(df: pd.DataFrame, rate: float, seed: int = 42) -> pd.DataFrame:
    rng = np.random.RandomState(seed)
    n_flip = int(len(df) * rate)
    idx = rng.choice(len(df), size=n_flip, replace=False)
    df = df.copy()
    df.loc[idx, "label"] = 1 - df.loc[idx, "label"]
    print(f"  Label noise: flipped {n_flip}/{len(df)} ({rate:.1%})")
    return df


def balance_per_scaffold(df: pd.DataFrame, seed: int = 42) -> pd.DataFrame:
    """Balance motif combos per scaffold and drop single-label scaffolds."""
    if not CFG.SCAFFOLD_BALANCE:
        return df
    lab_col = "clean_label" if "clean_label" in df.columns else "label"
    rng = np.random.RandomState(seed)
    n_before = len(df)

    # Drop single-label scaffolds
    scaff_labels = df.groupby("scaffold_id")[lab_col].nunique()
    multi_label = scaff_labels[scaff_labels >= 2].index
    single_label_scaffs = scaff_labels[scaff_labels < 2].index.tolist()
    n_dropped_sl = df[df["scaffold_id"].isin(single_label_scaffs)].shape[0]
    df = df[df["scaffold_id"].isin(multi_label)].copy()
    if single_label_scaffs:
        print(f"  Scaffold filter: dropped {n_dropped_sl} graphs from "
              f"{len(single_label_scaffs)} single-label scaffolds")

    # Equalise motif combos within each scaffold
    keep_idx = []
    for scaff, grp in df.groupby("scaffold_id"):
        combo_groups = grp.groupby(["has_A", "has_B"])
        counts = combo_groups.size()
        if len(counts) < 2:
            keep_idx.extend(grp.index.tolist())
            continue
        target_n = max(int(np.percentile(counts.values, 25)), counts.min())
        if target_n == 0:
            keep_idx.extend(grp.index.tolist())
            continue
        for _, sub in combo_groups:
            if len(sub) <= target_n:
                keep_idx.extend(sub.index.tolist())
            else:
                keep_idx.extend(sub.sample(n=target_n, random_state=rng).index.tolist())

    df_bal = df.loc[keep_idx].reset_index(drop=True)
    print(f"  Scaffold balance: {n_before} -> {len(df_bal)} "
          f"(dropped {n_before - len(df_bal)})")
    return df_bal


def generate_dataset_AND(N: int = 8000, seed: int = 42,
                         noise_rate: float = 0.03
                         ) -> Tuple[List[Data], pd.DataFrame]:
    """AND task: label = 1 iff graph contains both motif A and motif B."""
    rng = np.random.RandomState(seed)
    seen = set()
    n_pos = N // 2
    n_neg_per = N // 6

    all_data, all_recs = [], []
    for add_A, add_B, label, target, cname in [
        (True,  True,  1, n_pos,     "A_and_B"),
        (False, False, 0, n_neg_per, "none"),
        (True,  False, 0, n_neg_per, "A_only"),
        (False, True,  0, n_neg_per, "B_only"),
    ]:
        d, r = _gen_class(add_A, add_B, target, label, cname, rng,
                          SCAFFOLDS_GENERAL, seen, id_offset=len(all_data))
        all_data.extend(d)
        all_recs.extend(r)

    # Remaining negatives
    rem = N - len(all_data)
    if rem > 0:
        d, r = _gen_class(False, False, rem, 0, "none_extra", rng,
                          SCAFFOLDS_GENERAL, seen, id_offset=len(all_data))
        all_data.extend(d)
        all_recs.extend(r)

    df = pd.DataFrame(all_recs).sample(frac=1, random_state=seed).reset_index(drop=True)
    df = balance_per_scaffold(df, seed=seed)
    df["clean_label"] = df["label"].copy()
    df = add_label_noise(df, noise_rate, seed=seed + 100)

    # Re-index data list to match df
    id_to_data = {int(d.graph_id): d for d in all_data}
    data_list = [id_to_data[gid] for gid in df["graph_id"].values
                 if gid in id_to_data]

    # Sync labels after noise
    for i, (_, row) in enumerate(df.iterrows()):
        if i < len(data_list):
            data_list[i].y = torch.tensor([row["label"]], dtype=torch.float32)

    print(f"  AND final: {len(df)}, labels: {df['label'].value_counts().to_dict()}")
    return data_list, df


def generate_dataset_OR(N: int = 8000, seed: int = 43,
                        noise_rate: float = 0.03
                        ) -> Tuple[List[Data], pd.DataFrame]:
    """OR task: label = 1 iff graph contains motif A or motif B (or both)."""
    rng = np.random.RandomState(seed)
    seen = set()
    n_neg = N // 2
    n_pos_per = N // 6

    all_data, all_recs = [], []
    for add_A, add_B, label, target, cname in [
        (False, False, 0, n_neg,     "none"),
        (True,  False, 1, n_pos_per, "A_only"),
        (False, True,  1, n_pos_per, "B_only"),
        (True,  True,  1, n_pos_per, "A_and_B"),
    ]:
        d, r = _gen_class(add_A, add_B, target, label, cname, rng,
                          SCAFFOLDS_GENERAL, seen, id_offset=len(all_data))
        all_data.extend(d)
        all_recs.extend(r)

    rem = N - len(all_data)
    if rem > 0:
        d, r = _gen_class(True, False, rem, 1, "A_only_extra", rng,
                          SCAFFOLDS_GENERAL, seen, id_offset=len(all_data))
        all_data.extend(d)
        all_recs.extend(r)

    df = pd.DataFrame(all_recs).sample(frac=1, random_state=seed).reset_index(drop=True)
    df = balance_per_scaffold(df, seed=seed)
    df["clean_label"] = df["label"].copy()
    df = add_label_noise(df, noise_rate, seed=seed + 100)

    id_to_data = {int(d.graph_id): d for d in all_data}
    data_list = [id_to_data[gid] for gid in df["graph_id"].values
                 if gid in id_to_data]
    for i, (_, row) in enumerate(df.iterrows()):
        if i < len(data_list):
            data_list[i].y = torch.tensor([row["label"]], dtype=torch.float32)

    print(f"  OR final: {len(df)}, labels: {df['label'].value_counts().to_dict()}")
    return data_list, df


def generate_dataset_DIST(N: int = 8000, seed: int = 44,
                          noise_rate: float = 0.03,
                          dist_thresh: int = 3
                          ) -> Tuple[List[Data], pd.DataFrame]:
    """DIST task: label = 1 iff both motifs present AND distance <= thresh.

    Uses dumbbell scaffolds to control inter-motif distance.
    """
    rng = np.random.RandomState(seed)
    seen = set()
    all_data, all_recs = [], []

    # Phase 1: Generate AB pool (both close and far)
    ab_target = N // 2
    ab_close_data, ab_close_recs = [], []
    ab_far_data, ab_far_recs = [], []
    att = 0
    while (len(ab_close_data) + len(ab_far_data)) < ab_target * 2 and att < ab_target * 100:
        att += 1
        scaff = SCAFFOLDS_DIST[rng.randint(len(SCAFFOLDS_DIST))]
        result = generate_graph(scaff, True, True, rng,
                                max_nodes=CFG.MAX_NODES_EXPLAIN)
        if result is None:
            continue
        data, meta = result
        edge_hash = hash(tuple(sorted(
            (int(data.edge_index[0, i]), int(data.edge_index[1, i]))
            for i in range(data.edge_index.shape[1])
        )))
        if edge_hash in seen:
            continue
        seen.add(edge_hash)

        d = meta["dist_AB"]
        if d < 0:
            continue

        rec = {"graph_id": len(all_data) + len(ab_close_data) + len(ab_far_data),
               "has_A": 1, "has_B": 1, "dist_AB": d,
               "scaffold_id": meta["scaffold_id"], "n_nodes": meta["n_nodes"]}

        data.graph_id = rec["graph_id"]
        data.y = torch.tensor([0], dtype=torch.float32)  # set later

        if d <= dist_thresh:
            rec["struct_class"] = "AB_close"
            rec["label"] = 1
            data.y = torch.tensor([1], dtype=torch.float32)
            ab_close_data.append(data)
            ab_close_recs.append(rec)
        else:
            rec["struct_class"] = "AB_far"
            rec["label"] = 0
            ab_far_data.append(data)
            ab_far_recs.append(rec)

    # Balance close:far ratio (at most 2:1)
    n_c, n_f = len(ab_close_data), len(ab_far_data)
    if n_c > n_f * 2:
        idx = rng.choice(n_c, size=n_f * 2, replace=False)
        ab_close_data = [ab_close_data[i] for i in sorted(idx)]
        ab_close_recs = [ab_close_recs[i] for i in sorted(idx)]
    elif n_f > n_c * 2:
        idx = rng.choice(n_f, size=n_c * 2, replace=False)
        ab_far_data = [ab_far_data[i] for i in sorted(idx)]
        ab_far_recs = [ab_far_recs[i] for i in sorted(idx)]

    all_data.extend(ab_close_data)
    all_data.extend(ab_far_data)
    all_recs.extend(ab_close_recs)
    all_recs.extend(ab_far_recs)
    print(f"  DIST AB_close: {len(ab_close_data)}  AB_far: {len(ab_far_data)}")

    # Phase 2: Non-AB negatives from same scaffold pool
    n_neg_per = N // 8
    for add_A, add_B, cname in [(False, False, "none"),
                                 (True, False, "A_only"),
                                 (False, True, "B_only")]:
        d, r = _gen_class(add_A, add_B, n_neg_per, 0, cname, rng,
                          SCAFFOLDS_DIST, seen,
                          max_nodes=CFG.MAX_NODES_EXPLAIN,
                          id_offset=len(all_data))
        all_data.extend(d)
        all_recs.extend(r)

    df = pd.DataFrame(all_recs).sample(frac=1, random_state=seed).reset_index(drop=True)
    df = balance_per_scaffold(df, seed=seed)
    df["clean_label"] = df["label"].copy()
    df = add_label_noise(df, noise_rate, seed=seed + 100)

    id_to_data = {int(d.graph_id): d for d in all_data}
    data_list = [id_to_data[gid] for gid in df["graph_id"].values
                 if gid in id_to_data]
    for i, (_, row) in enumerate(df.iterrows()):
        if i < len(data_list):
            data_list[i].y = torch.tensor([row["label"]], dtype=torch.float32)

    print(f"  DIST final: {len(df)}, labels: {df['label'].value_counts().to_dict()}")
    return data_list, df


# ═══════════════════════════════════════════════════════════════════════════
# §6  SYNTHETIC GRAPH WRAPPER  (framework compatibility layer)
# ═══════════════════════════════════════════════════════════════════════════

class SyntheticGraph:
    """Graph-native wrapper for the explanation framework.

    Provides the same attribute interface that the framework functions
    (ValueFunction, compute_shapley_taylor_matrix, PartitionSearcher, etc.)
    expect.  Equivalent to InputGraph.from_pyg_data() for the case where
    every node is a player.

    For synthetic graphs there are no non-player nodes, so:
      heavy_idx = all node indices
      heavy_to_h = empty dict
      n_heavy = n_total
    """

    def __init__(self, x: torch.Tensor, edge_index: torch.Tensor):
        self.x = x.clone()
        self.edge_index = edge_index.clone()
        self.n_total = x.shape[0]
        self.n_heavy = self.n_total
        self.heavy_idx: List[int] = list(range(self.n_total))
        self.h_idx: List[int] = []
        self.heavy_to_h: Dict[int, List[int]] = defaultdict(list)
        self.feat_dim = x.shape[1]
        self.baseline = torch.zeros(self.feat_dim, dtype=torch.float32)

        # Build heavy-atom adjacency
        self.heavy_adj: Dict[int, Set[int]] = defaultdict(set)
        ei = edge_index.numpy()
        for k in range(ei.shape[1]):
            a, b = int(ei[0, k]), int(ei[1, k])
            self.heavy_adj[a].add(b)

    def pyg_data(self) -> Data:
        return Data(x=self.x.clone(), edge_index=self.edge_index.clone())

    def heavy_atom_symbol(self, idx: int) -> str:
        return f"v{idx}"

    def heavy_atom_labels(self) -> List[str]:
        return [f"{i}:v{i}" for i in self.heavy_idx]


# ═══════════════════════════════════════════════════════════════════════════
# §7  GRAPH-NATIVE EXPLANATION FUNCTION
# ═══════════════════════════════════════════════════════════════════════════

# Import framework functions (from the generalized framework file).
# In Colab, these will be in the same namespace since both cells are executed.
# The framework provides: ValueFunction, compute_shapley_taylor_matrix,
#   agglomerative_partition, enforce_connected_partition,
#   PartitionSearcher, aggregate_group_values

def explain_graph(
    model: nn.Module,
    data: Data,
    device: torch.device,
    n_permutations: int = 100,
    sa_iterations: int = 100,
    sa_T0: float = 1.0,
    sa_alpha: float = 0.99,
    gamma_H: float = 0.4,
    gamma_S: float = 0.3,
    seed: int = 42,
    verbose: bool = False,
) -> Optional[Dict[str, Any]]:
    """End-to-end explanation of a single graph.

    Takes a PyG Data object, wraps it in SyntheticGraph, and runs the
    full explanation pipeline from the framework.

    Steps:
      1. Wrap graph in SyntheticGraph
      2. Build ValueFunction
      3. Compute interaction matrix M via permutation-path estimator
      4. Build initial partition via agglomerative merging
      5. Run simulated annealing to maximise partition score
      6. Aggregate group-level values for best partition

    Returns dict with: sg, vfunc_stats, M, init_partition, init_score,
    best_partition, best_score, sa_log, group_agg, elapsed_seconds
    """
    t0 = time.time()
    try:
        sg = SyntheticGraph(data.x, data.edge_index)
    except Exception as e:
        logger.warning(f"SyntheticGraph creation failed: {e}")
        return None

    if verbose:
        logger.info(f"Graph: n={sg.n_heavy} nodes")

    # Value function
    vfunc = ValueFunction(model, sg, device)
    bs = vfunc.baseline_score
    fs = vfunc.full_score
    delta_z = fs - bs
    if verbose:
        logger.info(f"Baseline z={bs:.4f}  Full z={fs:.4f}  dz={delta_z:.4f}")

    # Interaction matrix
    M = compute_shapley_taylor_matrix(
        vfunc, sg, n_permutations=n_permutations, seed=seed)

    # Efficiency check
    n = sg.n_heavy
    reconstructed = M.diagonal().sum() + sum(
        M[i, j] for i in range(n) for j in range(i + 1, n))

    # Initial partition
    init_part = agglomerative_partition(sg, M)
    init_part = enforce_connected_partition(init_part, sg.heavy_adj)

    # SA search
    searcher = PartitionSearcher(sg, M, gamma_H=gamma_H, gamma_S=gamma_S)
    init_score = searcher._evaluate(init_part)
    best_part, best_score = searcher.search(
        init_partition=init_part, max_iter=sa_iterations,
        T0=sa_T0, alpha=sa_alpha, seed=seed, verbose=verbose)

    best_lists = [sorted(g) for g in best_part]
    best_agg = aggregate_group_values(M, best_lists, sg.heavy_idx)

    elapsed = time.time() - t0
    return {
        "sg": sg,
        "vfunc_stats": vfunc.stats(),
        "M": M,
        "init_partition": [sorted(g) for g in init_part],
        "init_score": init_score,
        "best_partition": best_lists,
        "best_score": best_score,
        "sa_log": searcher.log,
        "group_agg": best_agg,
        "elapsed_seconds": elapsed,
    }


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# §8  DATASET DIAGNOSTICS
# ═══════════════════════════════════════════════════════════════════════════

def dataset_diagnostics(df: pd.DataFrame, task_name: str):
    print(f"\n  -- DIAGNOSTICS: {task_name} --")
    print(f"  Total: {len(df)}  Labels (noisy): {df['label'].value_counts().to_dict()}")
    if "clean_label" in df.columns:
        n_flipped = (df["label"] != df["clean_label"]).sum()
        print(f"  Labels (clean):  {df['clean_label'].value_counts().to_dict()}")
        print(f"  Noise-flipped:   {n_flipped} ({100*n_flipped/len(df):.1f}%)")
    if "struct_class" in df.columns:
        print(f"  Classes: {df['struct_class'].value_counts().to_dict()}")
    for lab in [0, 1]:
        sub = df[df["label"] == lab]
        if "n_nodes" in sub.columns and len(sub):
            print(f"  Label {lab}: n_nodes mean={sub['n_nodes'].mean():.1f} "
                  f"std={sub['n_nodes'].std():.1f}")
    if "scaffold_id" in df.columns:
        print(f"  Unique scaffolds: {df['scaffold_id'].nunique()}")


def scaffold_leakage_report(task_dfs: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    all_rows = []
    for task_name, df in task_dfs.items():
        if "scaffold_id" not in df.columns:
            continue
        lab_col = "clean_label" if "clean_label" in df.columns else "label"
        scaff_stats = df.groupby("scaffold_id").apply(
            lambda g: pd.Series({
                "n_graphs": len(g),
                "n_label_states": g[lab_col].nunique(),
                "n_motif_combos": g.groupby(["has_A", "has_B"]).ngroups,
                "label_mean": g[lab_col].mean(),
            })
        ).reset_index()
        scaff_stats["task"] = task_name
        n_total = len(scaff_stats)
        n_single = (scaff_stats["n_label_states"] == 1).sum()
        print(f"\n  [SCAFFOLD LEAKAGE] {task_name}")
        print(f"    Total scaffolds: {n_total}")
        print(f"    Single-label scaffolds: {n_single} "
              f"({n_single/max(n_total,1):.1%})")
        all_rows.append(scaff_stats)
    return pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()


def artifact_diagnostics(df: pd.DataFrame, task_name: str) -> Dict:
    """Check if labels are predictable from nuisance graph-level features."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_score

    print(f"\n  [ARTIFACT CHECK] {task_name}")
    lab_col = "clean_label" if "clean_label" in df.columns else "label"
    y = df[lab_col].values

    X = df[["n_nodes"]].values if "n_nodes" in df.columns else np.zeros((len(df), 1))

    # Scaffold-only predictiveness
    scaffold_auroc = 0.5
    if "scaffold_id" in df.columns:
        from sklearn.preprocessing import LabelEncoder
        scaff_enc = LabelEncoder().fit_transform(df["scaffold_id"].values).reshape(-1, 1)
        try:
            sc = cross_val_score(LogisticRegression(max_iter=500, random_state=42),
                                 scaff_enc, y, cv=5, scoring="roc_auc")
            scaffold_auroc = sc.mean()
        except Exception:
            pass

    try:
        scores = cross_val_score(LogisticRegression(max_iter=500, random_state=42),
                                 X, y, cv=5, scoring="roc_auc")
        mean_auc, std_auc = scores.mean(), scores.std()
    except Exception:
        mean_auc, std_auc = 0.5, 0.0

    status = "OK (no artifact)" if mean_auc < 0.6 else "ARTIFACT RISK"
    print(f"    Nuisance-feature AUROC: {mean_auc:.3f} +/- {std_auc:.3f}  {status}")
    print(f"    Scaffold-only AUROC:    {scaffold_auroc:.3f}")
    return {"task": task_name, "nuisance_auroc_mean": mean_auc,
            "nuisance_auroc_std": std_auc, "scaffold_auroc": scaffold_auroc,
            "status": status}


# ═══════════════════════════════════════════════════════════════════════════
# §9  MODEL
# ═══════════════════════════════════════════════════════════════════════════
# DumplingGNN and MPNN are defined in the framework file
# (framework_generalized.py) and available in the Colab namespace.
# They are NOT redefined here to avoid shadowing.


# ═══════════════════════════════════════════════════════════════════════════
# §10  TRAINING & EVALUATION  (adapted for graph-native data)
# ═══════════════════════════════════════════════════════════════════════════

def stratified_split(
    df: pd.DataFrame,
    data_list: List[Data],
    train_f: float = 0.70,
    val_f: float = 0.15,
    seed: int = 42,
) -> Dict[str, Any]:
    """Scaffold-disjoint split when scaffold_id is available."""
    rng = np.random.RandomState(seed)

    # Build index mapping: df row -> position in data_list
    # data_list is aligned with df by construction
    n = min(len(df), len(data_list))

    if "scaffold_id" in df.columns:
        scaffolds = df["scaffold_id"].unique().tolist()
        rng.shuffle(scaffolds)
        n_s = len(scaffolds)
        s_train = set(scaffolds[:int(n_s * train_f)])
        s_val = set(scaffolds[int(n_s * train_f):int(n_s * (train_f + val_f))])
        s_test = set(scaffolds[int(n_s * (train_f + val_f)):])

        splits = {"train": ([], []), "val": ([], []), "test": ([], [])}
        for i in range(n):
            sc = df.iloc[i]["scaffold_id"]
            if sc in s_train:
                key = "train"
            elif sc in s_val:
                key = "val"
            else:
                key = "test"
            splits[key][0].append(data_list[i])
            splits[key][1].append(i)

        overlap = s_train & s_test
        print(f"  Scaffold split: train_scaff={len(s_train)} val={len(s_val)} "
              f"test={len(s_test)} overlap={len(overlap)}")
        return {f"{k}": splits[k][0] for k in splits} | \
               {f"{k}_rows": splits[k][1] for k in splits}

    # Fallback: stratified by struct_class
    strat_col = "struct_class" if "struct_class" in df.columns else "label"
    result = {k: [] for k in ["train","val","test","train_rows","val_rows","test_rows"]}
    for cls_val in df[strat_col].unique():
        cls_idx = df[df[strat_col] == cls_val].index.tolist()
        cls_idx = [i for i in cls_idx if i < n]
        rng.shuffle(cls_idx)
        n_tr = int(len(cls_idx) * train_f)
        n_va = int(len(cls_idx) * val_f)
        for i in cls_idx[:n_tr]:
            result["train"].append(data_list[i]); result["train_rows"].append(i)
        for i in cls_idx[n_tr:n_tr+n_va]:
            result["val"].append(data_list[i]); result["val_rows"].append(i)
        for i in cls_idx[n_tr+n_va:]:
            result["test"].append(data_list[i]); result["test_rows"].append(i)
    return result


def train_model(train_data, val_data, task_name, epochs=200, lr=1e-3,
                batch_size=64, patience=25, device=None):
    if device is None: device = CFG.DEVICE
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size)
    model = DumplingGNN(input_dim=CFG.INPUT_DIM, hidden_channels=CFG.HIDDEN,
                        dropout=CFG.DROPOUT).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    # Compute pos_weight from class balance for BCEWithLogitsLoss
    labels = [d.y.item() if isinstance(d.y, torch.Tensor) else d.y
              for d in train_data]
    n_pos = sum(1 for l in labels if l > 0.5)
    n_neg = len(labels) - n_pos
    if n_pos > 0 and n_neg > 0:
        pw = torch.tensor([n_neg / n_pos], dtype=torch.float32, device=device)
        print(f"  [{task_name}] Class balance: {n_neg} neg / {n_pos} pos, "
              f"pos_weight={pw.item():.2f}")
    else:
        pw = torch.tensor([1.0], device=device)
    crit = nn.BCEWithLogitsLoss(pos_weight=pw)

    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, "max", patience=8, factor=0.5, min_lr=1e-5)
    best_auc, best_state, wait = 0.0, None, 0
    history = []
    for epoch in range(1, epochs + 1):
        model.train(); tl = 0
        for b in train_loader:
            b = b.to(device); opt.zero_grad()
            loss = crit(model(b), b.y.view(-1)); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tl += loss.item() * b.num_graphs
        al = tl / len(train_data)
        model.eval(); yy, pp = [], []
        with torch.no_grad():
            for b in val_loader:
                b = b.to(device)
                pp.extend(torch.sigmoid(model(b)).cpu().tolist())
                yy.extend(b.y.view(-1).cpu().tolist())
        va = roc_auc_score(yy, pp) if len(set(yy)) > 1 else 0.5
        sched.step(va)
        history.append({"epoch": epoch, "train_loss": al, "val_auc": va})
        if va > best_auc: best_auc = va; best_state = copy.deepcopy(model.state_dict()); wait = 0
        else: wait += 1
        if epoch % 20 == 0 or epoch == 1:
            cur_lr = opt.param_groups[0]["lr"]
            print(f"  [{task_name}] Ep {epoch:3d}  loss={al:.4f}  val_auc={va:.4f}  "
                  f"best={best_auc:.4f}  lr={cur_lr:.1e}")
        if wait >= patience:
            print(f"  [{task_name}] Early stop ep {epoch}"); break
    model.load_state_dict(best_state)
    p = os.path.join(CFG.DATA_DIR, f"model_{task_name}.pth")
    torch.save(best_state, p)
    print(f"  [{task_name}] Best val AUC: {best_auc:.4f}")
    return model, history


def evaluate_model(model, test_data, device=None):
    if device is None: device = CFG.DEVICE
    loader = DataLoader(test_data, batch_size=256)
    model.eval(); yy, pp = [], []
    with torch.no_grad():
        for b in loader:
            b = b.to(device)
            pp.extend(torch.sigmoid(model(b)).cpu().tolist())
            yy.extend(b.y.view(-1).cpu().tolist())
    y, p = np.array(yy), np.array(pp)

    # Find optimal threshold from ROC curve (maximizes Youden's J)
    if len(set(yy)) > 1:
        auroc = roc_auc_score(y, p)
        fpr, tpr, thresholds = roc_curve(y, p)
        j_scores = tpr - fpr
        best_idx = np.argmax(j_scores)
        best_thresh = float(thresholds[best_idx])
    else:
        auroc = 0.5
        best_thresh = 0.5

    preds = (p >= best_thresh).astype(int)
    return {"auroc": auroc,
            "accuracy": accuracy_score(y, preds),
            "precision": precision_score(y, preds, zero_division=0),
            "recall": recall_score(y, preds, zero_division=0),
            "f1": f1_score(y, preds, zero_division=0),
            "threshold": best_thresh}, y, p


def scaffold_held_out_eval(model, dataset_info, df, device):
    """Evaluate on test graphs whose scaffold was not seen in training."""
    train_rows = dataset_info["train_rows"]
    test_rows = dataset_info["test_rows"]
    train_scaffolds = set(df.iloc[train_rows]["scaffold_id"].unique())
    test_scaffolds = set(df.iloc[test_rows]["scaffold_id"].unique())
    overlap = train_scaffolds & test_scaffolds
    novel = test_scaffolds - train_scaffolds
    result = {
        "n_test_scaffolds": len(test_scaffolds),
        "n_train_scaffolds": len(train_scaffolds),
        "n_overlap": len(overlap),
        "overlap_pct": len(overlap) / max(len(test_scaffolds), 1) * 100,
        "n_novel_mols": sum(1 for r in test_rows
                           if df.iloc[r]["scaffold_id"] in novel),
        "n_test_mols": len(test_rows),
    }
    novel_data = [dataset_info["test"][i] for i, r in enumerate(test_rows)
                  if df.iloc[r]["scaffold_id"] in novel]
    if len(novel_data) >= 5:
        met, _, _ = evaluate_model(model, novel_data, device)
        result["novel_auroc"] = met["auroc"]
    else:
        result["novel_auroc"] = None
    return result


# ═══════════════════════════════════════════════════════════════════════════
# §10b  STANDARDIZED FIDELITY METRICS  (graph-general)
# ═══════════════════════════════════════════════════════════════════════════

def get_target_logit(
    model: nn.Module,
    data: Data,
    device: torch.device,
    class_mode: str = "positive",
    target_class: Optional[int] = None,
) -> float:
    """Extract the target-class logit from the model's output.

    Parameters
    ----------
    model : nn.Module
        Trained GNN.
    data : PyG Data
        Single graph (no batch dim required; added automatically).
    device : torch.device
    class_mode : str
        "positive"  - use the positive-class logit (binary default).
        "predicted" - use the logit of the class the model predicts.
        "target"    - use the logit of the class given by target_class.
    target_class : int or None
        Required when class_mode == "target".

    Returns
    -------
    float  Scalar logit value.
    """
    data_c = data.clone().to(device)
    if not hasattr(data_c, "batch") or data_c.batch is None:
        data_c.batch = torch.zeros(data_c.x.size(0), dtype=torch.long,
                                   device=device)
    model.eval()
    with torch.no_grad():
        out = model(data_c)

    # Reshape to 2D: [1, C]
    if out.dim() == 0:
        out = out.unsqueeze(0).unsqueeze(0)
    elif out.dim() == 1:
        out = out.unsqueeze(0)
    # out is now [batch, C] where C >= 1

    n_classes = out.shape[-1]

    if n_classes == 1:
        # Binary single-logit: the logit IS the positive-class logit
        return float(out[0, 0].item())

    # Multi-class (C >= 2)
    if class_mode == "positive" and n_classes == 2:
        return float(out[0, 1].item())
    elif class_mode == "predicted":
        pred = int(out[0].argmax().item())
        return float(out[0, pred].item())
    elif class_mode == "target":
        if target_class is None:
            raise ValueError("target_class required when class_mode='target'")
        return float(out[0, target_class].item())
    elif class_mode == "positive":
        # For C > 2 with mode "positive", default to predicted
        pred = int(out[0].argmax().item())
        return float(out[0, pred].item())
    else:
        raise ValueError(f"Unknown class_mode: {class_mode}")


def mask_node_features(
    data: Data,
    keep_nodes: Optional[Set[int]] = None,
    drop_nodes: Optional[Set[int]] = None,
    baseline: Optional[torch.Tensor] = None,
) -> Data:
    """Return a deep-copied Data object with node features masked.

    Exactly one of keep_nodes or drop_nodes must be provided.
    Edge index and all other attributes are preserved unchanged.

    Parameters
    ----------
    data : PyG Data
        Original graph (not mutated).
    keep_nodes : set of int or None
        If provided, nodes NOT in this set are replaced by baseline.
    drop_nodes : set of int or None
        If provided, nodes IN this set are replaced by baseline.
    baseline : torch.Tensor or None
        Replacement feature vector.  Defaults to zeros.

    Returns
    -------
    PyG Data with masked features.
    """
    if (keep_nodes is None) == (drop_nodes is None):
        raise ValueError("Exactly one of keep_nodes or drop_nodes must be provided")

    data_new = data.clone()
    n = data_new.x.size(0)
    feat_dim = data_new.x.size(1)

    if baseline is None:
        bl = torch.zeros(feat_dim, dtype=data_new.x.dtype, device=data_new.x.device)
    else:
        bl = baseline.to(device=data_new.x.device, dtype=data_new.x.dtype)

    # Determine which nodes survive after masking
    if keep_nodes is not None:
        keep_set = {int(k) for k in keep_nodes if 0 <= int(k) < n}
    else:
        drop_set = {int(d) for d in drop_nodes if 0 <= int(d) < n}
        keep_set = set(range(n)) - drop_set

    # Zero features on dropped nodes
    for i in range(n):
        if i not in keep_set:
            data_new.x[i] = bl

    # HARD masking: drop edges that touch a masked node, matching the paper
    # claim in §5.1 ("zeroing features and removing edges"). Mirrors the
    # benchmark-side `_mask_nodes` in cell 13.
    if hasattr(data_new, "edge_index") and data_new.edge_index is not None:
        ei = data_new.edge_index
        keep_mask = torch.zeros(n, dtype=torch.bool, device=ei.device)
        for nd in keep_set:
            keep_mask[nd] = True
        edge_mask = keep_mask[ei[0]] & keep_mask[ei[1]]
        data_new.edge_index = ei[:, edge_mask]
        if hasattr(data_new, "edge_attr") and data_new.edge_attr is not None:
            data_new.edge_attr = data_new.edge_attr[edge_mask]

    return data_new


def explanation_to_node_set(
    explanation: Any,
) -> Tuple[Set[int], List[int]]:
    """Convert an explanation to a node set.

    Handles:
      - A set/list/tuple of node indices.
      - A list of groups, where each group is a set/list/tuple of indices.

    Returns
    -------
    (node_set, sorted_node_list)
    """
    if not explanation:
        return set(), []

    # Check if it's a list of groups (list of iterables of ints)
    first = next(iter(explanation))
    if isinstance(first, (set, list, tuple, frozenset)):
        # List of groups: union all
        node_set: Set[int] = set()
        for group in explanation:
            node_set.update(group)
    else:
        # Flat iterable of node indices
        node_set = set(int(x) for x in explanation)

    return node_set, sorted(node_set)


def deletion_fidelity(
    model: nn.Module,
    data: Data,
    explanation: Any,
    device: torch.device,
    class_mode: str = "positive",
    target_class: Optional[int] = None,
    baseline: Optional[torch.Tensor] = None,
) -> Tuple[float, Dict[str, Any]]:
    """Compute deletion fidelity (necessity).

    fid+ = s_full - s_drop

    where s_full is the target logit on the original graph and s_drop
    is the logit after masking all explanation nodes to baseline.

    A high fid+ means removing the explanation hurts the prediction,
    so the explanation captures necessary structure.

    Parameters
    ----------
    model, data, device : standard
    explanation : node set or list of groups
    class_mode, target_class : logit selection
    baseline : mask vector

    Returns
    -------
    (fid_plus, info_dict)
    """
    node_set, _ = explanation_to_node_set(explanation)
    n_nodes = data.x.size(0)

    s_full = get_target_logit(model, data, device, class_mode, target_class)

    if len(node_set) == 0:
        return 0.0, {"s_full": s_full, "s_drop": s_full,
                      "num_selected": 0, "num_nodes": n_nodes}

    if len(node_set) >= n_nodes:
        data_drop = mask_node_features(data, keep_nodes=set(), baseline=baseline)
    else:
        data_drop = mask_node_features(data, drop_nodes=node_set, baseline=baseline)
    s_drop = get_target_logit(model, data_drop, device, class_mode, target_class)

    fid_plus = s_full - s_drop
    return fid_plus, {"s_full": s_full, "s_drop": s_drop,
                      "num_selected": len(node_set), "num_nodes": n_nodes}


def inverse_fidelity(
    model: nn.Module,
    data: Data,
    explanation: Any,
    device: torch.device,
    class_mode: str = "positive",
    target_class: Optional[int] = None,
    baseline: Optional[torch.Tensor] = None,
) -> Tuple[float, Dict[str, Any]]:
    """Compute inverse fidelity (sufficiency).

    fid- = s_full - s_keep

    where s_keep is the target logit when ONLY the explanation nodes
    retain their features and all others are masked.

    A low fid- means the explanation alone is sufficient to reproduce
    the prediction.

    Parameters
    ----------
    Same as deletion_fidelity.

    Returns
    -------
    (fid_minus, info_dict)
    """
    node_set, _ = explanation_to_node_set(explanation)
    n_nodes = data.x.size(0)

    s_full = get_target_logit(model, data, device, class_mode, target_class)

    if len(node_set) == 0:
        data_keep = mask_node_features(data, keep_nodes=set(), baseline=baseline)
    elif len(node_set) >= n_nodes:
        return 0.0, {"s_full": s_full, "s_keep": s_full,
                      "num_selected": n_nodes, "num_nodes": n_nodes}
    else:
        data_keep = mask_node_features(data, keep_nodes=node_set, baseline=baseline)
    s_keep = get_target_logit(model, data_keep, device, class_mode, target_class)

    fid_minus = s_full - s_keep
    return fid_minus, {"s_full": s_full, "s_keep": s_keep,
                       "num_selected": len(node_set), "num_nodes": n_nodes}


def sparsity_score(data: Data, explanation: Any) -> float:
    """Compute explanation sparsity.

    sparsity = 1 - |E*| / |V|

    where E* is the selected node set and V is the full node set.
    Returns a float in [0, 1].
    """
    node_set, _ = explanation_to_node_set(explanation)
    n_nodes = data.x.size(0)
    if n_nodes == 0:
        return 1.0
    return 1.0 - len(node_set) / n_nodes


def harmonic_fidelity(
    fid_plus: float,
    fid_minus: float,
    num_selected: int,
    num_nodes: int,
    clamp_normalized: bool = True,
) -> Tuple[float, Dict[str, float]]:
    """Compute GStarX-style harmonic fidelity score.

    Combines deletion fidelity (necessity) and inverse fidelity (sufficiency)
    with a sparsity-aware normalization:

        m1 = fid_plus  * (1 - |E*|/|V|)     # necessity * sparsity
        m2 = fid_minus * (|E*|/|V|)          # insufficiency * density

    Then the harmonic combination:

        h = ((1 + m1) * (1 - m2)) / (2 + m1 - m2)

    Note: This formula assumes m1 and m2 are in a range where the
    denominator is positive.  When fid_plus or fid_minus take extreme
    values, the normalized scores may fall outside [0,1].  The
    clamp_normalized flag (default True) clips m1, m2 to [-1, 1]
    to ensure numerical stability.

    Parameters
    ----------
    fid_plus : float
        Deletion fidelity (s_full - s_drop).
    fid_minus : float
        Inverse fidelity (s_full - s_keep).
    num_selected : int
        |E*|, number of selected nodes.
    num_nodes : int
        |V|, total number of nodes.
    clamp_normalized : bool
        If True, clamp m1, m2 to [-1, 1] for numerical safety.

    Returns
    -------
    (h, {"m1": m1, "m2": m2})
    """
    if num_nodes == 0:
        return 0.0, {"m1": 0.0, "m2": 0.0}

    ratio = num_selected / num_nodes
    m1 = fid_plus * (1.0 - ratio)
    m2 = fid_minus * ratio

    if clamp_normalized:
        m1 = max(-1.0, min(1.0, m1))
        m2 = max(-1.0, min(1.0, m2))

    denom = 2.0 + m1 - m2
    if abs(denom) < 1e-12:
        h = 0.0
    else:
        h = ((1.0 + m1) * (1.0 - m2)) / denom

    return h, {"m1": m1, "m2": m2}


# ── Budget control utilities ─────────────────────────────────────────────

def select_topk_nodes(
    node_scores: np.ndarray,
    k: int,
) -> Set[int]:
    """Select the top-k nodes by absolute score.

    Parameters
    ----------
    node_scores : array of shape (n_nodes,)
        Per-node importance scores (e.g., main effects from M diagonal).
    k : int
        Number of nodes to select.

    Returns
    -------
    Set of selected node indices.
    """
    k = max(0, min(k, len(node_scores)))
    if k == 0:
        return set()
    ranked = np.argsort(-np.abs(node_scores))
    return set(int(ranked[i]) for i in range(k))


def select_topfrac_nodes(
    node_scores: np.ndarray,
    fraction: float,
) -> Set[int]:
    """Select the top fraction of nodes by absolute score.

    Parameters
    ----------
    node_scores : array of shape (n_nodes,)
    fraction : float in [0, 1]

    Returns
    -------
    Set of selected node indices.
    """
    fraction = max(0.0, min(1.0, fraction))
    k = max(1, int(math.ceil(len(node_scores) * fraction)))
    return select_topk_nodes(node_scores, k)


def select_topk_groups(
    partition: List[List[int]],
    group_scores: List[float],
    k: int,
) -> Tuple[Set[int], List[int]]:
    """Select the top-k groups by absolute score, preserving group integrity.

    Parameters
    ----------
    partition : list of list of int
        Groups of node indices.
    group_scores : list of float
        Score for each group (e.g., group_agg["group_values"]).
    k : int
        Number of groups to select.

    Returns
    -------
    (selected_node_set, selected_group_indices)
    """
    k = max(0, min(k, len(partition)))
    if k == 0:
        return set(), []
    ranked = sorted(range(len(group_scores)),
                    key=lambda i: abs(group_scores[i]), reverse=True)
    selected_gi = ranked[:k]
    node_set: Set[int] = set()
    for gi in selected_gi:
        node_set.update(partition[gi])
    return node_set, sorted(selected_gi)


def select_topfrac_groups(
    partition: List[List[int]],
    group_scores: List[float],
    fraction: float,
    num_total_nodes: int,
) -> Tuple[Set[int], List[int]]:
    """Select top groups until the fraction of total nodes is reached.

    Groups are added in descending order of |score| until the selected
    node set covers at least fraction * num_total_nodes nodes.  Group
    integrity is always preserved (no partial groups).

    Parameters
    ----------
    partition : list of list of int
    group_scores : list of float
    fraction : float in [0, 1]
    num_total_nodes : int

    Returns
    -------
    (selected_node_set, selected_group_indices)
    """
    fraction = max(0.0, min(1.0, fraction))
    budget = max(1, int(math.ceil(num_total_nodes * fraction)))

    ranked = sorted(range(len(group_scores)),
                    key=lambda i: abs(group_scores[i]), reverse=True)
    node_set: Set[int] = set()
    selected_gi: List[int] = []
    for gi in ranked:
        if len(node_set) >= budget:
            break
        node_set.update(partition[gi])
        selected_gi.append(gi)
    return node_set, sorted(selected_gi)


def compute_fidelity_metrics(
    model: nn.Module,
    data: Data,
    explanation: Any,
    device: torch.device,
    class_mode: str = "positive",
    target_class: Optional[int] = None,
    baseline: Optional[torch.Tensor] = None,
    clamp_normalized: bool = True,
) -> Dict[str, float]:
    """Compute all four standardized fidelity metrics for one explanation.

    Logit-space fid+/fid- are primary metrics (unbounded, more informative).
    Harmonic fidelity is computed in probability space because the GStarX
    formula assumes values in [0,1] — using logits causes clamping artifacts.

    Returns a flat dict with:
      fid_plus, fid_minus, sufficiency (1 - fid_minus), sparsity,
      prob_fid_plus, prob_fid_minus (probability-space versions),
      harmonic_fidelity, m1, m2, s_full, s_drop, s_keep,
      num_selected, num_nodes.
    """
    node_set, _ = explanation_to_node_set(explanation)
    n_nodes = data.x.size(0)
    n_sel = len(node_set)

    fid_plus, del_info = deletion_fidelity(
        model, data, explanation, device, class_mode, target_class, baseline)
    fid_minus, inv_info = inverse_fidelity(
        model, data, explanation, device, class_mode, target_class, baseline)
    spar = sparsity_score(data, explanation)

    # Probability-space fidelity for harmonic computation.
    # sigmoid maps logits to [0,1], so differences are bounded in [-1,1].
    def _sig(z):
        return 1.0 / (1.0 + math.exp(-max(min(z, 30), -30)))

    s_full = del_info["s_full"]
    s_drop = del_info["s_drop"]
    s_keep = inv_info["s_keep"]
    p_full = _sig(s_full)
    p_drop = _sig(s_drop)
    p_keep = _sig(s_keep)
    prob_fid_plus = p_full - p_drop    # in [-1, 1]
    prob_fid_minus = p_full - p_keep   # in [-1, 1]

    # Harmonic fidelity uses probability-space values
    h, hm = harmonic_fidelity(prob_fid_plus, prob_fid_minus, n_sel, n_nodes,
                              clamp_normalized=clamp_normalized)

    return {
        "fid_plus": fid_plus,
        "fid_minus": fid_minus,
        "sufficiency": 1.0 - fid_minus,
        "sparsity": spar,
        "prob_fid_plus": prob_fid_plus,
        "prob_fid_minus": prob_fid_minus,
        "harmonic_fidelity": h,
        "m1": hm["m1"],
        "m2": hm["m2"],
        "s_full": s_full,
        "s_drop": s_drop,
        "s_keep": s_keep,
        "num_selected": n_sel,
        "num_nodes": n_nodes,
    }


def evaluate_explanations_batch(
    model: nn.Module,
    dataset: List[Data],
    explanations: List[Any],
    device: torch.device,
    class_mode: str = "positive",
    target_class_map: Optional[List[int]] = None,
    baseline: Optional[torch.Tensor] = None,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Batch evaluation of explanation fidelity metrics.

    Parameters
    ----------
    model : nn.Module
    dataset : list of PyG Data
    explanations : list of explanations (aligned with dataset)
        Each element is either a node set or a list of groups.
    device : torch.device
    class_mode : str
    target_class_map : optional list of per-example target classes
    baseline : optional mask vector

    Returns
    -------
    (per_example_df, summary_dict)
        summary_dict has mean/std/median for each metric.
    """
    records = []
    for i, (data, expl) in enumerate(zip(dataset, explanations)):
        tc = target_class_map[i] if target_class_map is not None else None
        try:
            metrics = compute_fidelity_metrics(
                model, data, expl, device, class_mode, tc, baseline)
        except Exception as e:
            logger.warning(f"Fidelity eval failed for graph {i}: {e}")
            continue
        metrics["graph_idx"] = i
        records.append(metrics)

    df = pd.DataFrame(records)
    summary: Dict[str, Any] = {"n_evaluated": len(df)}

    if not df.empty:
        for col in ["fid_plus", "fid_minus", "sufficiency", "sparsity",
                     "harmonic_fidelity"]:
            if col in df.columns:
                summary[f"{col}_mean"] = float(df[col].mean())
                summary[f"{col}_std"] = float(df[col].std())
                summary[f"{col}_median"] = float(df[col].median())

    return df, summary


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# §11  VERIFICATION METRICS  (graph-native)
# ═══════════════════════════════════════════════════════════════════════════

def compute_loc_main(phi: np.ndarray, motif_positions: Set[int]) -> float:
    """Fraction of total absolute attribution on motif nodes."""
    total = np.sum(np.abs(phi))
    if total < 1e-12:
        return 0.0
    return float(sum(abs(phi[p]) for p in motif_positions) / total)


def find_best_group_for_motif(
    partition: List[List[int]],
    heavy_idx: List[int],
    motif_positions: Set[int],
) -> int:
    """Find which group in the partition best overlaps with motif positions."""
    g2p = {g: p for p, g in enumerate(heavy_idx)}
    best_gi, best_ov = -1, 0
    for gi, group in enumerate(partition):
        ov = len({g2p.get(g, -1) for g in group} & motif_positions)
        if ov > best_ov:
            best_ov = ov; best_gi = gi
    return best_gi


def compute_interaction_dominance(group_agg, gi_A, gi_B):
    if gi_A < 0 or gi_B < 0: return 0.0
    I = abs(group_agg["cross_scores"][gi_A][gi_B])
    VA = abs(group_agg["group_values"][gi_A])
    VB = abs(group_agg["group_values"][gi_B])
    return I / (I + VA + VB + 1e-8)


def compute_interaction_rank(group_agg, gi_A, gi_B):
    if gi_A < 0 or gi_B < 0: return -1
    cross = group_agg["cross_scores"]
    K = len(cross)
    target = abs(cross[gi_A][gi_B])
    all_c = sorted([abs(cross[i][j]) for i in range(K) for j in range(i+1, K)],
                   reverse=True)
    for r, v in enumerate(all_c, 1):
        if abs(v - target) < 1e-12: return r
    return len(all_c)


def bootstrap_ci(data, n_bootstrap=1000, ci=0.95, seed=42):
    rng = np.random.RandomState(seed)
    means = [np.mean(rng.choice(data, size=len(data), replace=True))
             for _ in range(n_bootstrap)]
    lo = np.percentile(means, (1 - ci) / 2 * 100)
    hi = np.percentile(means, (1 + ci) / 2 * 100)
    return float(lo), float(hi)


def _eval_coalition_graph(model, sg, active_nodes, device):
    """Score a coalition for a SyntheticGraph (masking-based)."""
    active_total = set(active_nodes)
    # For SyntheticGraph, heavy_to_h is empty, so no H expansion needed
    x_masked = sg.x.clone()
    for i in range(sg.n_total):
        if i not in active_total:
            x_masked[i] = sg.baseline
    data = Data(x=x_masked, edge_index=sg.edge_index)
    batch = Batch.from_data_list([data]).to(device)
    with torch.no_grad():
        model.eval()
        z = model(batch)
        # Handle scalar [1] (DumplingGNN) and [1,C] (multi-class) outputs
        if z.dim() == 0:
            return float(z.item())
        if z.dim() == 1:
            return float(z[0].item())
        if z.shape[-1] == 1:
            return float(z[0, 0].item())
        if z.shape[-1] == 2:
            return float(z[0, 1].item())
        return float(z[0].max().item())


def compute_deletion_faithfulness(
    model, sg, partition, group_agg, device, n_random=20,
):
    """Group-level deletion faithfulness in signed-logit space."""
    K = len(partition)
    empty = {"del_auc": 0., "del_auc_rand": 0., "del_auc_rand_std": 0.,
             "ins_auc": 0., "ins_auc_rand": 0., "ins_auc_rand_std": 0.,
             "del_diff_ci_lo": 0., "del_diff_ci_hi": 0.,
             "p0": 0., "p_final": 0., "K_groups": 0}
    if K == 0: return empty

    all_nodes = set(sg.heavy_idx)
    full_z = _eval_coalition_graph(model, sg, all_nodes, device)
    base_z = _eval_coalition_graph(model, sg, set(), device)
    p0 = 1.0 / (1.0 + math.exp(-max(min(full_z, 30), -30)))

    sign = 1.0 if full_z >= 0 else -1.0
    def _signed(z): return sign * z
    sz_full = _signed(full_z)
    sz_base = _signed(base_z)

    def _del_curve(order):
        active = set(all_nodes)
        vals = [sz_full]
        for gi in order:
            for a in partition[gi]:
                active.discard(a)
            vals.append(_signed(_eval_coalition_graph(model, sg, active, device)))
        return vals

    def _ins_curve(order):
        active = set()
        vals = [sz_base]
        for gi in order:
            for a in partition[gi]:
                active.add(a)
            vals.append(_signed(_eval_coalition_graph(model, sg, active, device)))
        return vals

    order_expl = sorted(range(K), key=lambda i: abs(group_agg["group_values"][i]),
                        reverse=True)
    szvals_del = _del_curve(order_expl)
    x_ax = np.linspace(0, 1, len(szvals_del))
    del_auc = float(np.trapz([sz_full - s for s in szvals_del], x_ax))

    ins_auc = 0.0
    if CFG.INSERTION_ENABLED:
        szvals_ins = _ins_curve(order_expl)
        x_ins = np.linspace(0, 1, len(szvals_ins))
        ins_auc = float(np.trapz([s - sz_base for s in szvals_ins], x_ins))

    rng = np.random.RandomState(42)
    rand_del, rand_ins = [], []
    for _ in range(n_random):
        ro = rng.permutation(K).tolist()
        sv = _del_curve(ro)
        x_r = np.linspace(0, 1, len(sv))
        rand_del.append(float(np.trapz([sz_full - s for s in sv], x_r)))
        if CFG.INSERTION_ENABLED:
            svi = _ins_curve(ro)
            rand_ins.append(float(np.trapz([s - sz_base for s in svi],
                                           np.linspace(0, 1, len(svi)))))

    del_rand_mean = float(np.mean(rand_del))
    diffs = [del_auc - ra for ra in rand_del]
    ci_lo, ci_hi = bootstrap_ci(diffs, CFG.DEL_BOOTSTRAP_N) if len(diffs) >= 5 else (0., 0.)

    z_final = szvals_del[-1] * sign
    p_final = 1.0 / (1.0 + math.exp(-max(min(z_final, 30), -30)))

    return {
        "del_auc": del_auc, "del_auc_rand": del_rand_mean,
        "del_auc_rand_std": float(np.std(rand_del)),
        "ins_auc": ins_auc,
        "ins_auc_rand": float(np.mean(rand_ins)) if rand_ins else 0.,
        "ins_auc_rand_std": float(np.std(rand_ins)) if rand_ins else 0.,
        "del_diff_ci_lo": ci_lo, "del_diff_ci_hi": ci_hi,
        "p0": p0, "p_final": p_final, "K_groups": K,
    }


def deletion_sanity_check(results_df):
    confident = results_df[abs(results_df["p0"] - 0.5) > 0.3]
    if len(confident) < 3:
        print("  [DEL SANITY] Too few confident samples")
        return
    m_e = confident["del_auc"].mean()
    m_r = confident["del_auc_rand"].mean()
    if m_e > m_r:
        print(f"  [DEL SANITY] PASS: expl={m_e:.4f} > rand={m_r:.4f} (n={len(confident)})")
    else:
        print(f"  [DEL SANITY] FAIL: expl={m_e:.4f} <= rand={m_r:.4f}")


# ═══════════════════════════════════════════════════════════════════════════
# §12  INTERACTION QUALITY METRICS
# ═══════════════════════════════════════════════════════════════════════════

def compute_synergy_sign_rate(results_df):
    both = results_df[(results_df["has_A"] == 1) & (results_df["has_B"] == 1)]
    if both.empty: return {}
    lab_col = "clean_label" if "clean_label" in both.columns else "label"
    out = {}
    for lab in both[lab_col].unique():
        sub = both[both[lab_col] == lab]
        if len(sub):
            out[f"synergy_positive_rate_label{lab}"] = float((sub["I_AB"] > 0).mean())
            out[f"n_both_label{lab}"] = len(sub)
    return out


def compute_interaction_separability(results_df):
    both = results_df[(results_df["has_A"] == 1) & (results_df["has_B"] == 1)]
    if both.empty: return {"interaction_sep_auroc": float("nan"), "interaction_sep_n": 0}
    lab_col = "clean_label" if "clean_label" in both.columns else "label"
    if both[lab_col].nunique() < 2:
        return {"interaction_sep_auroc": float("nan"), "interaction_sep_n": len(both)}
    try:
        return {"interaction_sep_auroc": roc_auc_score(both[lab_col], both["I_AB"]),
                "interaction_sep_n": len(both)}
    except Exception:
        return {"interaction_sep_auroc": float("nan"), "interaction_sep_n": len(both)}


def compute_distance_correlation(results_df):
    both = results_df[(results_df["has_A"]==1) & (results_df["has_B"]==1)]
    if "dist_AB" not in both.columns or both.empty:
        return {"spearman_I_AB_dist": float("nan"), "spearman_pvalue": float("nan")}
    valid = both[both["dist_AB"] >= 0]
    if len(valid) < 5:
        return {"spearman_I_AB_dist": float("nan"), "spearman_pvalue": float("nan")}
    rho, pval = spearmanr(valid["I_AB"], valid["dist_AB"])
    return {"spearman_I_AB_dist": float(rho), "spearman_pvalue": float(pval)}


# ═══════════════════════════════════════════════════════════════════════════
# §13  VERIFICATION LOOP
# ═══════════════════════════════════════════════════════════════════════════

def verify_explanations(
    model, test_data: List[Data], test_df: pd.DataFrame,
    task_name: str, device, n_explain: int = 30,
    n_permutations: int = 100, sa_iterations: int = 100,
):
    """Run the explanation verification loop on graph-native data."""
    print(f"\n{'='*60}")
    print(f"VERIFICATION: {task_name}  n_perm={n_permutations}")
    print(f"{'='*60}")

    MAX_N = CFG.MAX_NODES_EXPLAIN
    valid_mask = test_df["n_nodes"] <= MAX_N
    small_df = test_df[valid_mask].copy()
    small_indices = small_df.index.tolist()

    if len(small_df) < 10:
        small_df = test_df.copy()
        small_indices = list(range(len(test_df)))

    # Balanced sampling
    toxic = small_df[small_df["label"] == 1]
    nontoxic = small_df[small_df["label"] == 0]
    n_pos = min(n_explain, len(toxic))
    n_neg = min(n_explain, len(nontoxic))
    sample_idx = (toxic.sample(n=n_pos, random_state=42).index.tolist() +
                  nontoxic.sample(n=n_neg, random_state=42).index.tolist())

    print(f"  Explaining {len(sample_idx)} ({n_pos} pos, {n_neg} neg)")
    results = []
    t_start = time.time()

    for mol_i, idx in enumerate(sample_idx):
        row = test_df.iloc[idx]
        if idx >= len(test_data):
            continue
        data = test_data[idx]
        label = int(row["label"])
        clean_label = int(row.get("clean_label", label))
        has_A = int(row.get("has_A", 0))
        has_B = int(row.get("has_B", 0))
        dist_AB = float(row.get("dist_AB", -1))

        t_mol = time.time()
        try:
            res = explain_graph(model, data, device,
                                n_permutations=n_permutations,
                                sa_iterations=sa_iterations,
                                seed=42, verbose=False)
        except Exception as e:
            logger.warning(f"Explanation failed: {e}")
            res = None
        elapsed_mol = time.time() - t_mol

        if res is None:
            print(f"  [{mol_i+1}/{len(sample_idx)}] FAIL  ({elapsed_mol:.1f}s)")
            continue

        sg = res["sg"]
        M = res["M"]
        phi = M.diagonal().copy()  # main effects from interaction matrix
        partition = res["best_partition"]
        group_agg = res["group_agg"]

        # Ground-truth motif positions (from stored metadata)
        motif_A_nodes = set(data.motif_A_nodes.tolist()) if hasattr(data, "motif_A_nodes") else set()
        motif_B_nodes = set(data.motif_B_nodes.tolist()) if hasattr(data, "motif_B_nodes") else set()
        # Convert global indices to positional indices
        g2p = {g: p for p, g in enumerate(sg.heavy_idx)}
        A_pos = {g2p[n] for n in motif_A_nodes if n in g2p}
        B_pos = {g2p[n] for n in motif_B_nodes if n in g2p}
        motif_pos = A_pos | B_pos

        loc = compute_loc_main(phi, motif_pos)

        dom, rank, I_AB_val = 0.0, -1, 0.0
        gi_A = find_best_group_for_motif(partition, sg.heavy_idx, A_pos) if A_pos else -1
        gi_B = find_best_group_for_motif(partition, sg.heavy_idx, B_pos) if B_pos else -1
        if gi_A >= 0 and gi_B >= 0 and gi_A != gi_B:
            dom = compute_interaction_dominance(group_agg, gi_A, gi_B)
            rank = compute_interaction_rank(group_agg, gi_A, gi_B)
            I_AB_val = group_agg["cross_scores"][gi_A][gi_B]

        del_dict = {"del_auc": 0., "del_auc_rand": 0., "del_auc_rand_std": 0.,
                    "ins_auc": 0., "ins_auc_rand": 0., "ins_auc_rand_std": 0.,
                    "del_diff_ci_lo": 0., "del_diff_ci_hi": 0.,
                    "p0": 0., "p_final": 0., "K_groups": 0}
        if CFG.DEL_ENABLED:
            try:
                del_dict = compute_deletion_faithfulness(
                    model, sg, partition, group_agg, device, n_random=CFG.DEL_N_RANDOM)
            except Exception as e:
                logger.warning(f"Deletion failed: {e}")

        # ── Standardized fidelity metrics (§10b) ─────────────────────
        # Select top groups covering ~50% of nodes as "the explanation"
        fid_dict = {"fid_plus": 0., "fid_minus": 0., "sufficiency": 0.,
                    "sparsity": 1., "harmonic_fidelity": 0.}
        try:
            expl_nodes_50, _ = select_topfrac_groups(
                partition, group_agg["group_values"],
                fraction=0.5, num_total_nodes=sg.n_heavy)
            fid_dict = compute_fidelity_metrics(
                model, data, expl_nodes_50, device)
        except Exception as e:
            logger.warning(f"Fidelity metrics failed: {e}")

        rec = {
            "graph_id": int(row.get("graph_id", idx)),
            "label": label, "clean_label": clean_label,
            "has_A": has_A, "has_B": has_B,
            "method": "ours", "n_permutations": n_permutations,
            "n_nodes": sg.n_heavy,
            "loc_main": loc, "dom": dom, "rank": rank, "I_AB": I_AB_val,
            **del_dict,
            **{k: v for k, v in fid_dict.items()
               if k in ("fid_plus", "fid_minus", "sufficiency", "prob_fid_plus", "prob_fid_minus",
                        "sparsity", "harmonic_fidelity")},
        }
        if dist_AB >= 0:
            rec["dist_AB"] = dist_AB
        results.append(rec)

        elapsed_total = time.time() - t_start
        avg = elapsed_total / (mol_i + 1)
        eta = avg * (len(sample_idx) - mol_i - 1)
        print(f"  [{mol_i+1}/{len(sample_idx)}]  {elapsed_mol:.1f}s  "
              f"n={sg.n_heavy}  loc={loc:.2f}  dom={dom:.2f}  "
              f"rank={rank}  del={del_dict['del_auc']:.3f}  "
              f"pfid+={fid_dict.get('prob_fid_plus',0):.3f}  "
              f"h={fid_dict.get('harmonic_fidelity',0):.3f}  (ETA {eta:.0f}s)")

    df_out = pd.DataFrame(results)
    if CFG.DEL_ENABLED and len(df_out):
        deletion_sanity_check(df_out)
    return df_out


# ═══════════════════════════════════════════════════════════════════════════
# §14  GRAD x INPUT BASELINE
# ═══════════════════════════════════════════════════════════════════════════

def grad_x_input_attribution(model, data: Data, device):
    """Compute per-node Grad x Input scores."""
    data = data.clone().to(device)
    data.x.requires_grad_(True)
    data.batch = torch.zeros(data.x.size(0), dtype=torch.long, device=device)
    model.eval()
    out = model(data)
    out.backward()
    grad = data.x.grad
    scores = (grad * data.x).abs().sum(dim=1).detach().cpu().numpy()
    return scores


def evaluate_baseline_grad(model, test_data, test_df, task_name, device,
                           n_explain=30):
    """Grad x Input baseline with atom-level deletion."""
    print(f"\n  [GRAD x INPUT] {task_name}")
    MAX_N = CFG.MAX_NODES_EXPLAIN
    valid_mask = test_df["n_nodes"] <= MAX_N
    small_df = test_df[valid_mask]
    small_idx = small_df.index.tolist()
    if len(small_df) < 10:
        small_idx = list(range(len(test_df)))
        small_df = test_df

    toxic = small_df[small_df["label"]==1]
    nontoxic = small_df[small_df["label"]==0]
    sample_idx = (toxic.sample(n=min(n_explain, len(toxic)), random_state=42).index.tolist() +
                  nontoxic.sample(n=min(n_explain, len(nontoxic)), random_state=42).index.tolist())

    results = []
    for mol_i, idx in enumerate(sample_idx):
        if idx >= len(test_data): continue
        data = test_data[idx]
        row = test_df.iloc[idx]
        try:
            scores = grad_x_input_attribution(model, data, device)
        except Exception:
            continue

        # Localization
        motif_A_nodes = set(data.motif_A_nodes.tolist()) if hasattr(data, "motif_A_nodes") else set()
        motif_B_nodes = set(data.motif_B_nodes.tolist()) if hasattr(data, "motif_B_nodes") else set()
        motif_pos = motif_A_nodes | motif_B_nodes  # already positional for SyntheticGraph
        total = np.sum(np.abs(scores))
        loc = float(sum(abs(scores[p]) for p in motif_pos if p < len(scores)) / total) if total > 1e-12 else 0.0

        # Atom-level deletion
        sg = SyntheticGraph(data.x, data.edge_index)
        partition = [[n] for n in sg.heavy_idx]
        grad_group_vals = [float(abs(scores[n])) if n < len(scores) else 0.0
                          for n in sg.heavy_idx]
        mock_agg = {"group_values": grad_group_vals,
                    "cross_scores": [[0.0]*len(partition) for _ in partition]}
        try:
            del_dict = compute_deletion_faithfulness(
                model, sg, partition, mock_agg, device, n_random=CFG.DEL_N_RANDOM)
        except Exception:
            del_dict = {"del_auc": 0., "del_auc_rand": 0., "del_auc_rand_std": 0.,
                        "ins_auc": 0., "ins_auc_rand": 0., "ins_auc_rand_std": 0.,
                        "del_diff_ci_lo": 0., "del_diff_ci_hi": 0.,
                        "p0": 0., "p_final": 0., "K_groups": 0}

        # Standardized fidelity metrics using top-50% of nodes
        fid_dict = {"fid_plus": 0., "fid_minus": 0., "sufficiency": 0.,
                    "sparsity": 1., "harmonic_fidelity": 0.}
        try:
            expl_nodes = select_topfrac_nodes(scores, 0.5)
            fid_dict = compute_fidelity_metrics(model, data, expl_nodes, device)
        except Exception:
            pass

        results.append({
            "graph_id": int(row.get("graph_id", idx)),
            "label": int(row["label"]),
            "clean_label": int(row.get("clean_label", row["label"])),
            "has_A": int(row.get("has_A", 0)), "has_B": int(row.get("has_B", 0)),
            "method": "grad_x_input", "n_permutations": 0,
            "n_nodes": len(scores),
            "loc_main": loc, "dom": 0.0, "rank": -1, "I_AB": 0.0,
            **del_dict,
            **{k: v for k, v in fid_dict.items()
               if k in ("fid_plus", "fid_minus", "sufficiency", "prob_fid_plus", "prob_fid_minus",
                        "sparsity", "harmonic_fidelity")},
        })
        if (mol_i + 1) % 10 == 0:
            print(f"    [{mol_i+1}/{len(sample_idx)}] loc={loc:.2f}")

    df = pd.DataFrame(results)
    if not df.empty:
        print(f"  [GRAD x INPUT] Done: {len(df)}, mean loc={df['loc_main'].mean():.3f}")
    return df


# ═══════════════════════════════════════════════════════════════════════════
# §14b  MULTI-BASELINE EVALUATION (GNNExplainer, PGExplainer, SubgraphX)
# ═══════════════════════════════════════════════════════════════════════════

class _WrappedDumpling(nn.Module):
    """Wrap DumplingGNN(data) -> model(x, edge_index, batch) for baselines.

    Also converts scalar logit output to 2-class logits so DIG's SubgraphX works.
    """
    def __init__(self, inner):
        super().__init__()
        self.inner = inner

    def forward(self, x, edge_index=None, batch=None, **kwargs):
        if isinstance(x, Data):
            raw = self.inner(x)
        else:
            data = Data(x=x, edge_index=edge_index)
            data.batch = batch if batch is not None else torch.zeros(
                x.size(0), dtype=torch.long, device=x.device)
            raw = self.inner(data)
        # Convert scalar logit to 2-class for DIG compatibility
        if raw.dim() == 0:
            raw = raw.unsqueeze(0)
        if raw.dim() == 1:
            # [B] scalar logits -> [B, 2] class logits
            return torch.stack([-raw, raw], dim=-1)
        if raw.shape[-1] == 1:
            return torch.cat([-raw, raw], dim=-1)
        return raw


def _synth_gnnexplainer(wrapped_model, data, device, num_classes=2):
    """Run GNNExplainer on one graph, return node scores or None."""
    from torch_geometric.explain import Explainer, GNNExplainer as PyGGNNExplainer
    try:
        data_d = data.clone().to(device)
        num_nodes = data_d.x.size(0)
        if not hasattr(data_d, "batch") or data_d.batch is None:
            data_d.batch = torch.zeros(num_nodes, dtype=torch.long, device=device)
        out = wrapped_model(data_d.x, data_d.edge_index, data_d.batch)
        target = out.argmax(dim=-1)

        explainer = Explainer(
            model=wrapped_model,
            algorithm=PyGGNNExplainer(epochs=100, lr=0.01),
            explanation_type='model',
            node_mask_type='attributes',
            edge_mask_type='object',
            model_config=dict(mode='multiclass_classification', task_level='graph',
                              return_type='raw'))
        exp = explainer(data_d.x, data_d.edge_index, batch=data_d.batch,
                        target=target, index=0)
        mask = exp.node_mask
        if mask is not None:
            return mask.abs().sum(dim=1).detach().cpu().numpy()
    except Exception as e:
        logger.warning(f"GNNExplainer failed: {e}")
    return None


def _synth_pgexplainer_train(wrapped_model, train_graphs, device, epochs=20, lr=3e-3):
    """Train PGExplainer on a set of graphs, return explainer or None."""
    from torch_geometric.explain import Explainer, PGExplainer as PyGPGExplainer
    try:
        pg_algo = PyGPGExplainer(epochs=epochs, lr=lr)
        explainer = Explainer(
            model=wrapped_model,
            algorithm=pg_algo,
            explanation_type='phenomenon',
            node_mask_type=None,
            edge_mask_type='object',
            model_config=dict(mode='multiclass_classification', task_level='graph',
                              return_type='raw'))

        # Dummy forward to materialize PGE's lazy MLP, then move to device
        d0 = train_graphs[0].clone().to(device)
        if not hasattr(d0, "batch") or d0.batch is None:
            d0.batch = torch.zeros(d0.x.size(0), dtype=torch.long, device=device)
        out0 = wrapped_model(d0.x, d0.edge_index, d0.batch)
        t0 = (torch.sigmoid(out0) > 0.5).long().view(-1)
        try:
            explainer.algorithm.train(
                epoch=0, model=wrapped_model,
                x=d0.x, edge_index=d0.edge_index,
                batch=d0.batch, target=t0, index=0)
        except Exception:
            pass
        # Move MLP to device
        if hasattr(pg_algo, 'mlp') and pg_algo.mlp is not None:
            pg_algo.mlp = pg_algo.mlp.to(device)
        # Recreate optimizer after materialization
        if hasattr(pg_algo, 'mlp') and pg_algo.mlp is not None:
            pg_algo.optimizer = torch.optim.Adam(pg_algo.mlp.parameters(), lr=lr)

        # Train on subset
        for i, data in enumerate(train_graphs[:min(100, len(train_graphs))]):
            data_d = data.clone().to(device)
            if not hasattr(data_d, "batch") or data_d.batch is None:
                data_d.batch = torch.zeros(data_d.x.size(0), dtype=torch.long, device=device)
            out = wrapped_model(data_d.x, data_d.edge_index, data_d.batch)
            target = out.argmax(dim=-1)
            explainer.algorithm.train(
                epoch=i, model=wrapped_model,
                x=data_d.x, edge_index=data_d.edge_index,
                batch=data_d.batch, target=target, index=0)
        return explainer
    except Exception as e:
        logger.warning(f"PGExplainer training failed: {e}")
    return None


def _synth_pgexplainer_explain(explainer, wrapped_model, data, device):
    """Get node scores from trained PGExplainer."""
    try:
        data_d = data.clone().to(device)
        if not hasattr(data_d, "batch") or data_d.batch is None:
            data_d.batch = torch.zeros(data_d.x.size(0), dtype=torch.long, device=device)
        out = wrapped_model(data_d.x, data_d.edge_index, data_d.batch)
        target = out.argmax(dim=-1)
        exp = explainer(data_d.x, data_d.edge_index, batch=data_d.batch,
                        target=target, index=0)
        edge_mask = exp.edge_mask
        if edge_mask is not None:
            ei = data_d.edge_index.cpu()
            scores = np.zeros(data_d.x.size(0))
            em = edge_mask.detach().cpu().numpy()
            for k in range(ei.shape[1]):
                scores[int(ei[0, k])] += abs(em[k])
                scores[int(ei[1, k])] += abs(em[k])
            return scores
    except Exception as e:
        logger.warning(f"PGExplainer explain failed: {e}")
    return None


# ── SubgraphX MCTS (canonical implementation) ────────────────────────────────
# Moved up here from the benchmark section (originally in cell 12) so the
# synthetic baselines can call the real MCTS too — matching paper §5.1's
# claim of "20 MCTS rollouts, 100 MC steps".

import networkx as _subx_nx


class _MCTSNode:
    """MCTS node (Yuan et al. 2021, adapted from DGL)."""
    __slots__ = ["nodes", "num_visit", "total_reward", "immediate_reward", "children"]

    def __init__(self, nodes):
        self.nodes = nodes
        self.num_visit = 0
        self.total_reward = 0.0
        self.immediate_reward = 0.0
        self.children = []


def _subgraphx_explain(model, data, device, num_classes=2, min_atoms=3,
                       rollout=10, sample_num=30, num_hops=3, coef=10.0,
                       high2low=True, num_child=12):
    """Standalone SubgraphX (Yuan et al., ICML 2021). MCTS + MC Shapley.

    Returns per-node binary indicator scores (1 on the best subgraph) or None.
    Used by both the synthetic baseline (`_synth_subgraphx`) and the benchmark
    baseline (`run_subgraphx` in cell 12).
    """
    data = data.clone().to(device)
    num_nodes = data.x.size(0)
    x = data.x
    edge_index = data.edge_index

    G = _subx_nx.Graph()
    G.add_nodes_from(range(num_nodes))
    ei = edge_index.cpu().numpy()
    for k in range(ei.shape[1]):
        G.add_edge(int(ei[0, k]), int(ei[1, k]))

    model.eval()
    with torch.no_grad():
        d = data.clone()
        if not hasattr(d, "batch") or d.batch is None:
            d.batch = torch.zeros(num_nodes, dtype=torch.long, device=device)
        out = model(d)
        if out.dim() >= 2 and out.shape[-1] >= 2:
            target_class = int(out[0].argmax().item())
        else:
            target_class = int((out > 0).long().item()) if out.numel() == 1 else 0

    def mc_shapley(subgraph_nodes):
        sg_set = set(subgraph_nodes)
        local_set = set(sg_set)
        for _ in range(num_hops - 1):
            nbrs = set()
            for nd in local_set:
                nbrs.update(G.neighbors(nd))
            local_set = local_set | nbrs
        local_list = sorted(local_set)
        complement = [nd for nd in local_list if nd not in sg_set]
        marginals = []
        for _ in range(sample_num):
            perm = np.random.permutation(len(complement) + 1)
            split = int(np.where(perm == len(complement))[0])
            selected = [complement[i] for i in perm[:split] if i < len(complement)]
            exc_mask = torch.ones(num_nodes, 1, device=device)
            for nd in local_list:
                exc_mask[nd] = 0.0
            for nd in selected:
                exc_mask[nd] = 1.0
            inc_mask = exc_mask.clone()
            for nd in sg_set:
                inc_mask[nd] = 1.0
            with torch.no_grad():
                exc_out = model(x * exc_mask, edge_index).softmax(dim=-1)
                inc_out = model(x * inc_mask, edge_index).softmax(dim=-1)
                exc_val = exc_out[0, target_class] if exc_out.dim() >= 2 else exc_out[0]
                inc_val = inc_out[0, target_class] if inc_out.dim() >= 2 else inc_out[0]
            marginals.append((inc_val - exc_val).item())
        return float(np.mean(marginals))

    node_maps = {}
    _key = lambda nds: str(sorted(nds))
    root = _MCTSNode(list(range(num_nodes)))
    node_maps[_key(root.nodes)] = root

    def get_children(mn):
        if mn.children:
            return mn.children
        if len(mn.nodes) <= min_atoms:
            return []
        subG = G.subgraph(mn.nodes)
        deg = sorted(subG.degree(), key=lambda x: x[1], reverse=high2low)
        expand = [n for n, _ in deg[:min(len(deg), num_child)]]
        children_map = {}
        for rm_node in expand:
            remaining = [n for n in mn.nodes if n != rm_node]
            if not remaining:
                continue
            sub = G.subgraph(remaining)
            ccs = list(_subx_nx.connected_components(sub))
            if not ccs:
                continue
            largest_cc = sorted(max(ccs, key=len))
            key = _key(largest_cc)
            if key not in node_maps:
                node_maps[key] = _MCTSNode(largest_cc)
            if key not in children_map:
                children_map[key] = node_maps[key]
        mn.children = list(children_map.values())
        for child in mn.children:
            if child.immediate_reward == 0.0:
                child.immediate_reward = mc_shapley(child.nodes)
        return mn.children

    def mcts_rollout(mn):
        if len(mn.nodes) <= min_atoms:
            return mn.immediate_reward
        children = get_children(mn)
        if not children:
            return mn.immediate_reward
        visit_sum = sum(c.num_visit for c in children)
        visit_sqrt = math.sqrt(visit_sum) if visit_sum > 0 else 0.0
        chosen = max(
            children,
            key=lambda c: (c.total_reward / max(c.num_visit, 1)
                           + coef * c.immediate_reward * visit_sqrt / (1 + c.num_visit))
        )
        reward = mcts_rollout(chosen)
        chosen.num_visit += 1
        chosen.total_reward += reward
        return reward

    for _ in range(rollout):
        mcts_rollout(root)

    best_leaf = None
    best_reward = -float("inf")
    for mn in node_maps.values():
        if len(mn.nodes) > min_atoms:
            continue
        if mn.immediate_reward > best_reward:
            best_reward = mn.immediate_reward
            best_leaf = mn
    if best_leaf is None:
        return None
    scores = np.zeros(num_nodes, dtype=np.float64)
    for nid in best_leaf.nodes:
        if 0 <= nid < num_nodes:
            scores[nid] = 1.0
    return scores if scores.sum() > 0 else None


def _synth_subgraphx(wrapped_model, data, device, num_classes=2,
                     min_atoms=5, rollout=20, sample_num=100):
    """Real MCTS SubgraphX for synthetic graphs (replaces the previous
    leave-one-out stub). Matches paper §5.1 "20 MCTS rollouts, 100 MC steps".
    """
    try:
        inner_model = wrapped_model.inner if hasattr(wrapped_model, "inner") else wrapped_model
        # The MCTS expects a model callable on Data objects (DumplingGNN does);
        # the inner model handles that directly.
        return _subgraphx_explain(
            inner_model, data, device,
            num_classes=num_classes, min_atoms=min_atoms,
            rollout=rollout, sample_num=sample_num,
        )
    except Exception as e:
        logger.warning(f"SubgraphX failed: {e}")
    return None


def evaluate_all_baselines(
    model, test_data, test_df, task_name, device,
    n_explain=10, sparsity_grid=None,
):
    """Run GNNExplainer, PGExplainer, SubgraphX + Grad×Input on synthetic data.

    Returns DataFrame with fidelity metrics at multiple sparsity levels.
    """
    if sparsity_grid is None:
        # Paper §5.1: ten sparsity levels from 0.50 to 0.95 in steps of 0.05.
        sparsity_grid = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]

    print(f"\n  [ALL BASELINES] {task_name}")

    # Wrap model
    wrapped = _WrappedDumpling(model).to(device)
    wrapped.eval()

    MAX_N = CFG.MAX_NODES_EXPLAIN
    valid_mask = test_df["n_nodes"] <= MAX_N
    small_df = test_df[valid_mask]
    if len(small_df) < 10:
        small_df = test_df

    toxic = small_df[small_df["label"] == 1]
    nontoxic = small_df[small_df["label"] == 0]
    n_pos = min(n_explain, len(toxic))
    n_neg = min(n_explain, len(nontoxic))
    sample_idx = (toxic.sample(n=n_pos, random_state=42).index.tolist() +
                  nontoxic.sample(n=n_neg, random_state=42).index.tolist())
    print(f"    Explaining {len(sample_idx)} graphs ({n_pos} pos, {n_neg} neg)")

    # Train PGExplainer once
    pg_explainer = _synth_pgexplainer_train(wrapped, test_data[:100], device)
    if pg_explainer is not None:
        print(f"    PGExplainer trained")
    else:
        print(f"    PGExplainer training failed — skipping")

    all_records = []

    for mol_i, idx in enumerate(sample_idx):
        if idx >= len(test_data):
            continue
        data = test_data[idx]
        row = test_df.iloc[idx]
        num_nodes = data.x.size(0)

        # Ground truth
        motif_A = set(data.motif_A_nodes.tolist()) if hasattr(data, "motif_A_nodes") else set()
        motif_B = set(data.motif_B_nodes.tolist()) if hasattr(data, "motif_B_nodes") else set()
        gt_nodes = motif_A | motif_B

        methods_scores = {}

        # Ours (node-level Shapley values from interaction matrix)
        try:
            sg = SyntheticGraph(data.x, data.edge_index)
            vfunc = ValueFunction(model, sg, device)
            M = compute_shapley_taylor_matrix(
                vfunc, sg, n_permutations=CFG.N_PERM, seed=42)
            methods_scores["ours"] = M.diagonal().copy()
        except Exception as e:
            logger.warning(f"Our method failed: {e}")

        # Grad×Input
        try:
            gxi = grad_x_input_attribution(model, data, device)
            if gxi is not None:
                methods_scores["grad_x_input"] = gxi
        except Exception:
            pass

        # GNNExplainer
        gnn = _synth_gnnexplainer(wrapped, data, device)
        if gnn is not None:
            methods_scores["gnnexplainer"] = gnn

        # PGExplainer
        if pg_explainer is not None:
            pge = _synth_pgexplainer_explain(pg_explainer, wrapped, data, device)
            if pge is not None:
                methods_scores["pgexplainer"] = pge

        # SubgraphX (expensive — first 50 per task to match paper §5.1).
        if mol_i < 50:
            subx = _synth_subgraphx(wrapped, data, device,
                                     rollout=20, sample_num=100)
            if subx is not None:
                methods_scores["subgraphx"] = subx

        # Evaluate each method at each sparsity level
        for method_name, scores in methods_scores.items():
            # Localization
            total_attr = np.sum(np.abs(scores))
            loc = float(sum(abs(scores[p]) for p in gt_nodes if p < len(scores))
                        / total_attr) if total_attr > 1e-12 else 0.0

            for target_sp in sparsity_grid:
                k = max(1, int((1.0 - target_sp) * num_nodes))
                expl_nodes = select_topk_nodes(scores, k)

                try:
                    fid = compute_fidelity_metrics(model, data, expl_nodes, device)
                except Exception:
                    continue

                rec = {
                    "graph_id": int(row.get("graph_id", idx)),
                    "label": int(row["label"]),
                    "has_A": int(row.get("has_A", 0)),
                    "has_B": int(row.get("has_B", 0)),
                    "method": method_name,
                    "target_sparsity": target_sp,
                    "n_nodes": num_nodes,
                    "loc_main": loc,
                    **{k2: v for k2, v in fid.items()
                       if k2 in ("fid_plus", "fid_minus", "prob_fid_plus",
                                 "prob_fid_minus", "sparsity", "harmonic_fidelity")},
                }
                if gt_nodes:
                    tp = len(expl_nodes & gt_nodes)
                    rec["gt_precision"] = tp / max(len(expl_nodes), 1)
                    rec["gt_recall"] = tp / max(len(gt_nodes), 1)
                all_records.append(rec)

        if (mol_i + 1) % 5 == 0 or mol_i == 0:
            print(f"    [{mol_i+1}/{len(sample_idx)}]  methods: {list(methods_scores.keys())}")

    df = pd.DataFrame(all_records)

    # Print summary at sp=0.7
    if not df.empty:
        sp07 = df[abs(df["target_sparsity"] - 0.7) < 0.05]
        if not sp07.empty:
            print(f"\n    --- {task_name} H-Fidelity Summary (sp=0.7) ---")
            hf_col = "harmonic_fidelity"
            summary = sp07.groupby("method").agg(
                hf=(hf_col, "mean"),
                loc=("loc_main", "mean"),
                n=("graph_id", "nunique")
            ).sort_values("hf", ascending=False)
            print(f"    {'Method':<18s} {'H-Fid':>8s} {'Loc':>8s} {'n':>5s}")
            print(f"    {'-'*40}")
            for method, r in summary.iterrows():
                print(f"    {method:<18s} {r['hf']:>8.4f} {r['loc']:>8.4f} {int(r['n']):>5d}")

    save_path = os.path.join(CFG.DATA_DIR, f"baselines_{task_name}.csv")
    df.to_csv(save_path, index=False)
    print(f"    Saved {save_path}")
    return df


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# §15  COUNTERFACTUAL EXPERIMENT (graph-native, DIST task)
# ═══════════════════════════════════════════════════════════════════════════

def _bfs_shortest_path(adj, src, tgt):
    """Return shortest path from src to tgt as list of nodes, or [] if none."""
    if src == tgt: return [src]
    visited = {src}
    queue = [(src, [src])]
    while queue:
        node, path = queue.pop(0)
        for nb in adj.get(node, set()):
            if nb == tgt:
                return path + [nb]
            if nb not in visited:
                visited.add(nb)
                queue.append((nb, path + [nb]))
    return []


def _score_graph(model, data: Data, device) -> float:
    """Score a PyG Data object, returning p(positive)."""
    data = data.clone().to(device)
    data.batch = torch.zeros(data.x.size(0), dtype=torch.long, device=device)
    model.eval()
    with torch.no_grad():
        z = model(data)
        # Handle scalar [1] (DumplingGNN) and [1,C] (multi-class) outputs
        if z.dim() >= 2 and z.shape[-1] >= 2:
            return float(torch.sigmoid(z[0, 1]).item())
        if z.dim() >= 2 and z.shape[-1] == 1:
            return float(torch.sigmoid(z[0, 0]).item())
        if z.dim() == 1:
            return float(torch.sigmoid(z[0]).item())
        return float(torch.sigmoid(z).item())


def _insert_nodes_on_edge(data: Data, u: int, v: int, k: int) -> Optional[Data]:
    """Insert k nodes into the edge (u,v), creating u-n1-n2-...-nk-v.

    Returns new Data object with recomputed features, or None if edge
    doesn't exist.
    """
    ei = data.edge_index.numpy()
    edges = set()
    for col in range(ei.shape[1]):
        a, b = int(ei[0, col]), int(ei[1, col])
        if a < b:
            edges.add((a, b))
        else:
            edges.add((b, a))

    key = (min(u, v), max(u, v))
    if key not in edges:
        return None

    # Remove original edge
    edges.discard(key)

    n_old = int(data.x.shape[0])
    new_indices = list(range(n_old, n_old + k))

    # Wire: u -> new[0] -> new[1] -> ... -> new[k-1] -> v
    edges.add((min(u, new_indices[0]), max(u, new_indices[0])))
    for i in range(k - 1):
        edges.add((new_indices[i], new_indices[i + 1]))
    edges.add((min(new_indices[-1], v), max(new_indices[-1], v)))

    # Build new edge_index
    edge_list = []
    for a, b in edges:
        edge_list.append([a, b])
        edge_list.append([b, a])
    new_ei = torch.tensor(edge_list, dtype=torch.long).t().contiguous()

    # Recompute features
    new_x = compute_structural_features(n_old + k, new_ei, data.x.shape[1])

    new_data = Data(x=new_x, edge_index=new_ei)
    # Carry over motif metadata
    if hasattr(data, "motif_A_nodes"):
        new_data.motif_A_nodes = data.motif_A_nodes.clone()
    if hasattr(data, "motif_B_nodes"):
        new_data.motif_B_nodes = data.motif_B_nodes.clone()
    return new_data


def _attach_path_at_node(data: Data, attach_to: int, k: int) -> Optional[Data]:
    """Attach a path of k nodes at an existing node (control edit)."""
    n_old = int(data.x.shape[0])
    new_nodes = list(range(n_old, n_old + k))

    ei = data.edge_index.numpy()
    edges = set()
    for col in range(ei.shape[1]):
        a, b = int(ei[0, col]), int(ei[1, col])
        if a < b: edges.add((a, b))
        else: edges.add((b, a))

    # Attach: existing -> new[0] -> new[1] -> ...
    edges.add((min(attach_to, new_nodes[0]), max(attach_to, new_nodes[0])))
    for i in range(k - 1):
        edges.add((new_nodes[i], new_nodes[i + 1]))

    edge_list = []
    for a, b in edges:
        edge_list.append([a, b])
        edge_list.append([b, a])
    new_ei = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    new_x = compute_structural_features(n_old + k, new_ei, data.x.shape[1])

    new_data = Data(x=new_x, edge_index=new_ei)
    if hasattr(data, "motif_A_nodes"):
        new_data.motif_A_nodes = data.motif_A_nodes.clone()
    if hasattr(data, "motif_B_nodes"):
        new_data.motif_B_nodes = data.motif_B_nodes.clone()
    return new_data


def _compute_motif_distance_from_data(data: Data) -> int:
    """Compute shortest-path distance between motif A and B nodes."""
    if not hasattr(data, "motif_A_nodes") or not hasattr(data, "motif_B_nodes"):
        return -1
    A = data.motif_A_nodes.tolist()
    B = data.motif_B_nodes.tolist()
    if not A or not B: return -1

    adj = defaultdict(set)
    ei = data.edge_index.numpy()
    for k in range(ei.shape[1]):
        adj[int(ei[0, k])].add(int(ei[1, k]))

    best = float("inf")
    for a in A:
        path = _bfs_shortest_path(adj, a, B[0])
        for b in B:
            p = _bfs_shortest_path(adj, a, b)
            if p:
                best = min(best, len(p) - 1)
    return int(best) if best < float("inf") else -1


def run_counterfactual_experiment(
    model, test_data: List[Data], test_df: pd.DataFrame,
    tau: int, device, N: int = 30, seed: int = 42,
):
    """Paired counterfactual: break vs control edit on DIST task."""
    print(f"\n  [COUNTERFACTUAL] DIST task (tau={tau}, N={N})")
    rng = np.random.RandomState(seed)

    lab_col = "clean_label" if "clean_label" in test_df.columns else "label"
    cand_mask = ((test_df["has_A"] == 1) & (test_df["has_B"] == 1) &
                 (test_df[lab_col] == 1) & (test_df["n_nodes"] <= CFG.MAX_NODES_EXPLAIN))
    candidates = test_df[cand_mask]
    subset_idx = candidates.sample(n=min(N, len(candidates)), random_state=seed).index.tolist()
    print(f"    Selected {len(subset_idx)} AB_close graphs")

    pairs = []
    n_fail_break, n_fail_ctrl = 0, 0

    for mol_i, idx in enumerate(subset_idx):
        if idx >= len(test_data): continue
        data = test_data[idx]
        row = test_df.iloc[idx]
        orig_p = _score_graph(model, data, device)
        orig_dist = _compute_motif_distance_from_data(data)
        if orig_dist < 0 or orig_dist > tau:
            continue

        # Find shortest path between motifs
        A_nodes = data.motif_A_nodes.tolist()
        B_nodes = data.motif_B_nodes.tolist()
        adj = defaultdict(set)
        ei = data.edge_index.numpy()
        for k_e in range(ei.shape[1]):
            adj[int(ei[0, k_e])].add(int(ei[1, k_e]))

        # Find best (shortest) path
        best_path = []
        for a in A_nodes:
            for b in B_nodes:
                p = _bfs_shortest_path(adj, a, b)
                if p and (not best_path or len(p) < len(best_path)):
                    best_path = p

        if len(best_path) < 2:
            n_fail_break += 1; continue

        # Try inserting k nodes along path edges to push distance > tau
        cf_break_data = None
        edit_size = 0
        for k in range(1, CFG.CF_MAX_INSERT + 1):
            path_edges = [(best_path[i], best_path[i+1]) for i in range(len(best_path)-1)]
            edge_order = list(range(len(path_edges)))
            rng.shuffle(edge_order)
            for ei_idx in edge_order:
                u, v = path_edges[ei_idx]
                candidate = _insert_nodes_on_edge(data, u, v, k)
                if candidate is None: continue
                new_dist = _compute_motif_distance_from_data(candidate)
                if new_dist > tau:
                    cf_break_data = candidate
                    edit_size = k
                    break
            if cf_break_data is not None: break

        if cf_break_data is None:
            n_fail_break += 1; continue

        # Control edit: attach path of edit_size at a node NOT on the motif path
        path_set = set(best_path) | set(A_nodes) | set(B_nodes)
        other_nodes = [n for n in range(data.x.shape[0]) if n not in path_set]
        if not other_nodes:
            other_nodes = [n for n in range(data.x.shape[0]) if n not in set(A_nodes + B_nodes)]
        if not other_nodes:
            n_fail_ctrl += 1; continue

        attach_node = other_nodes[rng.randint(len(other_nodes))]
        cf_ctrl_data = _attach_path_at_node(data, attach_node, edit_size)
        if cf_ctrl_data is None:
            n_fail_ctrl += 1; continue

        ctrl_dist = _compute_motif_distance_from_data(cf_ctrl_data)
        if ctrl_dist > tau:
            n_fail_ctrl += 1; continue

        # Score
        p_break = _score_graph(model, cf_break_data, device)
        p_ctrl = _score_graph(model, cf_ctrl_data, device)

        pairs.append({
            "original_p": orig_p, "original_dist": orig_dist,
            "p_break": p_break, "break_dist": _compute_motif_distance_from_data(cf_break_data),
            "p_ctrl": p_ctrl, "ctrl_dist": ctrl_dist,
            "delta_break": p_break - orig_p,
            "delta_ctrl": p_ctrl - orig_p,
            "abs_delta_break": abs(p_break - orig_p),
            "abs_delta_ctrl": abs(p_ctrl - orig_p),
            "edit_size": edit_size,
        })

    pairs_df = pd.DataFrame(pairs)
    print(f"    Completed: {len(pairs_df)} pairs, "
          f"break_fail={n_fail_break}, ctrl_fail={n_fail_ctrl}")

    summary = {"n_pairs": len(pairs_df), "n_fail_break": n_fail_break,
               "n_fail_ctrl": n_fail_ctrl, "tau": tau}
    if len(pairs_df) >= 3:
        abs_db = pairs_df["abs_delta_break"].values
        abs_dc = pairs_df["abs_delta_ctrl"].values
        summary["mean_abs_delta_break"] = float(np.mean(abs_db))
        summary["mean_abs_delta_ctrl"] = float(np.mean(abs_dc))
        summary["mean_diff"] = float(np.mean(abs_db - abs_dc))
        summary["frac_break_bigger"] = float((abs_db > abs_dc).mean())
        try:
            stat, pval = wilcoxon(abs_db, abs_dc, alternative="greater")
            summary["wilcoxon_stat"] = float(stat)
            summary["wilcoxon_p"] = float(pval)
        except Exception:
            pass
        if len(abs_db) >= 5:
            ci_lo, ci_hi = bootstrap_ci(abs_db - abs_dc, CFG.DEL_BOOTSTRAP_N)
            summary["diff_ci_lo"] = ci_lo; summary["diff_ci_hi"] = ci_hi

        print(f"    |dbreak|={summary['mean_abs_delta_break']:.4f}  "
              f"|dctrl|={summary['mean_abs_delta_ctrl']:.4f}  "
              f"diff={summary['mean_diff']:.4f}")
        if "wilcoxon_p" in summary:
            print(f"    Wilcoxon p={summary['wilcoxon_p']:.4f}")

    return pairs_df, summary


# ═══════════════════════════════════════════════════════════════════════════
# §16  STABILITY
# ═══════════════════════════════════════════════════════════════════════════

def stability_test(model, test_data, test_df, task_name, device):
    print(f"\n  [STABILITY] {task_name}")
    seeds = CFG.STABILITY_SEEDS
    n_mol = CFG.STABILITY_N

    both_mask = (test_df["has_A"]==1) & (test_df["has_B"]==1) & (test_df["n_nodes"]<=CFG.MAX_NODES_EXPLAIN)
    pool = test_df[both_mask] if both_mask.sum() >= n_mol else test_df[test_df["n_nodes"]<=CFG.MAX_NODES_EXPLAIN]
    subset_idx = pool.sample(n=min(n_mol, len(pool)), random_state=42).index.tolist()

    rows = []
    for idx in subset_idx:
        if idx >= len(test_data): continue
        data = test_data[idx]
        row = test_df.iloc[idx]
        per_seed = []
        for seed in seeds:
            try:
                res = explain_graph(model, data, device, n_permutations=CFG.N_PERM,
                                     sa_iterations=CFG.SA_ITERATIONS, seed=seed, verbose=False)
            except Exception:
                continue
            if res is None: continue
            sg = res["sg"]
            M = res["M"]
            phi = M.diagonal().copy()
            partition = res["best_partition"]
            group_agg = res["group_agg"]

            motif_A = set(data.motif_A_nodes.tolist()) if hasattr(data,"motif_A_nodes") else set()
            motif_B = set(data.motif_B_nodes.tolist()) if hasattr(data,"motif_B_nodes") else set()
            g2p = {g:p for p,g in enumerate(sg.heavy_idx)}
            A_pos = {g2p[n] for n in motif_A if n in g2p}
            B_pos = {g2p[n] for n in motif_B if n in g2p}

            loc = compute_loc_main(phi, A_pos | B_pos)
            gi_A = find_best_group_for_motif(partition, sg.heavy_idx, A_pos) if A_pos else -1
            gi_B = find_best_group_for_motif(partition, sg.heavy_idx, B_pos) if B_pos else -1
            I_AB = group_agg["cross_scores"][gi_A][gi_B] if gi_A>=0 and gi_B>=0 and gi_A!=gi_B else 0.0
            per_seed.append({"loc": loc, "I_AB": I_AB})

        if len(per_seed) >= 2:
            rows.append({
                "task": task_name,
                "loc_mean": np.mean([s["loc"] for s in per_seed]),
                "loc_std": np.std([s["loc"] for s in per_seed]),
                "I_AB_mean": np.mean([s["I_AB"] for s in per_seed]),
                "I_AB_std": np.std([s["I_AB"] for s in per_seed]),
                "n_seeds": len(per_seed),
            })

    df = pd.DataFrame(rows)
    if not df.empty:
        print(f"    loc_std={df['loc_std'].mean():.3f}  I_AB_std={df['I_AB_std'].mean():.4f}")
    return df


# ═══════════════════════════════════════════════════════════════════════════
# §17  MAIN EFFECTS USELESSNESS TESTS
# ═══════════════════════════════════════════════════════════════════════════

def main_effects_useless_tests(df, task_name):
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import StratifiedKFold, cross_val_score

    lab_col = "clean_label" if "clean_label" in df.columns else "label"
    y = df[lab_col].values.astype(int)
    has_A = df["has_A"].values.astype(int)
    has_B = df["has_B"].values.astype(int)

    print(f"\n  [MAIN EFFECTS] {task_name}")
    logreg_rows = []
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    for fs_name, X in [
        ("has_A_only", has_A.reshape(-1,1)),
        ("has_B_only", has_B.reshape(-1,1)),
        ("has_A_has_B", np.column_stack([has_A, has_B])),
        ("A_B_interaction", np.column_stack([has_A, has_B, has_A*has_B])),
    ]:
        try:
            sc = cross_val_score(LogisticRegression(max_iter=500, random_state=42),
                                 X, y, cv=cv, scoring="roc_auc")
            auc = sc.mean(); std = sc.std()
        except Exception:
            auc, std = 0.5, 0.0
        logreg_rows.append({"task": task_name, "feature_set": fs_name,
                            "auroc_mean": auc, "auroc_std": std})
        print(f"    {fs_name:20s}  AUROC={auc:.3f} +/- {std:.3f}")

    # Within both-motif subset (critical for DIST)
    both = df[(df["has_A"]==1)&(df["has_B"]==1)]
    if len(both) >= 20 and both[lab_col].nunique() >= 2:
        y_both = both[lab_col].values.astype(int)
        if "n_nodes" in both.columns:
            try:
                sc = cross_val_score(LogisticRegression(max_iter=500, random_state=42),
                                     both["n_nodes"].values.reshape(-1,1), y_both,
                                     cv=min(5, min((y_both==0).sum(),(y_both==1).sum())),
                                     scoring="roc_auc")
                auc_nn = sc.mean()
            except Exception:
                auc_nn = 0.5
            logreg_rows.append({"task": task_name, "feature_set": "AB_only__n_nodes",
                                "auroc_mean": auc_nn, "subset": "both_motif"})
            print(f"    AB-only n_nodes      AUROC={auc_nn:.3f} (should be ~0.5)")

        if "dist_AB" in both.columns:
            valid_d = both[both["dist_AB"] >= 0]
            if len(valid_d) >= 20 and valid_d[lab_col].nunique() >= 2:
                try:
                    sc = cross_val_score(LogisticRegression(max_iter=500, random_state=42),
                                         valid_d["dist_AB"].values.reshape(-1,1),
                                         valid_d[lab_col].values.astype(int),
                                         cv=5, scoring="roc_auc")
                    auc_d = sc.mean()
                except Exception:
                    auc_d = 0.5
                logreg_rows.append({"task": task_name, "feature_set": "AB_only__dist_AB",
                                    "auroc_mean": auc_d, "subset": "both_motif"})
                print(f"    AB-only dist_AB      AUROC={auc_d:.3f} (high for DIST)")

    return logreg_rows


# ═══════════════════════════════════════════════════════════════════════════
# §18  PLOTS
# ═══════════════════════════════════════════════════════════════════════════

def plot_roc_curves(roc_data, save_path):
    fig, ax = plt.subplots(figsize=(8, 6))
    for n, (y, p) in roc_data.items():
        fpr, tpr, _ = roc_curve(y, p)
        ax.plot(fpr, tpr, label=f"{n} ({roc_auc_score(y,p):.3f})", lw=2)
    ax.plot([0,1],[0,1],"k--",alpha=.5)
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.set_title("ROC Curves"); ax.legend(); ax.grid(True,alpha=.3)
    plt.tight_layout(); plt.savefig(save_path, dpi=150); plt.close()

def plot_deletion_comparison(all_exp, path):
    tasks = all_exp["task"].unique()
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(tasks)); w = 0.35
    em = [all_exp[all_exp["task"]==t]["del_auc"].mean() for t in tasks]
    rm = [all_exp[all_exp["task"]==t]["del_auc_rand"].mean() for t in tasks]
    ax.bar(x-w/2, em, w, label="Explanation", color="coral")
    ax.bar(x+w/2, rm, w, label="Random", color="steelblue")
    ax.set_xticks(x); ax.set_xticklabels(tasks, fontsize=9)
    ax.set_ylabel("Deletion AUC"); ax.legend(); ax.grid(True,alpha=.3,axis="y")
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.close()

def plot_interaction_histogram(df, task, path):
    fig, ax = plt.subplots(figsize=(8, 5))
    t = df[df["label"]==1]["I_AB"].dropna()
    nt = df[df["label"]==0]["I_AB"].dropna()
    all_v = pd.concat([t,nt])
    if all_v.empty: plt.close(); return
    bins = np.linspace(all_v.min()-0.1, all_v.max()+0.1, 30)
    ax.hist(t, bins=bins, alpha=.7, label="Positive", color="red")
    ax.hist(nt, bins=bins, alpha=.7, label="Negative", color="blue")
    ax.set_xlabel("I(A,B)"); ax.set_ylabel("Count")
    ax.set_title(f"I(A,B) distribution - {task}")
    ax.legend(); ax.grid(True,alpha=.3)
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.close()

def plot_counterfactual_effects(pairs_df, path):
    if pairs_df.empty or len(pairs_df) < 3: return
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    ax = axes[0]
    ax.scatter(pairs_df["abs_delta_ctrl"], pairs_df["abs_delta_break"], alpha=.7, edgecolors="k")
    lim = max(pairs_df["abs_delta_break"].max(), pairs_df["abs_delta_ctrl"].max()) * 1.1
    ax.plot([0,lim],[0,lim],"k--",alpha=.4); ax.set_xlabel("|dp_ctrl|"); ax.set_ylabel("|dp_break|")
    ax.set_title("Counterfactual Pairs"); ax.grid(True,alpha=.3)
    ax = axes[1]
    bp = ax.boxplot([pairs_df["abs_delta_break"], pairs_df["abs_delta_ctrl"]],
                    labels=["Break","Control"], patch_artist=True)
    bp["boxes"][0].set_facecolor("coral"); bp["boxes"][1].set_facecolor("steelblue")
    ax.set_ylabel("|dp|"); ax.set_title(f"Effect Sizes (n={len(pairs_df)})"); ax.grid(True,alpha=.3,axis="y")
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.close()


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# §19  MAIN
# ═══════════════════════════════════════════════════════════════════════════

def main():
    set_seed()
    device = CFG.DEVICE
    save_config()

    # ── STEP 1: Generate synthetic datasets ───────────────────────────────
    print("\n" + "=" * 70)
    print("STEP 1: GENERATING SYNTHETIC DATASETS")
    print("=" * 70)

    print("\n--- Task 1: AND ---")
    data1, df1 = generate_dataset_AND(CFG.N_TOTAL_PER_TASK, 42, CFG.LABEL_NOISE_RATE)
    dataset_diagnostics(df1, "Task 1 (AND)")

    print("\n--- Task 2: OR ---")
    data2, df2 = generate_dataset_OR(CFG.N_TOTAL_PER_TASK, 43, CFG.LABEL_NOISE_RATE)
    dataset_diagnostics(df2, "Task 2 (OR)")

    print("\n--- Task 3: DIST ---")
    data3, df3 = generate_dataset_DIST(CFG.N_TOTAL_PER_TASK, 44, CFG.LABEL_NOISE_RATE, CFG.DIST_THRESH)
    dataset_diagnostics(df3, "Task 3 (DIST)")

    tasks_df = {"task1_AND": df1, "task2_OR": df2, "task3_DIST": df3}
    tasks_data = {"task1_AND": data1, "task2_OR": data2, "task3_DIST": data3}

    # Scaffold leakage
    scaff_leak = scaffold_leakage_report(tasks_df)

    # Artifact diagnostics
    artifact_rows = []
    if CFG.ARTIFACT_CHECK:
        print("\n--- Artifact diagnostics ---")
        for tname, tdf in tasks_df.items():
            artifact_rows.append(artifact_diagnostics(tdf, tname))

    # Main effects tests
    print("\n--- Main effects tests ---")
    all_me = []
    for tname, tdf in tasks_df.items():
        all_me.extend(main_effects_useless_tests(tdf, tname))
    pd.DataFrame(all_me).to_csv(os.path.join(CFG.DATA_DIR, "main_effects_logreg.csv"), index=False)

    # ── STEP 2: Split ─────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("STEP 2: SPLITTING")
    print("=" * 70)

    datasets = {}
    for tname in tasks_df:
        sp = stratified_split(tasks_df[tname], tasks_data[tname], seed=42)
        datasets[tname] = {**sp, "df": tasks_df[tname], "all_data": tasks_data[tname]}
        for s in ["train", "val", "test"]:
            n = len(sp[s])
            print(f"  {tname} {s:5s}: n={n}")

    # ── STEP 3: Train ─────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("STEP 3: TRAINING")
    print("=" * 70)

    models, training_log = {}, []
    for tname, ds in datasets.items():
        print(f"\n--- {tname} ---")
        m, h = train_model(ds["train"], ds["val"], tname,
                           epochs=CFG.EPOCHS, lr=CFG.LR,
                           batch_size=CFG.BATCH_SIZE, patience=CFG.PATIENCE, device=device)
        models[tname] = m
        for r in h: r["task"] = tname
        training_log.extend(h)
    pd.DataFrame(training_log).to_csv(os.path.join(CFG.DATA_DIR, "training_log.csv"), index=False)

    # ── STEP 4: Evaluate ──────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("STEP 4: TEST EVALUATION")
    print("=" * 70)

    roc_data, eval_summary = {}, []
    for tname, ds in datasets.items():
        met, y, p = evaluate_model(models[tname], ds["test"], device)
        roc_data[tname] = (y, p)
        eval_summary.append({"task": tname, **met})
        print(f"  {tname}: AUROC={met['auroc']:.4f}  F1={met['f1']:.4f}  "
              f"thresh={met.get('threshold',0.5):.3f}  "
              f"prec={met.get('precision',0):.3f}  rec={met.get('recall',0):.3f}")
    pd.DataFrame(eval_summary).to_csv(os.path.join(CFG.DATA_DIR, "test_evaluation.csv"), index=False)
    plot_roc_curves(roc_data, os.path.join(CFG.DATA_DIR, "roc_curves.png"))

    # Scaffold held-out
    for tname, ds in datasets.items():
        she = scaffold_held_out_eval(models[tname], ds, ds["df"], device)
        auc_str = f"{she.get('novel_auroc', 'N/A')}"
        print(f"  {tname}: novel_AUROC={auc_str}")

    # ── STEP 4b: Counterfactual ───────────────────────────────────────────
    cf_pairs_df, cf_summary = pd.DataFrame(), {}
    if "task3_DIST" in datasets:
        print("\n--- Counterfactual (DIST) ---")
        ds3 = datasets["task3_DIST"]
        test_data3 = [ds3["all_data"][i] for i in ds3["test_rows"]
                      if i < len(ds3["all_data"])]
        test_df3 = ds3["df"].iloc[ds3["test_rows"]].reset_index(drop=True)
        cf_pairs_df, cf_summary = run_counterfactual_experiment(
            models["task3_DIST"], test_data3, test_df3,
            CFG.DIST_THRESH, device, N=CFG.CF_N, seed=CFG.SEED)
        if not cf_pairs_df.empty:
            cf_pairs_df.to_csv(os.path.join(CFG.DATA_DIR, "counterfactual_pairs.csv"), index=False)
            plot_counterfactual_effects(cf_pairs_df,
                                        os.path.join(CFG.DATA_DIR, "counterfactual_effects.png"))

    # ── STEP 5: Explanation verification ──────────────────────────────────
    print("\n" + "=" * 70)
    print("STEP 5: EXPLANATION VERIFICATION")
    print("=" * 70)

    all_exp = []
    all_baseline = []
    stability_rows = []

    for tname, ds in datasets.items():
        model = models[tname]
        test_data_t = [ds["all_data"][i] for i in ds["test_rows"]
                       if i < len(ds["all_data"])]
        test_df_t = ds["df"].iloc[ds["test_rows"]].reset_index(drop=True)

        rdf = verify_explanations(
            model, test_data_t, test_df_t, tname, device,
            n_explain=CFG.N_EXPLAIN, n_permutations=CFG.N_PERM,
            sa_iterations=CFG.SA_ITERATIONS)
        if not rdf.empty:
            rdf["task"] = tname
            all_exp.append(rdf)

            both = rdf[(rdf["has_A"]==1)&(rdf["has_B"]==1)]
            pos = rdf[rdf["label"]==1]
            neg = rdf[rdf["label"]==0]

            # Summary split by label
            fid_str = ""
            if "prob_fid_plus" in rdf.columns:
                fid_str = (f"  pfid+={rdf['prob_fid_plus'].mean():.3f}  "
                           f"h={rdf['harmonic_fidelity'].mean():.3f}")
            elif "fid_plus" in rdf.columns:
                fid_str = (f"  fid+={rdf['fid_plus'].mean():.3f}  "
                           f"h={rdf['harmonic_fidelity'].mean():.3f}")
            print(f"    ALL:  loc={rdf['loc_main'].mean():.3f}  "
                  f"del={rdf['del_auc'].mean():.3f}{fid_str}")
            if not pos.empty:
                ploc = pos['loc_main'].mean()
                pfid = pos['prob_fid_plus'].mean() if 'prob_fid_plus' in pos.columns else 0
                ph = pos['harmonic_fidelity'].mean() if 'harmonic_fidelity' in pos.columns else 0
                print(f"    POS:  loc={ploc:.3f}  pfid+={pfid:.3f}  h={ph:.3f}  (n={len(pos)})")
            if not neg.empty:
                nloc = neg['loc_main'].mean()
                nfid = neg['prob_fid_plus'].mean() if 'prob_fid_plus' in neg.columns else 0
                nh = neg['harmonic_fidelity'].mean() if 'harmonic_fidelity' in neg.columns else 0
                print(f"    NEG:  loc={nloc:.3f}  pfid+={nfid:.3f}  h={nh:.3f}  (n={len(neg)})")
            if not both.empty:
                print(f"    BOTH: dom={both['dom'].mean():.3f}  "
                      f"top1={float((both[both['rank']>0]['rank']==1).mean()):.2f}  "
                      f"(n={len(both)})")

    # ── STEP 6: Baselines ─────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("STEP 6: BASELINES (Grad×Input + GNNExplainer + PGExplainer + SubgraphX)")
    print("=" * 70)

    if CFG.GRAD_BASELINE:
        for tname, ds in datasets.items():
            test_data_t = [ds["all_data"][i] for i in ds["test_rows"]
                           if i < len(ds["all_data"])]
            test_df_t = ds["df"].iloc[ds["test_rows"]].reset_index(drop=True)
            gdf = evaluate_baseline_grad(
                models[tname], test_data_t, test_df_t, tname, device,
                n_explain=CFG.BASELINE_N_EXPLAIN)
            if not gdf.empty:
                gdf["task"] = tname
                all_baseline.append(gdf)

    # Multi-baseline comparison (GNNExplainer, PGExplainer, SubgraphX, Grad×Input)
    all_multi_baseline = []
    for tname, ds in datasets.items():
        test_data_t = [ds["all_data"][i] for i in ds["test_rows"]
                       if i < len(ds["all_data"])]
        test_df_t = ds["df"].iloc[ds["test_rows"]].reset_index(drop=True)
        bdf = evaluate_all_baselines(
            models[tname], test_data_t, test_df_t, tname, device,
            n_explain=CFG.BASELINE_N_EXPLAIN)
        if not bdf.empty:
            bdf["task"] = tname
            all_multi_baseline.append(bdf)

    if all_multi_baseline:
        multi_df = pd.concat(all_multi_baseline, ignore_index=True)
        multi_df.to_csv(os.path.join(CFG.DATA_DIR, "multi_baseline_comparison.csv"), index=False)
        print(f"\n  multi_baseline_comparison.csv: {len(multi_df)} rows")

    # ── STEP 7: Stability ─────────────────────────────────────────────────
    if len(CFG.STABILITY_SEEDS) >= 2:
        print("\n--- Stability ---")
        for tname, ds in datasets.items():
            test_data_t = [ds["all_data"][i] for i in ds["test_rows"]
                           if i < len(ds["all_data"])]
            test_df_t = ds["df"].iloc[ds["test_rows"]].reset_index(drop=True)
            sdf = stability_test(models[tname], test_data_t, test_df_t, tname, device)
            if not sdf.empty:
                stability_rows.append(sdf)

    # ── STEP 8: Save ──────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("STEP 8: SAVING OUTPUTS")
    print("=" * 70)

    all_combined = all_exp + all_baseline
    if all_combined:
        all_exp_df = pd.concat(all_combined, ignore_index=True)
        all_exp_df.to_csv(os.path.join(CFG.DATA_DIR, "explanation_metrics.csv"), index=False)
        print(f"  explanation_metrics.csv: {len(all_exp_df)} rows")

        if not all_exp_df.empty:
            plot_deletion_comparison(all_exp_df, os.path.join(CFG.DATA_DIR, "deletion_comparison.png"))

        # Fidelity summary per task x method
        fid_cols = ["fid_plus", "fid_minus", "prob_fid_plus", "prob_fid_minus",
                    "sparsity", "harmonic_fidelity"]
        avail_cols = [c for c in fid_cols if c in all_exp_df.columns]
        if avail_cols:
            fid_rows = []
            group_keys = ["task", "method"] if "task" in all_exp_df.columns else ["method"]
            valid_keys = [k for k in group_keys if k in all_exp_df.columns]
            if valid_keys:
                for name, grp in all_exp_df.groupby(valid_keys):
                    rec = dict(zip(valid_keys, name if isinstance(name, tuple) else [name]))
                    rec["n"] = len(grp)
                    for c in avail_cols:
                        rec[f"{c}_mean"] = grp[c].mean()
                        rec[f"{c}_std"] = grp[c].std()
                    fid_rows.append(rec)
                pd.DataFrame(fid_rows).to_csv(
                    os.path.join(CFG.DATA_DIR, "fidelity_summary.csv"), index=False)
                print(f"  fidelity_summary.csv: {len(fid_rows)} rows")
    else:
        all_exp_df = pd.DataFrame()

    if stability_rows:
        pd.concat(stability_rows, ignore_index=True).to_csv(
            os.path.join(CFG.DATA_DIR, "stability.csv"), index=False)

    # ── RESULTS SUMMARY ───────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("RESULTS SUMMARY")
    print("=" * 70)

    print("\n[MODEL PERFORMANCE]")
    for r in eval_summary:
        print(f"  {r['task']:15s}  AUROC={r['auroc']:.4f}  F1={r['f1']:.4f}")

    if not all_exp_df.empty:
        print(f"\n[EXPLANATION METRICS]")
        for method in all_exp_df["method"].unique():
            msub = all_exp_df[all_exp_df["method"] == method]
            print(f"  {method}:  loc={msub['loc_main'].mean():.3f}  "
                  f"del={msub['del_auc'].mean():.4f}")
            # New standardized metrics
            for col in ["fid_plus", "fid_minus", "prob_fid_plus", "prob_fid_minus",
                        "sparsity", "harmonic_fidelity"]:
                if col in msub.columns and msub[col].notna().any():
                    print(f"    {col:22s}  mean={msub[col].mean():.4f}  "
                          f"std={msub[col].std():.4f}")

    if cf_summary and cf_summary.get("n_pairs", 0) >= 3:
        print(f"\n[COUNTERFACTUAL]")
        print(f"  |dbreak|={cf_summary.get('mean_abs_delta_break',0):.4f}  "
              f"|dctrl|={cf_summary.get('mean_abs_delta_ctrl',0):.4f}  "
              f"diff={cf_summary.get('mean_diff',0):.4f}")
        if "wilcoxon_p" in cf_summary:
            print(f"  Wilcoxon p={cf_summary['wilcoxon_p']:.4f}")

    print("\n[OUTPUT FILES]")
    for f in sorted(os.listdir(CFG.DATA_DIR)):
        fp = os.path.join(CFG.DATA_DIR, f)
        if os.path.isfile(fp):
            print(f"  {f:40s} {os.path.getsize(fp):>10,} bytes")

    print("\n" + "=" * 70)
    print("  Experiment run — COMPLETE")
    print("=" * 70)


In [ ]:
if __name__ == "__main__":
    main()


In [ ]:
"""
Baseline Comparison & Public Benchmark Evaluation
===================================================
Compares our partition-based explanation method against GNNExplainer,
PGExplainer, SubgraphX, and Grad×Input on DIG benchmark datasets.

Requires:
  pip install dive-into-graphs

This module is designed to run AFTER framework_generalized.py and
pipeline_generalized.py are loaded in the Colab namespace.  It uses:
  - DIG datasets:  BA-2Motifs, MUTAG, Graph-SST2, BA-LRP
  - DIG explainers: GNNExplainer, PGExplainer, SubgraphX
  - Our framework:  ValueFunction, PartitionSearcher, etc. from framework_generalized.py
  - Our wrappers:   SyntheticGraph, fidelity metrics from pipeline_generalized.py
  - Our models:     BenchmarkGCN / BenchmarkGIN (defined here, DIG-compatible)
"""

# Additional import needed only by the public-benchmark script section.
import networkx as nx

# ── Configuration for public benchmarks ────────────────────────────────

class EVAL_CFG:
    DEVICE           = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    OUT_DIR          = "outputs"

    # Training
    EPOCHS           = 300          # QUICK — production: 300
    LR               = 1e-3
    BATCH_SIZE       = 64
    HIDDEN           = 64
    PATIENCE         = 30          # QUICK — production: 30

    # Explanation — our method
    N_EXPLAIN        = 50           # QUICK — production: 50
    N_PERM           = 50          # QUICK — production: 50
    SA_ITER          = 100          # QUICK — production: 100
    # Paper §5.1: ten sparsity levels from 0.50 to 0.95 in steps of 0.05.
    SPARSITY_GRID    = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]

    # SubgraphX
    SUBX_ROLLOUT     = 20           # QUICK — production: 20
    SUBX_MIN_ATOMS   = 5
    SUBX_SAMPLE_NUM  = 100          # QUICK — production: 100
    SUBX_MAX_GRAPHS  = 50          # paper §5.1: SubgraphX on first 50 per dataset

    # PGExplainer
    PGE_EPOCHS       = 30          # QUICK — production: 30
    PGE_LR           = 3e-3

os.makedirs(EVAL_CFG.OUT_DIR, exist_ok=True)


# ═══════════════════════════════════════════════════════════════════════════
# §1  GNN MODEL FOR DIG BENCHMARKS
# ═══════════════════════════════════════════════════════════════════════════
# DIG explainers expect specific model interfaces.  We define a clean
# 3-layer GCN that works with both DIG's explainer wrappers and our
# own evaluation code.

class BenchmarkGCN(nn.Module):
    """3-layer GCN + batch norm + global mean pool + linear readout.

    DIG explainers require that the model accept (x, edge_index, batch)
    and return class logits of shape [batch_size, num_classes].
    """

    def __init__(self, input_dim: int, hidden_dim: int = 64,
                 num_classes: int = 2, dropout: float = 0.0):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.bn3 = nn.BatchNorm1d(hidden_dim)
        self.lin = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index=None, batch=None, **kwargs):
        """Accept both Data objects and (x, edge_index, batch) triples.

        DIG explainers call model(x=features, edge_index=edges), so the
        first parameter must be named 'x' to receive the keyword argument.
        """
        if isinstance(x, Data):
            data = x
            x, edge_index, batch = (
                data.x, data.edge_index,
                getattr(data, "batch", None))

        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)

        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = self.dropout(x)
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = self.dropout(x)
        x = F.relu(self.bn3(self.conv3(x, edge_index)))
        x = global_mean_pool(x, batch)
        return self.lin(x)

    def get_emb(self, x, edge_index, batch=None):
        """Return node embeddings matching PyG Explainer hook output.

        PyG's PGExplainer hooks capture the output of the LAST
        MessagePassing layer (raw conv output, before BN/ReLU).
        We must match that here for inference to be consistent
        with training.
        """
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = self.conv3(x, edge_index)  # raw output — no BN, no ReLU
        return x


class BenchmarkGIN(nn.Module):
    """3-layer GIN + batch norm + global mean pool + linear readout."""

    def __init__(self, input_dim: int, hidden_dim: int = 64,
                 num_classes: int = 2, dropout: float = 0.0):
        super().__init__()
        self.conv1 = GINConv(nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim),
            nn.ReLU(), nn.Linear(hidden_dim, hidden_dim)))
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.conv2 = GINConv(nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.BatchNorm1d(hidden_dim),
            nn.ReLU(), nn.Linear(hidden_dim, hidden_dim)))
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.conv3 = GINConv(nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.BatchNorm1d(hidden_dim),
            nn.ReLU(), nn.Linear(hidden_dim, hidden_dim)))
        self.bn3 = nn.BatchNorm1d(hidden_dim)
        self.lin = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index=None, batch=None, **kwargs):
        if isinstance(x, Data):
            data = x
            x, edge_index, batch = (
                data.x, data.edge_index,
                getattr(data, "batch", None))
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)

        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = self.dropout(x)
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = self.dropout(x)
        x = F.relu(self.bn3(self.conv3(x, edge_index)))
        x = global_mean_pool(x, batch)
        return self.lin(x)

    def get_emb(self, x, edge_index, batch=None):
        """Return node embeddings matching PyG hook output (raw conv3, no BN/ReLU)."""
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = self.conv3(x, edge_index)  # raw output — no BN, no ReLU
        return x


# ═══════════════════════════════════════════════════════════════════════════
# §2  DIG DATASET LOADING
# ═══════════════════════════════════════════════════════════════════════════

def load_dig_dataset(name: str, root: str = "outputs/dig_data"):
    """Load a DIG benchmark dataset for graph classification.

    Supported names: 'ba_2motifs', 'mutag', 'graph_sst2', 'bbbp', 'bace', 'proteins'

    Returns
    -------
    (dataset, dataset_info_dict)
    """
    import functools

    os.makedirs(root, exist_ok=True)
    info = {"name": name, "task": "graph_classification"}

    # ── PyTorch 2.6+ fix ────────────────────────────────────────────
    # DIG datasets use torch.save/torch.load with pickled PyG objects.
    # PyTorch 2.6 changed the default of weights_only from False to True,
    # which rejects PyG Data classes (DataEdgeAttr, etc.).
    # We temporarily patch torch.load to use weights_only=False while
    # loading DIG datasets (which are trusted first-party data).
    _original_torch_load = torch.load

    @functools.wraps(_original_torch_load)
    def _patched_torch_load(*args, **kwargs):
        if "weights_only" not in kwargs:
            kwargs["weights_only"] = False
        return _original_torch_load(*args, **kwargs)

    torch.load = _patched_torch_load
    try:
        dataset = _load_dig_dataset_inner(name, root, info)
    finally:
        # Always restore original torch.load
        torch.load = _original_torch_load

    # Extract metadata
    sample = dataset[0]
    info["input_dim"] = sample.x.size(1) if sample.x is not None else 1
    info["num_graphs"] = len(dataset)
    info["avg_nodes"] = float(np.mean([d.x.size(0) for d in dataset]))

    print(f"  Loaded {name}: {len(dataset)} graphs, "
          f"input_dim={info['input_dim']}, avg_nodes={info['avg_nodes']:.1f}")
    return dataset, info


def _load_dig_dataset_inner(name, root, info):
    """Inner loader dispatching to DIG dataset classes."""

    if name.lower() == "mutag":
        # MUTAG requires rdkit via DIG's MoleculeDataset.
        # Check availability and give a clear message.
        try:
            from rdkit import Chem  # noqa: F401
        except (ImportError, OSError):
            raise ImportError(
                "MUTAG dataset requires rdkit. "
                "Install with: pip install rdkit-pypi")
        from dig.xgraph.dataset import MoleculeDataset
        dataset = MoleculeDataset(root, name="MUTAG")
        info.update({"num_classes": 2, "has_ground_truth": False,
                     "ground_truth_type": "domain_knowledge"})

    elif name.lower() in ("graph_sst2", "graphsst2"):
        from dig.xgraph.dataset import SentiGraphDataset
        # Graph-SST2 has ~70K graphs. Caller can subsample by passing the
        # desired number of graphs through their own slicing if memory-bound.
        dataset = SentiGraphDataset(root, name="Graph-SST2")
        info.update({"num_classes": 2, "has_ground_truth": False,
                     "ground_truth_type": "none"})

    elif name.lower() == "bbbp":
        # BBBP (Blood-Brain Barrier Penetration) from MoleculeNet via PyG
        try:
            from torch_geometric.datasets import MoleculeNet as PyG_MoleculeNet
        except ImportError:
            raise ImportError("BBBP requires torch_geometric.datasets.MoleculeNet")
        bbbp_root = os.path.join(root, "bbbp")
        os.makedirs(bbbp_root, exist_ok=True)
        raw_dataset = PyG_MoleculeNet(bbbp_root, name="BBBP")
        # Filter out graphs with missing features or labels
        dataset = []
        for d in raw_dataset:
            if d.x is None or d.edge_index is None or d.y is None:
                continue
            if d.x.size(0) < 3:
                continue
            g = d.clone()
            # BBBP features are integer atom types — cast to float for GCN
            if g.x.dtype != torch.float:
                g.x = g.x.float()
            g.y = g.y.squeeze().long()
            if g.y.dim() > 0 and g.y.size(0) > 1:
                g.y = g.y[0]
            dataset.append(g)
        # MoleculeNet BBBP yields ~2,039 valid graphs after RDKit filter;
        # paper §5.1 reports 2,039. No cap applied.
        info.update({"num_classes": 2, "has_ground_truth": False,
                     "ground_truth_type": "none"})

    elif name.lower() in ("ba_2motifs", "ba2motifs"):
        # BA-2Motifs (Luo et al., NeurIPS 2020). 1000 graphs, 25 nodes each,
        # binary classification: house motif vs. 5-cycle motif on BA base.
        # Edge-level ground truth: edges with both endpoints in indices [20, 25)
        # belong to the motif.
        try:
            from torch_geometric.datasets import BA2MotifDataset
            dataset = BA2MotifDataset(root=os.path.join(root, "ba2motifs"))
        except ImportError:
            # Fallback: load via DIG's SynGraphDataset.
            from dig.xgraph.dataset import SynGraphDataset
            dataset = SynGraphDataset(root, name="BA_2Motifs")
        # Attach ground-truth motif node set so localization metrics can use it.
        info.update({"num_classes": 2, "has_ground_truth": True,
                     "ground_truth_type": "motif_nodes",
                     "motif_node_indices": list(range(20, 25))})

    elif name.lower() == "proteins":
        from torch_geometric.datasets import TUDataset
        dataset = TUDataset(
            root=os.path.join(root, "tu"), name="PROTEINS",
            use_node_attr=True,
        )
        info.update({"num_classes": 2, "has_ground_truth": False,
                     "ground_truth_type": "none"})

    elif name.lower() == "bace":
        from torch_geometric.datasets import MoleculeNet as PyG_MoleculeNet
        bace_root = os.path.join(root, "bace")
        raw_dataset = PyG_MoleculeNet(bace_root, name="BACE")
        dataset = []
        for d in raw_dataset:
            if d.x is None or d.edge_index is None or d.y is None:
                continue
            if d.x.size(0) < 3:
                continue
            g = d.clone()
            if g.x.dtype != torch.float:
                g.x = g.x.float()
            g.y = g.y.squeeze().long()
            if g.y.dim() > 0 and g.y.size(0) > 1:
                g.y = g.y[0]
            dataset.append(g)
        info.update({"num_classes": 2, "has_ground_truth": False,
                     "ground_truth_type": "none"})

    return dataset


def split_dataset(dataset, train_frac=0.8, val_frac=0.1, seed=42):
    """Split dataset into train/val/test."""
    rng = np.random.RandomState(seed)
    n = len(dataset)
    perm = rng.permutation(n)
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)
    return {
        "train": [dataset[int(i)] for i in perm[:n_train]],
        "val": [dataset[int(i)] for i in perm[n_train:n_train + n_val]],
        "test": [dataset[int(i)] for i in perm[n_train + n_val:]],
        "test_indices": perm[n_train + n_val:].tolist(),
    }


# ═══════════════════════════════════════════════════════════════════════════
# §3  TRAINING ON DIG DATASETS
# ═══════════════════════════════════════════════════════════════════════════

def train_benchmark_model(
    model: nn.Module,
    train_data: List[Data],
    val_data: List[Data],
    dataset_name: str,
    device: torch.device,
    epochs: int = 300,
    lr: float = 1e-3,
    batch_size: int = 64,
    patience: int = 30,
) -> Tuple[nn.Module, Dict]:
    """Train a GNN model for graph classification on a DIG dataset."""
    model = model.to(device)
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, "max", patience=10, factor=0.5, min_lr=1e-5)
    crit = nn.CrossEntropyLoss()

    best_acc, best_state, wait = 0.0, None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0
        for batch in train_loader:
            batch = batch.to(device)
            opt.zero_grad()
            out = model(batch)
            # Squeeze handles varying y shapes: [B], [B,1], scalar
            y = batch.y.long().view(-1)
            loss = crit(out, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total_loss += loss.item() * batch.num_graphs

        # Validation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                pred = model(batch).argmax(dim=1)
                y = batch.y.long().view(-1)
                correct += (pred == y).sum().item()
                total += batch.num_graphs
        val_acc = correct / max(total, 1)
        sched.step(val_acc)

        if val_acc > best_acc:
            best_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            break
        if epoch % 10 == 0 or epoch == 1:
            avg_loss = total_loss / len(train_data)
            cur_lr = opt.param_groups[0]["lr"]
            print(f"  [{dataset_name}] Ep {epoch:3d}  "
                  f"loss={avg_loss:.4f}  val_acc={val_acc:.4f}  "
                  f"best={best_acc:.4f}  lr={cur_lr:.1e}")

    if best_state is None:
        best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)

    info = {"dataset": dataset_name, "best_val_acc": best_acc,
            "epochs_trained": epoch}
    print(f"  [{dataset_name}] Best val acc: {best_acc:.4f}")
    return model, info


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# §4  FAIR COMPARISON PROTOCOL — EXPLANATION FORMAT CONVERSION
# ═══════════════════════════════════════════════════════════════════════════

def edge_mask_to_node_scores(edge_mask: torch.Tensor,
                              edge_index: torch.Tensor,
                              num_nodes: int) -> np.ndarray:
    """Convert an edge importance mask to per-node importance scores.

    For each node, its score is the sum of importance weights of all
    edges incident to it, normalized by degree.  This is the standard
    conversion used in SubgraphX and DIG evaluation.

    Parameters
    ----------
    edge_mask : (num_edges,) tensor of edge importance weights
    edge_index : (2, num_edges) tensor
    num_nodes : int

    Returns
    -------
    node_scores : np.ndarray of shape (num_nodes,)
    """
    scores = np.zeros(num_nodes, dtype=np.float64)
    degree = np.zeros(num_nodes, dtype=np.float64)
    em = edge_mask.detach().cpu().numpy()
    ei = edge_index.detach().cpu().numpy()

    for k in range(ei.shape[1]):
        u, v = int(ei[0, k]), int(ei[1, k])
        if u < num_nodes:
            scores[u] += em[k]
            degree[u] += 1
        if v < num_nodes:
            scores[v] += em[k]
            degree[v] += 1

    # Normalize: average edge weight per node (avoids high-degree bias)
    mask = degree > 0
    scores[mask] /= degree[mask]
    return scores


def node_scores_to_explanation(
    node_scores: np.ndarray,
    target_sparsity: float,
) -> Set[int]:
    """Select top nodes to achieve a target sparsity level.

    sparsity = 1 - |selected| / |V|
    So |selected| = (1 - sparsity) * |V|

    Parameters
    ----------
    node_scores : per-node importance scores
    target_sparsity : float in [0, 1]

    Returns
    -------
    Set of selected node indices.
    """
    n = len(node_scores)
    k = max(1, int(round((1.0 - target_sparsity) * n)))
    k = min(k, n)
    ranked = np.argsort(-np.abs(node_scores))
    return set(int(ranked[i]) for i in range(k))


def subgraph_nodes_to_explanation(
    subgraph_nodes: List[int],
) -> Set[int]:
    """Convert a SubgraphX subgraph result to a node set."""
    return set(int(n) for n in subgraph_nodes)


# ═══════════════════════════════════════════════════════════════════════════
# §5  BASELINE WRAPPERS
# ═══════════════════════════════════════════════════════════════════════════
# Each wrapper takes a trained model and a single Data object,
# and returns per-node importance scores (np.ndarray).

def run_gnnexplainer(model, data, device, num_classes=2):
    """Run GNNExplainer on one graph, return node scores.

    Tries PyG native GNNExplainer first (torch_geometric.explain),
    falls back to DIG's version if PyG is too old.

    Returns np.ndarray of shape (num_nodes,) or None on failure.
    """
    data = data.clone().to(device)
    num_nodes = data.x.size(0)

    # Get predicted class
    model.eval()
    with torch.no_grad():
        batch_d = data.clone()
        if not hasattr(batch_d, "batch") or batch_d.batch is None:
            batch_d.batch = torch.zeros(num_nodes, dtype=torch.long, device=device)
        out = model(batch_d)
        if out.dim() >= 2 and out.shape[-1] >= 2:
            target = out.argmax(dim=-1)
        else:
            target = (out > 0).long()

    # --- Try PyG native GNNExplainer ---
    try:
        from torch_geometric.explain import Explainer, ModelConfig
        from torch_geometric.explain.algorithm import GNNExplainer as PyG_GNNExplainer

        explainer = Explainer(
            model=model,
            algorithm=PyG_GNNExplainer(epochs=100, lr=0.01),
            explanation_type='phenomenon',
            edge_mask_type='object',
            model_config=ModelConfig(
                mode='multiclass_classification',
                task_level='graph',
                return_type='raw',
            ),
        )
        explanation = explainer(data.x, data.edge_index, target=target)
        if hasattr(explanation, 'edge_mask') and explanation.edge_mask is not None:
            em = explanation.edge_mask.detach()
            return edge_mask_to_node_scores(em, data.edge_index, num_nodes)
    except ImportError:
        pass  # Fall through to DIG
    except Exception as e:
        logger.warning(f"PyG GNNExplainer failed: {e}")
        # Fall through to DIG

    # --- Fallback: DIG GNNExplainer ---
    try:
        from dig.xgraph.method import GNNExplainer as DIG_GNNExplainer

        pred_class = int(target[0].item()) if target.dim() >= 1 else int(target.item())
        explainer = DIG_GNNExplainer(
            model, epochs=100, lr=0.01, explain_graph=True)
        soft_masks, hard_masks, _ = explainer(
            data.x, data.edge_index,
            sparsity=0.5, num_classes=num_classes,
            target_label=torch.tensor(pred_class, device=device))
        masks_to_use = soft_masks if soft_masks else hard_masks
        if masks_to_use and len(masks_to_use) > 0:
            em = masks_to_use[0]
            if isinstance(em, torch.Tensor):
                return edge_mask_to_node_scores(em, data.edge_index, num_nodes)
    except (ImportError, OSError):
        pass
    except Exception as e:
        logger.warning(f"DIG GNNExplainer failed: {e}")

    return None


def run_pgexplainer(pg_explainer, data, device):
    """Run a trained PGExplainer on one graph, return node scores.

    Bypasses PyG's Explainer wrapper post-processing (which can zero out
    the mask) and calls the PGExplainer MLP directly on node embeddings.
    Returns np.ndarray of shape (num_nodes,) or None on failure.
    """
    data = data.clone().to(device)
    num_nodes = data.x.size(0)
    batch = torch.zeros(num_nodes, dtype=torch.long, device=device)

    try:
        model = pg_explainer.model
        pg_algo = pg_explainer.algorithm
        model.eval()
        pg_algo.mlp.eval()

        with torch.no_grad():
            # Get node embeddings from the GNN
            if hasattr(model, 'get_emb'):
                emb = model.get_emb(data.x, data.edge_index, batch=batch)
            else:
                # Fallback: run model hooks or use last-layer features
                emb = data.x
                for i, layer in enumerate(model.children()):
                    if isinstance(layer, (torch.nn.Linear,)) and i == len(list(model.children())) - 1:
                        break
                    emb = layer(emb, data.edge_index) if hasattr(layer, 'forward') else layer(emb)

            # For each edge, concatenate source and target embeddings
            src, dst = data.edge_index[0], data.edge_index[1]
            edge_emb = torch.cat([emb[src], emb[dst]], dim=-1)  # (num_edges, 2*hidden)

            # Pass through PGExplainer MLP to get raw logits
            raw_logits = pg_algo.mlp(edge_emb).squeeze(-1)  # (num_edges,)
            edge_mask = torch.sigmoid(raw_logits)

        node_sc = edge_mask_to_node_scores(edge_mask, data.edge_index, num_nodes)
        return node_sc

    except Exception as e:
        logger.warning(f"PGExplainer failed: {e}")

    return None


def train_pgexplainer(model, dataset, device, in_channels=128,
                      epochs=30, lr=3e-3):
    """Train a PGExplainer using PyG's native Explainer API.

    Uses torch_geometric.explain.algorithm.PGExplainer instead of the
    DIG implementation, which is incompatible with newer PyG versions.

    Returns a trained PyG Explainer object or None.
    """
    try:
        from torch_geometric.explain import Explainer, ModelConfig
        from torch_geometric.explain.algorithm import PGExplainer as PyG_PGExplainer
    except ImportError:
        logger.warning("PyG PGExplainer not available (requires torch_geometric >= 2.3)")
        return None

    try:
        pg_algo = PyG_PGExplainer(epochs=epochs, lr=lr)
        explainer = Explainer(
            model=model,
            algorithm=pg_algo,
            explanation_type='phenomenon',
            edge_mask_type='object',
            model_config=ModelConfig(
                mode='multiclass_classification',
                task_level='graph',
                return_type='raw',
            ),
        )

        # Materialize lazy MLP with a dummy forward pass so that
        # (a) the Linear(-1, ...) layer discovers its input dimension,
        # (b) all parameters live on the correct device.
        with torch.no_grad():
            d0 = dataset[0].to(device)
            b0 = torch.zeros(d0.x.size(0), dtype=torch.long, device=device)
            dummy_out = model(d0.x, d0.edge_index, batch=b0)
            dummy_tgt = dummy_out.argmax(dim=-1) if dummy_out.dim() >= 2 else (dummy_out > 0).long()
            try:
                _ = explainer(d0.x, d0.edge_index, target=dummy_tgt,
                              batch=b0, index=0)
            except Exception:
                pass  # First call may fail but still materializes MLP

        # Now the MLP is materialized — move to device and create optimizer
        pg_algo.mlp.to(device)
        pg_algo.optimizer = torch.optim.Adam(pg_algo.mlp.parameters(), lr=lr)

        # Train: iterate epochs × graphs
        n_train = min(len(dataset), 200)
        for epoch in range(epochs):
            total_loss = 0.0
            for gi in range(n_train):
                data = dataset[gi].to(device)
                batch = torch.zeros(data.x.size(0), dtype=torch.long, device=device)
                # Target = model prediction (phenomenon explanation)
                model.eval()
                with torch.no_grad():
                    out = model(data.x, data.edge_index, batch=batch)
                    if out.dim() >= 2:
                        target = out.argmax(dim=-1)
                    else:
                        target = (out > 0).long()

                loss = explainer.algorithm.train(
                    epoch, model, data.x, data.edge_index,
                    target=target, batch=batch, index=0)
                total_loss += loss
            if epoch % 10 == 0 or epoch == epochs - 1:
                print(f"    PGE epoch {epoch}: loss={total_loss/n_train:.4f}")

        return explainer

    except Exception as e:
        logger.warning(f"PGExplainer training failed: {e}")
        return None


def run_subgraphx(model, data, device, num_classes=2, min_atoms=3,
                  rollout=10, sample_num=30):
    """Standalone SubgraphX for PyG (adapted from DGL source, no DIG dependency).

    Uses MCTS to find the highest-scoring connected subgraph, scoring
    candidates via Monte Carlo Shapley value approximation with zero-filling.

    Based on: Yuan et al., "On Explainability of Graph Neural Networks via
    Subgraph Explorations" (ICML 2021), arXiv:2102.05152.
    Implementation adapted from dgl.nn.pytorch.explain.SubgraphX.

    Returns np.ndarray of shape (num_nodes,) or None on failure.
    """
    try:
        return _subgraphx_explain(model, data, device, num_classes,
                                  min_atoms, rollout, sample_num)
    except Exception as e:
        logger.warning(f"SubgraphX failed: {e}")
        return None


# `_MCTSNode` and `_subgraphx_explain` are now defined in cell 7 (synthetic
# baselines cell) so both synthetic and benchmark experiments can share the
# same canonical SubgraphX MCTS implementation. The wrapper `run_subgraphx`
# below references them via the shared kernel namespace.


# _subgraphx_explain definition lives in cell 7. See note above.


def run_grad_x_input(model, data, device):
    """Compute per-node Grad×Input scores.

    Returns np.ndarray of shape (num_nodes,) or None on failure.
    """
    try:
        data = data.clone().to(device)
        data.x.requires_grad_(True)
        if not hasattr(data, "batch") or data.batch is None:
            data.batch = torch.zeros(data.x.size(0), dtype=torch.long,
                                     device=device)
        model.eval()
        out = model(data)
        # Use predicted class logit
        if out.dim() == 1:
            target = out.sum()
        else:
            pred_class = out.argmax(dim=1)
            target = out[0, pred_class[0]]
        target.backward()

        grad = data.x.grad
        scores = (grad * data.x).abs().sum(dim=1).detach().cpu().numpy()
        return scores
    except Exception as e:
        logger.warning(f"Grad×Input failed: {e}")
        return None


def _make_graph_wrapper(x_cpu, ei_cpu):
    """Create a graph wrapper compatible with the framework's ValueFunction.

    Tries SyntheticGraph from the pipeline (Colab namespace) first.
    Falls back to a minimal inline implementation if pipeline is not loaded.
    """
    # Try pipeline's SyntheticGraph first
    try:
        sg = SyntheticGraph(x_cpu, ei_cpu)
        return sg
    except NameError:
        pass

    # Minimal inline implementation
    n = x_cpu.shape[0]
    feat_dim = x_cpu.shape[1]

    class _GraphWrapper:
        pass

    gw = _GraphWrapper()
    gw.x = x_cpu.clone()
    gw.edge_index = ei_cpu.clone()
    gw.n_total = n
    gw.n_heavy = n
    gw.heavy_idx = list(range(n))
    gw.h_idx = []
    gw.heavy_to_h = defaultdict(list)
    gw.feat_dim = feat_dim
    gw.baseline = torch.zeros(feat_dim, dtype=torch.float32)

    # Build adjacency
    gw.heavy_adj = defaultdict(set)
    ei = ei_cpu.numpy()
    for k in range(ei.shape[1]):
        a, b = int(ei[0, k]), int(ei[1, k])
        gw.heavy_adj[a].add(b)

    gw.pyg_data = lambda: Data(x=gw.x.clone(), edge_index=gw.edge_index.clone())
    gw.heavy_atom_symbol = lambda idx: f"v{idx}"
    gw.heavy_atom_labels = lambda: [f"{i}:v{i}" for i in gw.heavy_idx]

    # Backward-compat aliases
    gw.player_idx = gw.heavy_idx
    gw.player_adj = gw.heavy_adj
    gw.n_players = gw.n_heavy

    return gw


def run_our_method(model, data, device, n_permutations=50,
                   sa_iterations=100, seed=42):
    """Run our partition-based method on one graph.

    Returns (node_scores, partition, group_agg, sg) or None.
    node_scores is np.ndarray of shape (num_nodes,).
    """
    try:
        # Graph wrapper needs CPU tensors (calls .numpy() internally)
        x_cpu = data.x.detach().cpu()
        ei_cpu = data.edge_index.detach().cpu()
        sg = _make_graph_wrapper(x_cpu, ei_cpu)
        vfunc = ValueFunction(model, sg, device)

        M = compute_shapley_taylor_matrix(
            vfunc, sg, n_permutations=n_permutations, seed=seed)

        init_part = agglomerative_partition(sg, M)
        init_part = enforce_connected_partition(init_part, sg.heavy_adj)

        searcher = PartitionSearcher(sg, M, gamma_H=0.4, gamma_S=0.3)
        best_part, best_score = searcher.search(
            init_partition=init_part, max_iter=sa_iterations,
            seed=seed, verbose=False)

        best_lists = [sorted(g) for g in best_part]
        best_agg = aggregate_group_values(M, best_lists, sg.heavy_idx)

        node_scores = M.diagonal().copy()  # main effects
        return node_scores, best_lists, best_agg, sg

    except Exception as e:
        logger.warning(f"Our method failed: {e}")
        return None


# ═══════════════════════════════════════════════════════════════════════════
# §5b  ADDITIONAL SoTA BASELINES (reviewer-requested: MAGE, GraphSHAP-IQ, SME)
# ═══════════════════════════════════════════════════════════════════════════
# Each wrapper follows the same contract as run_gnnexplainer / run_subgraphx
# above: takes a trained PyG model and a single Data object, returns
# np.ndarray of per-node importance scores or None on failure (missing
# library, incompatible model, runtime error). Graceful-import pattern keeps
# the benchmark runnable even when these optional packages are absent.


def _is_linear_readout(model) -> bool:
    """Heuristic: GraphSHAP-IQ exact estimator requires a *linear* pooling
    followed by a *linear* output layer. BenchmarkGCN qualifies; DumplingGNN
    (whose GAT attention is nonlinear) does not.
    """
    cls = type(model).__name__
    # Whitelist known-linear-readout architectures used in this repo.
    return cls in ("BenchmarkGCN", "BenchmarkGIN")


def run_mage(model, data, device, num_classes=2,
             order=2, num_samples=100, num_motifs=3, beta=0.5):
    """MAGE — Myerson--Taylor Structure-Aware Graph Explainer
    (Bui et al., ICML 2024, arXiv:2405.14352).

    Requires the official MAGE repo cloned to `_vendor/MAGE/` (we add it to
    sys.path here). Returns per-node Myerson--Taylor scores or None.
    """
    try:
        import sys as _sys, os as _os
        # Try a few likely PYTHONPATH-relative locations for the MAGE clone.
        for _cand in (_os.path.abspath("_vendor/MAGE"),
                      _os.path.abspath("../_vendor/MAGE"),
                      _os.path.abspath("../../_vendor/MAGE")):
            if _os.path.isdir(_cand) and _cand not in _sys.path:
                _sys.path.insert(0, _cand)
        from mage.mage import Mage
        from mage.maskers import PyGDataMasker
    except ImportError:
        if not hasattr(run_mage, "_warned"):
            logger.warning(
                "MAGE not importable; skipping. Clone github.com/ngocbh/MAGE "
                "into _vendor/MAGE/ (patched MAGE source replaces "
                "`from torch_geometric.utils.subgraph import ...` with "
                "`from torch_geometric.utils import ...`)."
            )
            run_mage._warned = True
        return None

    try:
        data_d = data.clone().to(device)
        model.eval()
        masker = PyGDataMasker()
        explainer = Mage(model, masker=masker,
                         payoff_type="norm_prob", device=device)
        explanation = explainer.explain(
            data_d,
            num_motifs=num_motifs, beta=beta,
            ord=order, method="myt",
            num_samples=num_samples,
            target_class="max", connectivity="viky",
        )
        # MAGE returns a tuple (motifs, ...). Walk it for per-node scores.
        import numpy as _np
        num_nodes = data.x.size(0)
        scores = _np.zeros(num_nodes, dtype=_np.float64)
        if isinstance(explanation, dict):
            phi = explanation.get("phi") or explanation.get("node_scores")
            if phi is not None:
                scores += _np.asarray(phi)[:num_nodes]
        elif isinstance(explanation, (list, tuple)):
            # Convention: list of motif dicts with `nodes` and `score`.
            for item in explanation:
                if isinstance(item, dict) and "nodes" in item and "score" in item:
                    for n in item["nodes"]:
                        if 0 <= n < num_nodes:
                            scores[n] += float(item["score"])
        return scores if scores.sum() != 0 else None
    except Exception as e:
        logger.warning(f"MAGE failed: {e}")
        return None


def run_graphshapiq(model, data, device, num_classes=2, order=2,
                    budget=10_000):
    """GraphSHAP-IQ — Exact Computation of Any-Order Shapley Interactions for GNNs
    (Fumagalli et al., ICLR 2025, arXiv:2501.16944).

    Requires `pip install shapiq` and (optionally) the GraphSHAP-IQ repo at
    https://github.com/FFmgll/GraphSHAP-IQ for the model-specific exact
    estimator. The exact path requires a *linear* readout — see _is_linear_readout.
    Falls back to the model-agnostic KernelSHAP-IQ approximator if the exact
    estimator is unavailable.
    """
    if not _is_linear_readout(model):
        return None  # incompatible model
    try:
        from shapiq import Explainer  # shapiq >=1.0
    except ImportError:
        if not hasattr(run_graphshapiq, "_warned"):
            logger.warning(
                "shapiq not installed; skipping GraphSHAP-IQ. Run "
                "`pip install shapiq` and (optionally) clone "
                "github.com/FFmgll/GraphSHAP-IQ for the exact estimator."
            )
            run_graphshapiq._warned = True
        return None

    try:
        data_d = data.clone().to(device)
        num_nodes = data_d.x.size(0)
        # Build a callable game over node coalitions: f({S}) := target-class
        # margin when only nodes in S are unmasked. Matches our hard-masking.
        def _value_fn(coalitions):
            import numpy as _np
            out = _np.zeros(len(coalitions), dtype=_np.float64)
            for k, c in enumerate(coalitions):
                keep = {int(i) for i in range(num_nodes) if bool(c[i])}
                masked = _mask_nodes(data_d, keep, baseline_val=0.0)
                z, _ = _compute_logit(model, masked, device)
                out[k] = z
            return out

        # shapiq has a high-level Explainer(model=..., index=...) API but it
        # expects a fitted ML model, not a custom game. For a custom game
        # over node coalitions we use PermutationSamplingSTII directly,
        # which IS the canonical ST estimator and matches paper §4.1.
        from shapiq import PermutationSamplingSTII
        from shapiq.game_theory.aggregation import aggregate_to_one_dimension
        from shapiq import InteractionValues

        # Wrap _value_fn into a shapiq Game subclass on the fly.
        from shapiq import Game as _ShapiqGame
        class _NodeGame(_ShapiqGame):
            def __init__(self, n_players):
                super().__init__(n_players=n_players, normalize=False)
            def value_function(self, coalitions):
                return _value_fn(coalitions)
        game = _NodeGame(num_nodes)
        approx = PermutationSamplingSTII(n=num_nodes, max_order=order, top_order=False)
        interaction_values = approx.approximate(budget=budget, game=game)
        # Extract main-effect (order-1) and interaction (order-2) tensors,
        # aggregate per-node as in run_mage.
        import numpy as _np
        scores = _np.zeros(num_nodes, dtype=_np.float64)
        try:
            phi = interaction_values.get_n_order_values(1)
            scores += _np.asarray(phi).reshape(num_nodes)
        except Exception as _e:
            logger.warning(f"GraphSHAP-IQ order-1 extract failed: {_e}")
        try:
            phi_ij = interaction_values.get_n_order_values(2)
            arr = _np.asarray(phi_ij)
            if arr.shape == (num_nodes, num_nodes):
                scores += _np.abs(arr).sum(axis=1) * 0.5
        except Exception as _e:
            logger.warning(f"GraphSHAP-IQ order-2 extract failed: {_e}")
        return scores
    except Exception as e:
        logger.warning(f"GraphSHAP-IQ failed: {e}")
        return None


def _sme_enumerate_fragments(mol):
    """Build a SME-style fragment cover from a molecule:
       BRICS bonds  +  Murcko scaffold  +  a small functional-group catalogue.
    Returns a list of {"atoms": frozenset[int], "name": str} dicts.
    """
    fragments = []
    try:
        from rdkit.Chem import BRICS, AllChem
        from rdkit.Chem.Scaffolds import MurckoScaffold
    except ImportError:
        return fragments

    # BRICS: produces SMILES of fragments. Map back to atom indices via
    # GetMolFrags after cutting BRICS bonds.
    try:
        brics_bonds = list(BRICS.FindBRICSBonds(mol))
        if brics_bonds:
            bond_indices = []
            for (a_idx, b_idx), _label in brics_bonds:
                bond = mol.GetBondBetweenAtoms(a_idx, b_idx)
                if bond is not None:
                    bond_indices.append(bond.GetIdx())
            if bond_indices:
                from rdkit import Chem as _Chem
                fragged = _Chem.FragmentOnBonds(mol, bond_indices)
                # Pass asMols=False to get tuples of atom indices per fragment.
                groups = _Chem.GetMolFrags(fragged, asMols=False, sanitizeFrags=False)
                for g in groups:
                    atoms = frozenset(a for a in g if a < mol.GetNumAtoms())
                    if atoms:
                        fragments.append({"atoms": atoms, "name": "BRICS"})
    except Exception:
        pass

    # Murcko scaffold
    try:
        scaffold = MurckoScaffold.GetScaffoldForMol(mol)
        if scaffold is not None and scaffold.GetNumAtoms() > 0:
            # Map scaffold atoms back via substructure match.
            match = mol.GetSubstructMatch(scaffold)
            if match:
                fragments.append({"atoms": frozenset(match), "name": "Murcko"})
    except Exception:
        pass

    # 8 representative functional groups (SMARTS catalogue from SME paper).
    _FG_SMARTS = [
        ("Nitro",          "[NX3+](=O)[O-]"),
        ("Carboxyl",       "[CX3](=O)[OX2H1]"),
        ("Hydroxyl",       "[OX2H]"),
        ("Carbonyl",       "[CX3]=[OX1]"),
        ("Amine",          "[NX3;H2,H1,H0;!$(NC=O)]"),
        ("Amide",          "[NX3][CX3](=[OX1])"),
        ("Aromatic-ring",  "a1aaaaa1"),
        ("Halide",         "[F,Cl,Br,I]"),
    ]
    try:
        from rdkit import Chem
        for fg_name, smarts in _FG_SMARTS:
            patt = Chem.MolFromSmarts(smarts)
            if patt is None:
                continue
            for match in mol.GetSubstructMatches(patt):
                fragments.append({"atoms": frozenset(match), "name": fg_name})
    except Exception:
        pass
    return fragments


def run_gstarx(model, data, device, num_classes=2,
                max_sample_size=10, tau=0.01, num_samples=10, k=3):
    """GStarX — Structure-Aware Cooperative Game Theory (Zhang et al., NeurIPS 2022).

    Computes Hamiache-Navarro values over node coalitions; structure-aware via
    graph-based payoff aggregation. Returns per-node scores in [-1, 1].

    Requires the official repo cloned to `_vendor/GStarX/`.
    """
    try:
        import sys as _sys, os as _os
        for _cand in (_os.path.abspath("_vendor/GStarX"),
                      _os.path.abspath("../_vendor/GStarX"),
                      _os.path.abspath("../../_vendor/GStarX")):
            if _os.path.isdir(_cand) and _cand not in _sys.path:
                _sys.path.insert(0, _cand)
        from gstarx import GStarX as _GStarX
    except ImportError as _e:
        if not hasattr(run_gstarx, "_warned"):
            logger.warning(
                f"GStarX not importable ({_e}); skipping. Clone "
                "github.com/ShichangZh/GStarX into _vendor/GStarX/."
            )
            run_gstarx._warned = True
        return None

    try:
        data_d = data.clone().to(device)

        # GStarX calls `model(data=data)` (positional + kw style). Our
        # BenchmarkGCN.forward takes `(x, edge_index, batch)`. Wrap with an
        # adapter that translates the keyword.
        import torch.nn as _nn
        class _GStarXAdapter(_nn.Module):
            def __init__(self, inner):
                super().__init__()
                self.inner = inner
            def forward(self, data=None, **kw):
                if data is None and "x" in kw:
                    return self.inner(kw["x"], kw.get("edge_index"),
                                      kw.get("batch"))
                # Standard Data-object call path
                out = self.inner(data)
                # GStarX expects 2-class logits even for single-output binary.
                if out.dim() == 1:
                    return torch.stack([-out, out], dim=-1)
                if out.dim() >= 2 and out.shape[-1] == 1:
                    return torch.cat([-out, out], dim=-1)
                return out

        wrapped = _GStarXAdapter(model).to(device)
        explainer = _GStarX(
            wrapped, device=device,
            max_sample_size=max_sample_size, tau=tau,
            payoff_type="prob",  # avoids needing precomputed payoff_avg
        )
        scores = explainer.explain(
            data_d, superadditive_ext=True,
            sample_method="khop", num_samples=num_samples, k=k,
        )
        import numpy as _np
        return _np.asarray(scores, dtype=_np.float64)
    except Exception as e:
        logger.warning(f"GStarX failed: {e}")
        return None


def run_insideshap(model, data, device, num_classes=2):
    """INSIDE-Shap (Kamal et al., DMKD 2025) — Shapley values over learned
    activation rules. Code at github.com/atakml/INSIDE_SHAP.

    NOT integrated as a drop-in baseline because INSIDE-Shap is a 6-step
    pipeline (train black-box GNN, mine activation patterns, train surrogate
    GIN, compute Shapley over rules, generate rule masks) tied to its own
    model architecture and pattern-mining routine. Adapting it to a
    pre-trained BenchmarkGCN would require porting their pattern miner and
    surrogate, an undertaking of several days.

    We cite INSIDE-Shap in Related Work (§2.3) as the closest prior work that
    computes Shapley values over learned *internal representations* rather
    than over the structural cooperative game over node coalitions.

    For now this returns None so the benchmark sweep skips it.
    """
    if not hasattr(run_insideshap, "_warned"):
        logger.info(
            "INSIDE-Shap not integrated as drop-in baseline (requires its own "
            "6-step training pipeline). Discussed in Related Work §2.3."
        )
        run_insideshap._warned = True
    return None


def run_graphchef(model, data, device, num_classes=2):
    """GraphChef (Mueller et al., ICLR 2024) — self-explainable GNN with
    integrated decision trees.

    NOT integrated as a drop-in baseline because GraphChef is a
    *self-explainable model*, not a post-hoc explainer. It must replace the
    GNN being explained, not run on top of an existing one. Comparing it
    head-to-head requires training a separate GraphChef model from scratch
    for each dataset, which is a different evaluation paradigm.

    GraphChef code ships only as the OpenReview supplementary attachment
    (no GitHub release), with pinned dependencies (torch==1.11, PyG==2.0.4)
    that conflict with the rest of our environment (torch 2.12, PyG 2.7).

    Cited in Related Work §2.4 as orthogonal: their explanation is the
    model itself (decision tree rules), ours is a post-hoc decomposition
    of an arbitrary trained GNN.

    For now returns None.
    """
    if not hasattr(run_graphchef, "_warned"):
        logger.info(
            "GraphChef not integrated as drop-in baseline (self-explainable "
            "model, different paradigm). Discussed in Related Work §2.4."
        )
        run_graphchef._warned = True
    return None


def run_sme(model, data, smiles, device, num_classes=2):
    """SME — Substructure Mask Explanation
    (Wu et al., Nature Communications 14:2585, 2023).

    Re-implementation on top of our PyG pipeline (the official code is DGL).
    For each chemistry-informed fragment, score = f(G) minus f(G ∖ fragment).
    Returns per-atom scores (aggregated by averaging the fragment-score over
    constituent atoms, matching the SME paper's atom-level visualisation).
    Returns None if RDKit cannot parse the SMILES.
    """
    try:
        from rdkit import Chem
    except ImportError:
        return None
    if smiles is None or smiles == "":
        return None
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fragments = _sme_enumerate_fragments(mol)
    if not fragments:
        return None
    try:
        num_nodes = data.x.size(0)
        # Full-graph target logit
        s_full, target_class = _compute_logit(model, data, device)
        import numpy as _np
        atom_counts = _np.zeros(num_nodes, dtype=_np.float64)
        atom_scores = _np.zeros(num_nodes, dtype=_np.float64)
        for frag in fragments:
            atoms = [a for a in frag["atoms"] if 0 <= a < num_nodes]
            if not atoms:
                continue
            drop_set = set(atoms)
            keep_set = set(range(num_nodes)) - drop_set
            masked = _mask_nodes(data, keep_set, baseline_val=0.0)
            s_drop, _ = _compute_logit(model, masked, device,
                                       target_class=target_class)
            attribution = s_full - s_drop  # SME's per-fragment score
            for a in atoms:
                atom_scores[a] += attribution
                atom_counts[a] += 1.0
        nonzero = atom_counts > 0
        atom_scores[nonzero] /= atom_counts[nonzero]
        return atom_scores
    except Exception as e:
        logger.warning(f"SME failed: {e}")
        return None



In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# §6  EVALUATION ENGINE
# ═══════════════════════════════════════════════════════════════════════════


def _compute_logit(model, data, device, target_class=None):
    """Get the class-margin logit for a single graph.

    For multi-class (output dim >= 2): returns z[target] - z[runner_up].
    This is the log-odds that directly measures prediction confidence.
    Unlike a single-class logit, the margin captures the FULL effect
    of masking: when both logits shift together (model becomes less
    confident overall), single-class logit barely changes, but the
    margin changes proportionally to the actual prediction shift.

    For single-output binary: returns the raw scalar logit (already a margin).

    Returns (margin_value, class_index).
    """
    data = data.clone().to(device)
    if not hasattr(data, "batch") or data.batch is None:
        data.batch = torch.zeros(data.x.size(0), dtype=torch.long, device=device)
    model.eval()
    with torch.no_grad():
        out = model(data)
    if out.dim() >= 2 and out.shape[-1] >= 2:
        logits = out[0]
        if target_class is None:
            pred = logits.argmax().item()
        else:
            pred = target_class
        # Margin = z[target] - max(z[others])
        z_target = logits[pred].item()
        mask = torch.ones(logits.shape[0], dtype=torch.bool)
        mask[pred] = False
        z_runner = logits[mask].max().item()
        margin = z_target - z_runner
        return float(margin), pred
    # Single-output binary: logit IS the margin
    if out.dim() == 1:
        val = float(out[0].item())
        return val, int(val > 0)
    val = float(out.item())
    return val, int(val > 0)


def _mask_nodes(data, keep_nodes, baseline_val=0.0):
    """Copy data with non-selected nodes masked.

    Hard masking: zero features AND remove edges involving masked nodes.
    This is critical for topology-dependent datasets like BA-2Motifs
    where the model detects motifs by graph structure, not node features.
    Soft masking (zero features only) leaves edges intact, so the GCN
    can still detect motif topology through message passing even when
    features are zeroed.
    """
    data_new = data.clone()
    data_new.x = data.x.clone()
    n = data.x.size(0)
    keep_set = set()
    for nd in keep_nodes:
        if 0 <= nd < n:
            keep_set.add(nd)

    # Zero features of masked nodes
    mask = torch.zeros(n, dtype=torch.bool, device=data.x.device)
    for nd in keep_set:
        mask[nd] = True
    data_new.x[~mask] = baseline_val

    # Remove edges involving masked nodes
    if hasattr(data, 'edge_index') and data.edge_index is not None:
        ei = data.edge_index
        src_ok = torch.zeros(n, dtype=torch.bool, device=ei.device)
        for nd in keep_set:
            src_ok[nd] = True
        edge_mask = src_ok[ei[0]] & src_ok[ei[1]]
        data_new.edge_index = ei[:, edge_mask]
        # Also filter edge_attr if present
        if hasattr(data, 'edge_attr') and data.edge_attr is not None:
            data_new.edge_attr = data.edge_attr[edge_mask]

    return data_new


def _eval_fidelity(model, data, expl_nodes, device):
    """Fidelity computation in logit space with normalized probability variant.

    Primary metrics (logit-space, no saturation):
      fid_plus  = z_full - z_drop   (necessity: positive if explanation matters)
      fid_minus = z_full - z_keep   (sufficiency: near-zero if explanation is sufficient)

    Normalized probability metrics (bounded [0,1]):
      prob_fid_plus/minus use sigmoid with adaptive temperature to prevent
      saturation while preserving relative ordering.

    Returns dict with all metrics.
    """
    num_nodes = data.x.size(0)
    all_nodes = set(range(num_nodes))
    non_expl = all_nodes - expl_nodes

    # Raw logits for target class (locked from full graph)
    z_full, target_class = _compute_logit(model, data, device)
    z_drop, _ = _compute_logit(model, _mask_nodes(data, non_expl), device,
                               target_class=target_class)
    z_keep, _ = _compute_logit(model, _mask_nodes(data, expl_nodes), device,
                               target_class=target_class)

    # Logit-space fidelity (primary — no saturation)
    fid_plus = z_full - z_drop
    fid_minus = z_full - z_keep

    # Normalized probability fidelity:
    # Use adaptive temperature so that sigmoid(z/T) doesn't saturate.
    # T = max(|z_full|, |z_drop|, |z_keep|, 1) / 3
    # This maps the largest logit to sigmoid(~3) ≈ 0.95, leaving room
    # for variation.
    z_max = max(abs(z_full), abs(z_drop), abs(z_keep), 1.0)
    temp = z_max / 3.0

    def _sig_t(z):
        return 1.0 / (1.0 + math.exp(-max(min(z / temp, 30), -30)))

    p_full = _sig_t(z_full)
    p_drop = _sig_t(z_drop)
    p_keep = _sig_t(z_keep)
    prob_fp = p_full - p_drop
    prob_fm = p_full - p_keep

    # Harmonic fidelity (GStarX formula, using normalized probs)
    ratio = len(expl_nodes) / max(num_nodes, 1)
    m1 = max(-1.0, min(1.0, prob_fp * (1.0 - ratio)))
    m2 = max(-1.0, min(1.0, prob_fm * ratio))
    denom = 2.0 + m1 - m2
    h_val = ((1.0 + m1) * (1.0 - m2)) / denom if abs(denom) > 1e-12 else 0.0

    # ── GraphFramEx-canonical extensions (Amara et al., LoG 2022) ──────────
    # Accuracy-space (indicator-based) Fid+ / Fid-: argmax-equality test.
    # Model-focus convention: |1{pred(G) == pred_full} - 1{pred(G_S/_neg) == pred_full}|.
    def _argmax_class(z):
        # Re-evaluate via the full model (target class is already locked to
        # full_graph prediction at evaluation time).
        return 1 if z > 0 else 0  # binary single-logit; multi-class handled in get_target_logit
    pred_full = 1 if z_full > 0 else 0
    pred_drop = 1 if z_drop > 0 else 0
    pred_keep = 1 if z_keep > 0 else 0
    fid_plus_acc = 1.0 if pred_drop != pred_full else 0.0
    fid_minus_acc = 1.0 if pred_keep != pred_full else 0.0

    # Probability-space Fid+ / Fid- without adaptive temperature (canonical
    # GraphFramEx / GStarX / SubgraphX form).
    def _sig_raw(z):
        return 1.0 / (1.0 + math.exp(-max(min(z, 30), -30)))
    p_full_raw = _sig_raw(z_full)
    p_drop_raw = _sig_raw(z_drop)
    p_keep_raw = _sig_raw(z_keep)
    fid_plus_prob = p_full_raw - p_drop_raw
    fid_minus_prob = p_full_raw - p_keep_raw

    # Characterization score (GraphFramEx). Weighted harmonic of necessity
    # (Fid+) and sufficiency (1 - Fid-). Defaults w+ = w- = 0.5.
    # Operates on accuracy-space numbers since charact is conventionally
    # accuracy-based; we also expose a probability-space variant.
    def _charact(fp, fm, w_plus=0.5, w_minus=0.5):
        denom = w_plus * (1.0 - fm) + w_minus * fp
        if abs(denom) < 1e-12:
            return 0.0
        return (w_plus + w_minus) * fp * (1.0 - fm) / denom
    charact_acc = _charact(fid_plus_acc, fid_minus_acc)
    charact_prob = _charact(max(0.0, fid_plus_prob), max(0.0, fid_minus_prob))

    return {
        "s_full": z_full, "s_drop": z_drop, "s_keep": z_keep,
        # logit space
        "fid_plus": fid_plus, "fid_minus": fid_minus,
        # tempered probability space (for H-Fid harmonic)
        "prob_fid_plus": prob_fp, "prob_fid_minus": prob_fm,
        "harmonic_fidelity": h_val, "m1": m1, "m2": m2,
        # untempered probability space (GraphFramEx / GStarX canonical)
        "fid_plus_prob": fid_plus_prob,
        "fid_minus_prob": fid_minus_prob,
        # accuracy space (GraphFramEx indicator-based)
        "fid_plus_acc": fid_plus_acc,
        "fid_minus_acc": fid_minus_acc,
        # characterization scores (GraphFramEx weighted harmonic)
        "characterization_acc": charact_acc,
        "characterization_prob": charact_prob,
    }


def evaluate_one_graph(
    model: nn.Module,
    data: Data,
    node_scores: np.ndarray,
    device: torch.device,
    sparsity_grid: List[float],
    method_name: str,
    graph_idx: int,
    ground_truth_nodes: Optional[Set[int]] = None,
) -> List[Dict]:
    """Evaluate a single explanation at multiple sparsity levels.

    For each sparsity in the grid, select the top nodes, compute:
      - fid+, fid-, sufficiency, sparsity, harmonic fidelity
      - localization (if ground truth available)

    Returns a list of result dicts (one per sparsity level).
    """
    records = []
    num_nodes = data.x.size(0)

    for target_sp in sparsity_grid:
        expl_set = node_scores_to_explanation(node_scores, target_sp)
        actual_sp = 1.0 - len(expl_set) / num_nodes if num_nodes > 0 else 1.0

        # Fidelity metrics (standalone — no pipeline dependency)
        try:
            fid = _eval_fidelity(model, data, expl_set, device)
            fid_plus_val = fid["fid_plus"]
            fid_minus_val = fid["fid_minus"]
            prob_fp = fid["prob_fid_plus"]
            prob_fm = fid["prob_fid_minus"]
            h_val = fid["harmonic_fidelity"]
            hm = {"m1": fid["m1"], "m2": fid["m2"]}
        except Exception as e:
            if graph_idx < 3 and target_sp == sparsity_grid[0]:
                logger.warning(f"Fidelity failed (graph {graph_idx}): {e}")
            fid_plus_val, fid_minus_val, h_val = 0., 0., 0.
            prob_fp, prob_fm = 0., 0.
            hm = {"m1": 0., "m2": 0.}

        rec = {
            "method": method_name,
            "graph_idx": graph_idx,
            "num_nodes": num_nodes,
            "target_sparsity": target_sp,
            "actual_sparsity": actual_sp,
            "num_selected": len(expl_set),
        }
        # Pull every fidelity metric (including the GraphFramEx-canonical
        # additions: characterization_*, fid_*_acc, fid_*_prob) into the
        # baseline record so all methods get the same metric columns.
        if "fid" in dir():
            try:
                rec.update({k: v for k, v in fid.items()
                            if isinstance(v, (int, float))})
            except Exception:
                pass
        # Fall back to legacy explicit keys (some paths may not have `fid`).
        rec.setdefault("fid_plus", fid_plus_val)
        rec.setdefault("fid_minus", fid_minus_val)
        rec.setdefault("prob_fid_plus", prob_fp)
        rec.setdefault("prob_fid_minus", prob_fm)
        rec.setdefault("harmonic_fidelity", h_val)
        rec.setdefault("m1", hm["m1"])
        rec.setdefault("m2", hm["m2"])

        # Localization (if ground truth available)
        if ground_truth_nodes is not None and len(ground_truth_nodes) > 0:
            total_score = np.sum(np.abs(node_scores))
            if total_score > 1e-12:
                gt_score = sum(abs(node_scores[n]) for n in ground_truth_nodes
                               if n < len(node_scores))
                rec["localization"] = gt_score / total_score
            else:
                rec["localization"] = 0.0

            # Precision/recall of selected set vs ground truth
            selected = expl_set
            tp = len(selected & ground_truth_nodes)
            rec["gt_precision"] = tp / max(len(selected), 1)
            rec["gt_recall"] = tp / max(len(ground_truth_nodes), 1)
            rec["gt_f1"] = (2 * rec["gt_precision"] * rec["gt_recall"]
                            / max(rec["gt_precision"] + rec["gt_recall"], 1e-12))

            # AUC of node scores vs. ground-truth membership (PGExplainer
            # / GNNExplainer canonical) — uses RAW scores, not thresholded.
            auc_f1 = evaluate_gt_auc_f1(
                node_scores, ground_truth_nodes, num_nodes,
                top_k=len(ground_truth_nodes),
            )
            rec.update(auc_f1)

        # Robust α-fidelity (Zheng et al. 2023; primary metric in MAGE).
        # Wall-clock cost is significant (200 model calls per sparsity per graph),
        # so we only compute it at the smallest sparsity (most-discriminating).
        if abs(target_sp - sparsity_grid[0]) < 1e-6:
            try:
                rob = evaluate_robust_fidelity(
                    model, data, expl_set, device,
                    alpha=0.7, num_samples=50, seed=42,
                )
                rec.update(rob)
            except Exception:
                pass

        records.append(rec)

    return records


def get_ground_truth_nodes(data: Data,
                           dataset_name: str = "") -> Optional[Set[int]]:
    """Extract ground-truth explanation nodes from a DIG dataset sample.

    BA-2Motifs: ground truth is implicit — the first 20 nodes are the
    BA base graph and nodes >= 20 are the attached motif (house or cycle).
    This is hardcoded in DIG's SynGraphDataset.gen_motif_edge_mask():
        torch.logical_and(data.edge_index[0] >= 20, data.edge_index[1] >= 20)

    Other datasets may store ground truth in data.node_label,
    data.edge_label, or data.ground_truth_mask attributes.
    """
    gt_nodes = set()

    # BA-2Motifs: motif nodes are indices >= 20 (out of 25 total)
    if dataset_name.lower() in ("ba2motifs", "ba_2motifs"):
        num_nodes = data.x.size(0) if data.x is not None else 0
        if num_nodes == 25:
            return set(range(20, 25))
        elif num_nodes > 20:
            # Variable-size graphs: motif is last nodes attached to BA base
            # BA base is 20 nodes, motif is the rest
            return set(range(20, num_nodes))
        return None

    # Check for explicit node labels
    if hasattr(data, "node_label"):
        nl = data.node_label
        if isinstance(nl, torch.Tensor):
            for i in range(nl.size(0)):
                if nl[i].item() > 0:
                    gt_nodes.add(i)
        if gt_nodes:
            return gt_nodes

    # Check for edge labels (convert to nodes)
    if hasattr(data, "edge_label") and hasattr(data, "edge_index"):
        el = data.edge_label
        if isinstance(el, torch.Tensor):
            ei = data.edge_index
            for k in range(el.size(0)):
                if el[k].item() > 0:
                    gt_nodes.add(int(ei[0, k].item()))
                    gt_nodes.add(int(ei[1, k].item()))
        if gt_nodes:
            return gt_nodes

    # Check for ground_truth_mask
    if hasattr(data, "ground_truth_mask"):
        mask = data.ground_truth_mask
        if isinstance(mask, torch.Tensor):
            for i in range(mask.size(0)):
                if mask[i].item() > 0:
                    gt_nodes.add(i)
        if gt_nodes:
            return gt_nodes

    return None if not gt_nodes else gt_nodes


# ═══════════════════════════════════════════════════════════════════════════
# §7  FULL BENCHMARK EVALUATION LOOP
# ═══════════════════════════════════════════════════════════════════════════

def run_benchmark_evaluation(
    dataset_name: str,
    device: torch.device = None,
    n_explain: int = 50,
    model_class: str = "gcn",
) -> Tuple[pd.DataFrame, Dict]:
    """Run complete benchmark evaluation on one DIG dataset.

    1. Load dataset
    2. Train GNN
    3. Run all baselines + our method
    4. Evaluate at multiple sparsity levels
    5. Return results DataFrame and summary

    Parameters
    ----------
    dataset_name : str
        One of 'ba2motifs', 'mutag', 'graph_sst2', 'bbbp', 'bace'
    device : torch.device
    n_explain : int
        Number of test graphs to explain.
    model_class : str
        'gcn' or 'gin'
    """
    if device is None:
        device = EVAL_CFG.DEVICE

    print(f"\n{'='*70}")
    print(f"BENCHMARK: {dataset_name} (model={model_class})")
    print(f"{'='*70}")

    # ── Load dataset ──────────────────────────────────────────────────
    dataset, info = load_dig_dataset(dataset_name)
    splits = split_dataset(dataset, seed=42)
    input_dim = info["input_dim"]
    num_classes = info.get("num_classes", 2)

    # ── Train model ───────────────────────────────────────────────────
    if model_class == "gin":
        model = BenchmarkGIN(input_dim, EVAL_CFG.HIDDEN, num_classes)
    else:
        model = BenchmarkGCN(input_dim, EVAL_CFG.HIDDEN, num_classes)

    model, train_info = train_benchmark_model(
        model, splits["train"], splits["val"], dataset_name, device,
        epochs=EVAL_CFG.EPOCHS, lr=EVAL_CFG.LR,
        batch_size=EVAL_CFG.BATCH_SIZE, patience=EVAL_CFG.PATIENCE)

    # ── Select test graphs to explain ─────────────────────────────────
    test_data = splits["test"]
    n_eval = min(n_explain, len(test_data))
    rng = np.random.RandomState(42)
    eval_indices = rng.choice(len(test_data), size=n_eval, replace=False)
    eval_graphs = [test_data[int(i)] for i in eval_indices]

    print(f"  Explaining {n_eval} test graphs")

    # All explainers are now DIG-independent:
    #   GNNExplainer  → PyG native (with DIG fallback)
    #   PGExplainer   → PyG native
    #   SubgraphX     → standalone MCTS implementation
    #   Our method    → standalone via framework primitives

    # ── Train PGExplainer (uses PyG native) ───────────────────────────
    pg_explainer = None
    try:
        print("  Training PGExplainer...")
        pg_explainer = train_pgexplainer(
            model, splits["train"][:200],
            device, epochs=EVAL_CFG.PGE_EPOCHS, lr=EVAL_CFG.PGE_LR)
    except Exception as e:
        print(f"  PGExplainer training failed: {e}")
        pg_explainer = None

    # ── Run all methods ───────────────────────────────────────────────
    all_records = []
    methods_run = set()
    _skip_methods = set()  # methods that failed on first graph

    # Suppress repeated warnings (one per method is enough)
    class _DeduplicateFilter(logging.Filter):
        def __init__(self):
            super().__init__()
            self._seen = set()
        def filter(self, record):
            key = record.getMessage()[:60]
            if key in self._seen:
                return False
            self._seen.add(key)
            return True

    _dedup = _DeduplicateFilter()
    logger.addFilter(_dedup)

    for gi, data in enumerate(eval_graphs):
        data = data.to(device)
        num_nodes = data.x.size(0)
        gt_nodes = get_ground_truth_nodes(data, dataset_name=dataset_name)
        t0 = time.time()

        # --- Grad×Input ---
        gxi_scores = run_grad_x_input(model, data, device)
        if gxi_scores is not None:
            recs = evaluate_one_graph(
                model, data, gxi_scores, device,
                EVAL_CFG.SPARSITY_GRID, "grad_x_input", gi, gt_nodes)
            all_records.extend(recs)
            methods_run.add("grad_x_input")

        # --- GNNExplainer (PyG native, DIG fallback) ---
        if "gnnexplainer" not in _skip_methods:
            gnn_scores = run_gnnexplainer(model, data, device, num_classes)
            if gnn_scores is not None:
                recs = evaluate_one_graph(
                    model, data, gnn_scores, device,
                    EVAL_CFG.SPARSITY_GRID, "gnnexplainer", gi, gt_nodes)
                all_records.extend(recs)
                methods_run.add("gnnexplainer")
            elif gi == 0:
                _skip_methods.add("gnnexplainer")
                print("  GNNExplainer failed on first graph — skipping")

        # --- PGExplainer ---
        if pg_explainer is not None and "pgexplainer" not in _skip_methods:
            pge_scores = run_pgexplainer(pg_explainer, data, device)
            if pge_scores is not None:
                recs = evaluate_one_graph(
                    model, data, pge_scores, device,
                    EVAL_CFG.SPARSITY_GRID, "pgexplainer", gi, gt_nodes)
                all_records.extend(recs)
                methods_run.add("pgexplainer")
            elif gi == 0:
                _skip_methods.add("pgexplainer")
                print("  PGExplainer failed on first graph — skipping")

        # --- SubgraphX (expensive — limit to fewer graphs) ---
        # --- SubgraphX (standalone, expensive — limit to fewer graphs) ---
        if gi < min(EVAL_CFG.SUBX_MAX_GRAPHS, n_eval) and "subgraphx" not in _skip_methods:
            subx_scores = run_subgraphx(
                model, data, device, num_classes,
                min_atoms=EVAL_CFG.SUBX_MIN_ATOMS,
                rollout=EVAL_CFG.SUBX_ROLLOUT,
                sample_num=EVAL_CFG.SUBX_SAMPLE_NUM)
            if subx_scores is not None:
                recs = evaluate_one_graph(
                    model, data, subx_scores, device,
                    EVAL_CFG.SPARSITY_GRID, "subgraphx", gi, gt_nodes)
                all_records.extend(recs)
                methods_run.add("subgraphx")
            elif gi == 0:
                _skip_methods.add("subgraphx")
                print("  SubgraphX failed on first graph — skipping")

        # --- MAGE (Bui et al., ICML 2024) ---
        if "mage" not in _skip_methods:
            mage_scores = run_mage(model, data, device, num_classes)
            if mage_scores is not None:
                recs = evaluate_one_graph(
                    model, data, mage_scores, device,
                    EVAL_CFG.SPARSITY_GRID, "mage", gi, gt_nodes)
                all_records.extend(recs)
                methods_run.add("mage")
            elif gi == 0:
                _skip_methods.add("mage")
                print("  MAGE failed on first graph (likely not installed) - skipping")

        # --- GraphSHAP-IQ (Fumagalli et al., ICLR 2025) ---
        # Requires linear-readout GNN (BenchmarkGCN / BenchmarkGIN); restrict
        # to the first few graphs since exact estimation is expensive.
        if gi < min(5, n_eval) and "graphshapiq" not in _skip_methods:
            gsi_scores = run_graphshapiq(model, data, device, num_classes)
            if gsi_scores is not None:
                recs = evaluate_one_graph(
                    model, data, gsi_scores, device,
                    EVAL_CFG.SPARSITY_GRID, "graphshapiq", gi, gt_nodes)
                all_records.extend(recs)
                methods_run.add("graphshapiq")
            elif gi == 0:
                _skip_methods.add("graphshapiq")
                print("  GraphSHAP-IQ failed on first graph - skipping "
                      "(not installed, or model has non-linear readout)")

        # --- GStarX (Zhang et al., NeurIPS 2022) ---
        if "gstarx" not in _skip_methods:
            gst_scores = run_gstarx(model, data, device, num_classes)
            if gst_scores is not None:
                recs = evaluate_one_graph(
                    model, data, gst_scores, device,
                    EVAL_CFG.SPARSITY_GRID, "gstarx", gi, gt_nodes)
                all_records.extend(recs)
                methods_run.add("gstarx")
            elif gi == 0:
                _skip_methods.add("gstarx")
                print("  GStarX failed on first graph - skipping")

        # --- SME (Wu et al., Nature Comm 2023) — molecular datasets only ---
        # We need a SMILES string to fragment; attempt to read it from the
        # Data object (PyG MoleculeNet attaches it as `data.smiles`).
        if "sme" not in _skip_methods:
            smi = getattr(data, "smiles", None)
            if smi is not None:
                sme_scores = run_sme(model, data, smi, device, num_classes)
                if sme_scores is not None:
                    recs = evaluate_one_graph(
                        model, data, sme_scores, device,
                        EVAL_CFG.SPARSITY_GRID, "sme", gi, gt_nodes)
                    all_records.extend(recs)
                    methods_run.add("sme")
                elif gi == 0:
                    _skip_methods.add("sme")
                    print("  SME failed on first graph - skipping")

        # --- Our method ---
        ours_result = run_our_method(
            model, data, device,
            n_permutations=EVAL_CFG.N_PERM,
            sa_iterations=EVAL_CFG.SA_ITER)
        if ours_result is not None:
            node_scores, partition, group_agg, sg = ours_result
            recs = evaluate_one_graph(
                model, data, node_scores, device,
                EVAL_CFG.SPARSITY_GRID, "ours", gi, gt_nodes)

            # Also add group-level sparsity evaluation (our unique output)
            # Evaluate at natural group-level sparsities: top-1, top-2, ..., top-k groups.
            # Each sparsity grid point maps to the closest prefix.
            try:
                gv = group_agg.get("group_values", [])
                cross = group_agg.get("cross_scores", [])
                if gv and len(partition) > 0:
                    # ROUND 2 TWEAK: augment group score with cross-group
                    # interaction magnitudes. Groups with strong synergy
                    # partners rank higher.
                    def _aug_score(i):
                        intrinsic = abs(gv[i])
                        if i < len(cross):
                            cross_sum = sum(abs(cross[i][j]) for j in range(len(cross[i]))
                                            if j != i and j < len(cross))
                        else:
                            cross_sum = 0.0
                        return intrinsic + 0.5 * cross_sum
                    ranked = sorted(range(len(gv)),
                                    key=_aug_score, reverse=True)
                    # Build cumulative prefixes
                    prefixes = []  # (node_set, natural_sparsity)
                    cumul = set()
                    for gi_sel in ranked:
                        if gi_sel < len(partition):
                            cumul = cumul | set(partition[gi_sel])
                            prefixes.append((set(cumul), 1.0 - len(cumul) / num_nodes))

                    if prefixes:
                        for target_sp in EVAL_CFG.SPARSITY_GRID:
                            # Pick prefix with sparsity closest to target
                            best_idx = min(range(len(prefixes)),
                                           key=lambda i: abs(prefixes[i][1] - target_sp))
                            expl_nodes, actual_sp = prefixes[best_idx]

                            fid_dict = _eval_fidelity(model, data, expl_nodes, device)
                            fid_dict["method"] = "ours_groups"
                            fid_dict["graph_idx"] = gi
                            fid_dict["target_sparsity"] = target_sp
                            fid_dict["actual_sparsity"] = actual_sp
                            fid_dict["num_selected"] = len(expl_nodes)
                            fid_dict["num_nodes"] = num_nodes
                            if gt_nodes is not None:
                                tp = len(expl_nodes & gt_nodes)
                                fid_dict["gt_precision"] = tp / max(len(expl_nodes), 1)
                                fid_dict["gt_recall"] = tp / max(len(gt_nodes), 1)
                            all_records.append(fid_dict)
            except Exception:
                pass

            all_records.extend(recs)
            methods_run.add("ours")
            methods_run.add("ours_groups")

        elapsed = time.time() - t0
        if (gi + 1) % 5 == 0 or gi == 0:
            print(f"  [{gi+1}/{n_eval}]  {elapsed:.1f}s  "
                  f"n={num_nodes}  methods={len(methods_run)}")

    logger.removeFilter(_dedup)

    # ── Build results ─────────────────────────────────────────────────
    df = pd.DataFrame(all_records)
    df["dataset"] = dataset_name
    df["model"] = model_class

    summary = {
        "dataset": dataset_name, "model": model_class,
        "n_evaluated": n_eval,
        "methods": sorted(methods_run),
        "train_acc": train_info.get("best_val_acc", 0),
    }

    if not df.empty:
        print(f"\n  --- Results Summary ---")
        for method in sorted(df["method"].unique()):
            msub = df[df["method"] == method]
            for sp in EVAL_CFG.SPARSITY_GRID:
                sp_sub = msub[abs(msub["target_sparsity"] - sp) < 0.01]
                if sp_sub.empty:
                    continue
                # Use probability-space fidelity if available
                fp_col = "prob_fid_plus" if "prob_fid_plus" in sp_sub.columns else "fid_plus"
                fm_col = "prob_fid_minus" if "prob_fid_minus" in sp_sub.columns else "fid_minus"
                fp = sp_sub[fp_col].mean()
                fm = sp_sub[fm_col].mean()
                hf = sp_sub["harmonic_fidelity"].mean()
                loc_str = ""
                if "localization" in sp_sub.columns:
                    loc_str = f"  loc={sp_sub['localization'].mean():.3f}"
                print(f"    {method:16s} sp={sp:.1f}  "
                      f"pfid+={fp:.4f}  pfid-={fm:.4f}  h={hf:.4f}{loc_str}")

    return df, summary



# ═══════════════════════════════════════════════════════════════════════════
# §6c  GROUND-TRUTH-BASED METRICS (PGExplainer / GNNExplainer canonical)
# ═══════════════════════════════════════════════════════════════════════════
# These metrics treat ground-truth motif nodes as positive labels and the
# explainer's importance scores as predictions. Two complementary measures:
#   - AUC: rank-based (uses raw scores, no thresholding)
#   - F1 @ top-k: set-based (top-k nodes vs. ground-truth set)


def evaluate_gt_auc_f1(node_scores, gt_nodes, num_nodes, top_k=None):
    """Compute AUC of node-importance scores vs. ground-truth motif membership,
    plus F1 at a chosen top-k threshold (default top-k = |gt_nodes|).

    Returns dict with `gt_auc`, `gt_f1`, `gt_precision_at_k`, `gt_recall_at_k`.
    If gt_nodes is empty or covers everything, returns NaN.
    """
    import numpy as _np
    from sklearn.metrics import roc_auc_score as _roc_auc

    if not gt_nodes or len(gt_nodes) >= num_nodes:
        return {"gt_auc": float("nan"), "gt_f1": float("nan"),
                "gt_precision_at_k": float("nan"), "gt_recall_at_k": float("nan")}

    y_true = _np.zeros(num_nodes, dtype=_np.int64)
    for n in gt_nodes:
        if 0 <= n < num_nodes:
            y_true[n] = 1
    scores = _np.abs(_np.asarray(node_scores[:num_nodes], dtype=_np.float64))

    # AUC
    try:
        if y_true.sum() in (0, num_nodes):
            gt_auc = float("nan")
        else:
            gt_auc = float(_roc_auc(y_true, scores))
    except Exception:
        gt_auc = float("nan")

    # F1 @ top-k
    k = int(top_k) if top_k is not None else int(y_true.sum())
    k = max(1, min(k, num_nodes))
    top_idx = set(_np.argsort(-scores)[:k].tolist())
    gt_set = set(int(n) for n in gt_nodes if 0 <= int(n) < num_nodes)
    tp = len(top_idx & gt_set)
    prec = tp / max(len(top_idx), 1)
    rec = tp / max(len(gt_set), 1)
    f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0

    return {"gt_auc": gt_auc, "gt_f1": float(f1),
            "gt_precision_at_k": float(prec), "gt_recall_at_k": float(rec)}


def evaluate_robust_fidelity(model, data, expl_nodes, device,
                              alpha=0.7, num_samples=200, seed=42):
    """Robust α-Fidelity (Zheng et al. 2023, arXiv:2310.01820) — the OOD-mitigated
    fidelity that MAGE adopted as its primary metric.

    For each Monte-Carlo sample: independently include each non-explanation node
    w.p. (1 - alpha) when computing fid+, and include each explanation node
    w.p. alpha when computing fid-. This mitigates the OOD distribution shift
    of hard-masking by smoothly interpolating.

    Returns dict with `fid_alpha_plus`, `fid_alpha_minus`, `fid_alpha_delta`.
    """
    import numpy as _np
    rng = _np.random.RandomState(seed)
    num_nodes = data.x.size(0)
    all_set = set(range(num_nodes))
    expl_set = set(int(n) for n in expl_nodes if 0 <= int(n) < num_nodes)
    non_expl = all_set - expl_set
    if num_nodes == 0:
        return {"fid_alpha_plus": 0.0, "fid_alpha_minus": 0.0, "fid_alpha_delta": 0.0}

    z_full, target_class = _compute_logit(model, data, device)

    # Fid_α+ : average f(G) - f(G with non-expl-nodes alpha-sampled)
    # i.e. we keep the explanation, plus randomly *include* (1-alpha) fraction
    # of non-explanation nodes. (alpha=1.0 means standard fid+.)
    plus_drops = []
    for _ in range(num_samples):
        keep = set(expl_set)
        for n in non_expl:
            if rng.rand() > alpha:  # drop with prob alpha
                keep.add(n)
        masked = _mask_nodes(data, keep)
        z_sub, _ = _compute_logit(model, masked, device, target_class=target_class)
        plus_drops.append(z_full - z_sub)
    fid_alpha_plus = float(_np.mean(plus_drops))

    # Fid_α- : average f(G) - f(non-expl with expl-nodes alpha-sampled)
    # we drop the explanation, plus randomly *exclude* alpha fraction of
    # explanation nodes from coming back.
    minus_drops = []
    for _ in range(num_samples):
        keep = set(non_expl)
        for n in expl_set:
            if rng.rand() > alpha:  # drop with prob alpha
                keep.add(n)
        masked = _mask_nodes(data, keep)
        z_sub, _ = _compute_logit(model, masked, device, target_class=target_class)
        minus_drops.append(z_full - z_sub)
    fid_alpha_minus = float(_np.mean(minus_drops))

    return {
        "fid_alpha_plus": fid_alpha_plus,
        "fid_alpha_minus": fid_alpha_minus,
        "fid_alpha_delta": fid_alpha_plus - fid_alpha_minus,
    }



In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# §6b  ABLATION EXPERIMENTS (reviewer R-3.D)
# ═══════════════════════════════════════════════════════════════════════════
# Three sub-experiments addressing Reviewer 2:
#   (i)   sqrt(n) sensitivity — sweep k_0 in {2, ceil(sqrt(n)), n/3, n/2}
#   (ii)  Phase 1 quality — Shapley-Taylor vs. Grad x Input main-effects
#   (iii) Phase 3 importance — SA-refined vs. agglomerative-only
#
# Each function returns a long-format DataFrame; `run_ablation_suite` runs all
# three on a sample of test graphs from a chosen dataset and writes a single
# combined CSV to EVAL_CFG.OUT_DIR/ablation.csv.


def ablation_sqrtn(model, dataset, device, n_graphs=20,
                    k_targets=("sqrt", "constant_2", "n_div_3", "n_div_2"),
                    n_perm=50, sa_iter=100):
    """(i) Vary the initial number of groups k_0 in agglomerative phase."""
    records = []
    rng = np.random.RandomState(42)
    sample = rng.choice(len(dataset), size=min(n_graphs, len(dataset)),
                        replace=False)
    for gi in sample:
        data = dataset[int(gi)]
        sg = _make_graph_wrapper(data.x.detach().cpu(),
                                  data.edge_index.detach().cpu())
        n = sg.n_heavy
        # Shapley-Taylor matrix (shared across k_0 sweeps).
        vfunc = ValueFunction(model, sg, device)
        M = compute_shapley_taylor_matrix(vfunc, sg, n_permutations=n_perm,
                                           seed=42)
        for kt in k_targets:
            if kt == "sqrt":
                k0 = max(1, int(math.ceil(math.sqrt(n))))
            elif kt == "constant_2":
                k0 = 2
            elif kt == "n_div_3":
                k0 = max(1, n // 3)
            elif kt == "n_div_2":
                k0 = max(1, n // 2)
            else:
                continue
            # Run agglomerative-with-custom-k inline.
            adj = sg.heavy_adj
            players = sg.heavy_idx
            global_to_pos = {g: p for p, g in enumerate(players)}
            groups = [{h} for h in players]
            while len(groups) > k0:
                best_s, best_p = -1.0, None
                for i in range(len(groups)):
                    for j in range(i + 1, len(groups)):
                        if not any(b in adj.get(a, set())
                                   for a in groups[i] for b in groups[j]):
                            continue
                        aff = sum(abs(M[global_to_pos[a], global_to_pos[b]])
                                  for a in groups[i] for b in groups[j])
                        if aff > best_s:
                            best_s, best_p = aff, (i, j)
                if best_p is None:
                    break
                i, j = best_p
                groups[i] = groups[i] | groups[j]
                groups.pop(j)
            groups = enforce_connected_partition(groups, adj)
            searcher = PartitionSearcher(sg, M, gamma_H=0.4, gamma_S=0.3)
            best_part, best_score = searcher.search(
                init_partition=groups, max_iter=sa_iter, seed=42,
                verbose=False)
            agg = aggregate_group_values(
                M, [sorted(g) for g in best_part], sg.heavy_idx)
            records.append({
                "ablation": "sqrt_n",
                "graph_idx": int(gi),
                "n": n,
                "k_target": kt,
                "k0": k0,
                "n_groups_final": len(best_part),
                "score": float(best_score),
                "total_within": agg["total_within"],
                "total_cross": agg["total_cross"],
            })
    return pd.DataFrame(records)


def ablation_phase1_estimator(model, dataset, device, n_graphs=20,
                              estimators=("shapley_taylor", "grad_x_input",
                                          "uniform")):
    """(ii) Phase 1 quality: how does the interaction matrix source affect
    downstream H-Fidelity?"""
    records = []
    rng = np.random.RandomState(42)
    sample = rng.choice(len(dataset), size=min(n_graphs, len(dataset)),
                        replace=False)
    for gi in sample:
        data = dataset[int(gi)]
        sg = _make_graph_wrapper(data.x.detach().cpu(),
                                  data.edge_index.detach().cpu())
        n = sg.n_heavy
        for est in estimators:
            if est == "shapley_taylor":
                vfunc = ValueFunction(model, sg, device)
                M = compute_shapley_taylor_matrix(vfunc, sg,
                                                  n_permutations=50, seed=42)
            elif est == "grad_x_input":
                # Diagonal := Grad x Input scores; off-diagonal := zero.
                # This isolates the value of *pairwise interactions* in our
                # downstream pipeline; if SA still wins, the interactions
                # carry signal beyond the main effects.
                gxi = run_grad_x_input(model, data, device)
                if gxi is None:
                    continue
                M = np.zeros((n, n))
                np.fill_diagonal(M, gxi[:n])
            elif est == "uniform":
                # Sanity-check baseline: all-ones matrix.
                M = np.ones((n, n))
                np.fill_diagonal(M, 1.0)
            else:
                continue
            init_part = agglomerative_partition(sg, M)
            init_part = enforce_connected_partition(init_part, sg.heavy_adj)
            searcher = PartitionSearcher(sg, M, gamma_H=0.4, gamma_S=0.3)
            best_part, best_score = searcher.search(
                init_partition=init_part, max_iter=100, seed=42, verbose=False)
            # Evaluate top-50% nodes with our fidelity.
            best_lists = [sorted(g) for g in best_part]
            agg = aggregate_group_values(M, best_lists, sg.heavy_idx)
            order = sorted(range(len(best_lists)),
                           key=lambda i: abs(agg["group_values"][i]),
                           reverse=True)
            budget = max(1, n // 2)
            cumul = set()
            for gi_sel in order:
                cumul = cumul | set(best_lists[gi_sel])
                if len(cumul) >= budget:
                    break
            fid = _eval_fidelity(model, data, cumul, device)
            records.append({
                "ablation": "phase1_estimator",
                "graph_idx": int(gi),
                "estimator": est,
                "best_score": float(best_score),
                "fid_plus": fid["fid_plus"],
                "fid_minus": fid["fid_minus"],
                "harmonic_fidelity": fid["harmonic_fidelity"],
            })
    return pd.DataFrame(records)


def ablation_phase3_sa(model, dataset, device, n_graphs=20,
                       sa_iter_list=(0, 25, 100, 250)):
    """(iii) Phase 3 importance: SA-refined vs. agglomerative-only.
    `sa_iter=0` means no SA refinement (use the agglomerative output directly)."""
    records = []
    rng = np.random.RandomState(42)
    sample = rng.choice(len(dataset), size=min(n_graphs, len(dataset)),
                        replace=False)
    for gi in sample:
        data = dataset[int(gi)]
        sg = _make_graph_wrapper(data.x.detach().cpu(),
                                  data.edge_index.detach().cpu())
        n = sg.n_heavy
        vfunc = ValueFunction(model, sg, device)
        M = compute_shapley_taylor_matrix(vfunc, sg, n_permutations=50, seed=42)
        init_part = agglomerative_partition(sg, M)
        init_part = enforce_connected_partition(init_part, sg.heavy_adj)
        for sa_iter in sa_iter_list:
            if sa_iter == 0:
                best_part = init_part
                best_score = PartitionSearcher(sg, M)._evaluate(best_part)
            else:
                searcher = PartitionSearcher(sg, M, gamma_H=0.4, gamma_S=0.3)
                best_part, best_score = searcher.search(
                    init_partition=init_part, max_iter=sa_iter, seed=42,
                    verbose=False)
            best_lists = [sorted(g) for g in best_part]
            agg = aggregate_group_values(M, best_lists, sg.heavy_idx)
            order = sorted(range(len(best_lists)),
                           key=lambda i: abs(agg["group_values"][i]),
                           reverse=True)
            cumul = set()
            for gi_sel in order:
                cumul = cumul | set(best_lists[gi_sel])
                if len(cumul) >= max(1, n // 2):
                    break
            fid = _eval_fidelity(model, data, cumul, device)
            records.append({
                "ablation": "phase3_sa",
                "graph_idx": int(gi),
                "sa_iter": sa_iter,
                "best_score": float(best_score),
                "n_groups": len(best_part),
                "fid_plus": fid["fid_plus"],
                "harmonic_fidelity": fid["harmonic_fidelity"],
            })
    return pd.DataFrame(records)


def run_ablation_suite(dataset_name="ba_2motifs", n_graphs=20, model_class="gcn"):
    """End-to-end ablation: load dataset, train a small model, run the three
    ablation experiments, write a combined CSV.

    Returns the combined DataFrame.
    """
    print(f"\n{'='*70}\nABLATION SUITE on {dataset_name}\n{'='*70}")
    dataset, info = load_dig_dataset(dataset_name)
    splits = split_dataset(dataset, seed=42)
    input_dim = info["input_dim"]
    num_classes = info.get("num_classes", 2)
    if model_class == "gin":
        model = BenchmarkGIN(input_dim, EVAL_CFG.HIDDEN, num_classes)
    else:
        model = BenchmarkGCN(input_dim, EVAL_CFG.HIDDEN, num_classes)
    model, _ = train_benchmark_model(
        model, splits["train"], splits["val"], dataset_name, EVAL_CFG.DEVICE,
        epochs=EVAL_CFG.EPOCHS, lr=EVAL_CFG.LR,
        batch_size=EVAL_CFG.BATCH_SIZE, patience=EVAL_CFG.PATIENCE)
    test_data = splits["test"]

    df_a = ablation_sqrtn(model, test_data, EVAL_CFG.DEVICE, n_graphs=n_graphs)
    df_b = ablation_phase1_estimator(model, test_data, EVAL_CFG.DEVICE,
                                      n_graphs=n_graphs)
    df_c = ablation_phase3_sa(model, test_data, EVAL_CFG.DEVICE,
                              n_graphs=n_graphs)
    combined = pd.concat([df_a, df_b, df_c], ignore_index=True)
    out_path = os.path.join(EVAL_CFG.OUT_DIR, f"ablation_{dataset_name}.csv")
    combined.to_csv(out_path, index=False)
    print(f"  Wrote {out_path}: {len(combined)} rows")

    # Quick summary
    print("\n  --- sqrt(n) sensitivity (mean harmonic_fidelity per k_target) ---")
    if "k_target" in df_a.columns and not df_a.empty:
        print(df_a.groupby("k_target")["score"].agg(["mean", "std"]).to_string())
    print("\n  --- Phase 1 estimator (mean H-Fid per estimator) ---")
    if "estimator" in df_b.columns and not df_b.empty:
        print(df_b.groupby("estimator")["harmonic_fidelity"].agg(["mean", "std"]).to_string())
    print("\n  --- Phase 3 SA importance (mean H-Fid per sa_iter) ---")
    if "sa_iter" in df_c.columns and not df_c.empty:
        print(df_c.groupby("sa_iter")["harmonic_fidelity"].agg(["mean", "std"]).to_string())
    return combined


def format_meanstd(values, fmt="{:.3f} ± {:.3f}"):
    """Helper: format a sequence of floats as 'mean +/- std' (Reviewer R-3.E).

    Used in result-summary cells; pass df[col].values to get a publication-ready
    string. Returns 'n/a' for empty inputs.
    """
    arr = np.asarray(list(values), dtype=np.float64)
    arr = arr[~np.isnan(arr)]
    if arr.size == 0:
        return "n/a"
    return fmt.format(float(arr.mean()), float(arr.std()))


if __name__ == "__main__":
    # Uncomment to run the ablation suite manually:
    # _ab = run_ablation_suite("ba_2motifs", n_graphs=10)
    pass


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# §8  COMPARISON TABLE BUILDERS
# ═══════════════════════════════════════════════════════════════════════════

def build_faithfulness_table(results_df: pd.DataFrame) -> pd.DataFrame:
    """Build Table C: Faithfulness metrics across methods and datasets.

    Aggregates per sparsity level: mean fid+, fid-, H-fidelity.
    """
    if results_df.empty:
        return pd.DataFrame()

    rows = []
    for (dataset, method, sp), grp in results_df.groupby(
            ["dataset", "method", "target_sparsity"]):
        row = {
            "dataset": dataset,
            "method": method,
            "sparsity": sp,
            "fid_plus_mean": grp["fid_plus"].mean(),
            "fid_plus_std": grp["fid_plus"].std(),
            "fid_minus_mean": grp["fid_minus"].mean(),
            "fid_minus_std": grp["fid_minus"].std(),
            "harmonic_mean": grp["harmonic_fidelity"].mean(),
            "harmonic_std": grp["harmonic_fidelity"].std(),
            "n": len(grp),
        }
        if "prob_fid_plus" in grp.columns:
            row["prob_fid_plus_mean"] = grp["prob_fid_plus"].mean()
            row["prob_fid_minus_mean"] = grp["prob_fid_minus"].mean()
        rows.append(row)
    return pd.DataFrame(rows)


def build_localization_table(results_df: pd.DataFrame) -> pd.DataFrame:
    """Build Table B: Localization and ground-truth recovery."""
    if results_df.empty or "localization" not in results_df.columns:
        return pd.DataFrame()

    has_loc = results_df[results_df["localization"].notna()]
    if has_loc.empty:
        return pd.DataFrame()

    rows = []
    for (dataset, method), grp in has_loc.groupby(["dataset", "method"]):
        rec = {
            "dataset": dataset, "method": method,
            "localization_mean": grp["localization"].mean(),
            "localization_std": grp["localization"].std(),
            "n": len(grp),
        }
        if "gt_precision" in grp.columns:
            rec["gt_precision_mean"] = grp["gt_precision"].mean()
            rec["gt_recall_mean"] = grp["gt_recall"].mean()
        if "gt_f1" in grp.columns:
            rec["gt_f1_mean"] = grp["gt_f1"].mean()
        rows.append(rec)
    return pd.DataFrame(rows)


def plot_fidelity_vs_sparsity(results_df: pd.DataFrame, save_dir: str):
    """Plot publication-quality Fidelity+ and Fidelity- vs sparsity curves.

    Generates:
      - Per-dataset PNG (3 panels: Fid+, Fid-, H-Fid)
      - Combined PDF for paper (2-column, all datasets)
    """
    if results_df.empty:
        return

    # Style config
    METHOD_STYLE = {
        "ours_groups":   {"color": "#E74C3C", "marker": "s", "ls": "-",  "lw": 2.2, "label": "Ours (groups)"},
        "ours":          {"color": "#3498DB", "marker": "^", "ls": "-",  "lw": 1.8, "label": "Ours (node)"},
        "gnnexplainer":  {"color": "#2ECC71", "marker": "o", "ls": "--", "lw": 1.4, "label": "GNNExplainer"},
        "pgexplainer":   {"color": "#9B59B6", "marker": "D", "ls": "--", "lw": 1.4, "label": "PGExplainer"},
        "subgraphx":     {"color": "#E67E22", "marker": "v", "ls": "--", "lw": 1.4, "label": "SubgraphX"},
        "grad_x_input":  {"color": "#7F8C8D", "marker": "x", "ls": ":",  "lw": 1.4, "label": "Grad×Input"},
        # Reviewer-requested SoTA baselines (R-1.B, R-3 SoTA):
        "mage":          {"color": "#16A085", "marker": "P", "ls": "--", "lw": 1.6, "label": "MAGE"},
        "gstarx":        {"color": "#34495E", "marker": "X", "ls": "--", "lw": 1.6, "label": "GStarX"},
        "graphshapiq":   {"color": "#8E44AD", "marker": "*", "ls": "--", "lw": 1.6, "label": "GraphSHAP-IQ"},
        "sme":           {"color": "#D4A017", "marker": "h", "ls": "--", "lw": 1.4, "label": "SME"},
    }
    DATASET_TITLE = {
        "ba2motifs": "BA-2Motifs",
        "mutag": "MUTAG",
        "graph_sst2": "Graph-SST2",
        "bbbp": "BBBP",
    }

    datasets = sorted(results_df["dataset"].unique(),
                       key=lambda d: list(DATASET_TITLE.keys()).index(d)
                       if d in DATASET_TITLE else 99)

    # ── Per-dataset PNG (for debugging) ──────────────────────────────
    for dataset in datasets:
        ds_df = results_df[results_df["dataset"] == dataset]
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        for method in sorted(ds_df["method"].unique()):
            m_df = ds_df[ds_df["method"] == method]
            sp_means = m_df.groupby("target_sparsity").agg(
                fid_plus=("fid_plus", "mean"),
                fid_minus=("fid_minus", "mean"),
                harmonic=("harmonic_fidelity", "mean")).reset_index()
            if sp_means.empty:
                continue
            x = sp_means["target_sparsity"]
            sty = METHOD_STYLE.get(method, {"color": "gray", "marker": ".", "ls": "-", "lw": 1, "label": method})
            for ax_idx, (col, title) in enumerate([("fid_plus", "Fid+"), ("fid_minus", "Fid-"), ("harmonic", "H-Fid")]):
                axes[ax_idx].plot(x, sp_means[col], marker=sty["marker"], ls=sty["ls"],
                                  lw=sty["lw"], color=sty["color"], label=sty["label"], markersize=5)

        for ax, title in zip(axes, ["Fid+ (necessity)", "Fid- (sufficiency)", "H-Fidelity"]):
            ax.set_xlabel("Sparsity"); ax.set_ylabel(title)
            ax.set_title(f"{DATASET_TITLE.get(dataset, dataset)}: {title}")
            ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
        plt.tight_layout()
        path = os.path.join(save_dir, f"fidelity_vs_sparsity_{dataset}.png")
        plt.savefig(path, dpi=150); plt.close()
        print(f"  Saved {path}")

    # ── Combined paper figure (PDF) ──────────────────────────────────
    n_ds = len(datasets)
    if n_ds == 0:
        return
    fig, axes = plt.subplots(n_ds, 2, figsize=(7.0, 2.5 * n_ds), squeeze=False)
    plt.rcParams.update({"font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9})

    for row, dataset in enumerate(datasets):
        ds_df = results_df[results_df["dataset"] == dataset]
        ds_title = DATASET_TITLE.get(dataset, dataset)

        for method in ["ours_groups", "ours", "gnnexplainer", "pgexplainer",
                        "subgraphx", "grad_x_input",
                        "mage", "gstarx", "graphshapiq", "sme"]:
            m_df = ds_df[ds_df["method"] == method]
            if m_df.empty:
                continue
            sp_means = m_df.groupby("target_sparsity").agg(
                fp=("prob_fid_plus", "mean"),
                fm=("prob_fid_minus", "mean")).reset_index()
            if sp_means.empty:
                continue
            sty = METHOD_STYLE.get(method, {"color": "gray", "marker": ".", "ls": "-", "lw": 1, "label": method})
            x = sp_means["target_sparsity"]
            axes[row, 0].plot(x, sp_means["fp"], marker=sty["marker"], ls=sty["ls"],
                              lw=sty["lw"], color=sty["color"], label=sty["label"], markersize=4)
            axes[row, 1].plot(x, sp_means["fm"], marker=sty["marker"], ls=sty["ls"],
                              lw=sty["lw"], color=sty["color"], label=sty["label"], markersize=4)

        axes[row, 0].set_ylabel(f"{ds_title}\nFid+ (necessity)")
        axes[row, 1].set_ylabel("Fid- (sufficiency)")
        for col in range(2):
            axes[row, col].grid(True, alpha=0.2, ls="--")
            axes[row, col].set_xlim(0.45, 0.95)
            if row == n_ds - 1:
                axes[row, col].set_xlabel("Sparsity")
            else:
                axes[row, col].set_xticklabels([])

    # Single legend at top
    handles, labels = axes[0, 0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="upper center", ncol=min(len(handles), 6),
                   fontsize=8, frameon=False, bbox_to_anchor=(0.5, 1.02))

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    pdf_path = os.path.join(save_dir, "fidelity_vs_sparsity_all.pdf")
    plt.savefig(pdf_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"  Saved {pdf_path}")

    # Also save PNG version
    fig2, axes2 = plt.subplots(n_ds, 2, figsize=(7.0, 2.5 * n_ds), squeeze=False)
    for row, dataset in enumerate(datasets):
        ds_df = results_df[results_df["dataset"] == dataset]
        ds_title = DATASET_TITLE.get(dataset, dataset)
        for method in ["ours_groups", "ours", "gnnexplainer", "pgexplainer",
                        "subgraphx", "grad_x_input",
                        "mage", "gstarx", "graphshapiq", "sme"]:
            m_df = ds_df[ds_df["method"] == method]
            if m_df.empty:
                continue
            sp_means = m_df.groupby("target_sparsity").agg(
                fp=("prob_fid_plus", "mean"),
                fm=("prob_fid_minus", "mean")).reset_index()
            if sp_means.empty:
                continue
            sty = METHOD_STYLE.get(method, {"color": "gray", "marker": ".", "ls": "-", "lw": 1, "label": method})
            x = sp_means["target_sparsity"]
            axes2[row, 0].plot(x, sp_means["fp"], marker=sty["marker"], ls=sty["ls"],
                               lw=sty["lw"], color=sty["color"], label=sty["label"], markersize=4)
            axes2[row, 1].plot(x, sp_means["fm"], marker=sty["marker"], ls=sty["ls"],
                               lw=sty["lw"], color=sty["color"], label=sty["label"], markersize=4)
        axes2[row, 0].set_ylabel(f"{ds_title}\nFid+ (necessity)")
        axes2[row, 1].set_ylabel("Fid- (sufficiency)")
        for col in range(2):
            axes2[row, col].grid(True, alpha=0.2, ls="--")
            axes2[row, col].set_xlim(0.45, 0.95)
            if row == n_ds - 1:
                axes2[row, col].set_xlabel("Sparsity")
            else:
                axes2[row, col].set_xticklabels([])
    handles2, labels2 = axes2[0, 0].get_legend_handles_labels()
    if handles2:
        fig2.legend(handles2, labels2, loc="upper center", ncol=min(len(handles2), 6),
                    fontsize=8, frameon=False, bbox_to_anchor=(0.5, 1.02))
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    png_path = os.path.join(save_dir, "fidelity_vs_sparsity_all.png")
    plt.savefig(png_path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"  Saved {png_path}")


# ═══════════════════════════════════════════════════════════════════════════
# §9  MAIN EVALUATION ENTRY POINT
# ═══════════════════════════════════════════════════════════════════════════

def run_all_benchmarks():
    """Run evaluation on all DIG benchmark datasets.

    Produces:
      - faithfulness_comparison.csv
      - localization_comparison.csv
      - fidelity_vs_sparsity_*.png
      - benchmark_summary.csv
    """
    print("\n" + "=" * 70)
    print("PUBLIC BENCHMARK EVALUATION (DIG)")
    print("=" * 70)

    device = EVAL_CFG.DEVICE
    all_results = []
    all_summaries = []

    # Define benchmark suite
    benchmarks = [
        ("mutag",         "gcn"),  # Real molecular, proxy GT
        ("ba_2motifs",    "gcn"),  # Structural reasoning
        ("bbbp",          "gcn"),  # Molecular, MoleculeNet
        # New (reviewer-requested) — uncomment to include in the sweep:
        # ("ba_2motifs",   "gcn"),  # Synthetic with edge ground truth
        # ("proteins",     "gcn"),  # TU biological benchmark
        # ("bace",         "gcn"),  # MoleculeNet drug discovery (scaffold split)
        # ("dd",           "gcn"),  # Large protein structures
    ]

    # Optional: uncomment for extended evaluation
    # benchmarks += [
    #     ("graph_sst2", "gcn"),
    #     ("ba2motifs",  "gin"),
    #     ("mutag",      "gin"),
    # ]

    for dataset_name, model_class in benchmarks:
        try:
            df, summary = run_benchmark_evaluation(
                dataset_name, device,
                n_explain=EVAL_CFG.N_EXPLAIN,
                model_class=model_class)
            if not df.empty:
                all_results.append(df)
            all_summaries.append(summary)
        except Exception as e:
            print(f"  FAILED {dataset_name}/{model_class}: {e}")
            import traceback; traceback.print_exc()

    if not all_results:
        print("  No results produced")
        return

    # ── Combine and save ──────────────────────────────────────────────
    combined_df = pd.concat(all_results, ignore_index=True)
    combined_df.to_csv(os.path.join(EVAL_CFG.OUT_DIR,
                                     "benchmark_results_raw.csv"), index=False)
    print(f"\n  benchmark_results_raw.csv: {len(combined_df)} rows")

    # Faithfulness table
    faith_table = build_faithfulness_table(combined_df)
    if not faith_table.empty:
        faith_table.to_csv(os.path.join(EVAL_CFG.OUT_DIR,
                                         "faithfulness_comparison.csv"), index=False)
        print(f"  faithfulness_comparison.csv: {len(faith_table)} rows")

    # Localization table
    loc_table = build_localization_table(combined_df)
    if not loc_table.empty:
        loc_table.to_csv(os.path.join(EVAL_CFG.OUT_DIR,
                                       "localization_comparison.csv"), index=False)
        print(f"  localization_comparison.csv: {len(loc_table)} rows")

    # Plots
    plot_fidelity_vs_sparsity(combined_df, EVAL_CFG.OUT_DIR)

    # Summary
    pd.DataFrame(all_summaries).to_csv(
        os.path.join(EVAL_CFG.OUT_DIR, "benchmark_summary.csv"), index=False)

    # ── Print comparison table ────────────────────────────────────────
    print(f"\n{'='*70}")
    print("FAITHFULNESS COMPARISON (mean across graphs)")
    print(f"{'='*70}")

    if not faith_table.empty:
        for dataset in faith_table["dataset"].unique():
            ds = faith_table[faith_table["dataset"] == dataset]
            print(f"\n  Dataset: {dataset}")
            # Show at sparsity = 0.7
            sp_target = 0.7
            ds_sp = ds[abs(ds["sparsity"] - sp_target) < 0.05]
            if ds_sp.empty:
                ds_sp = ds
            # Prefer probability-space fidelity if available
            fp_col = "prob_fid_plus_mean" if "prob_fid_plus_mean" in ds_sp.columns else "fid_plus_mean"
            fm_col = "prob_fid_minus_mean" if "prob_fid_minus_mean" in ds_sp.columns else "fid_minus_mean"
            # Also show logit-space if available
            has_logit = "fid_plus_mean" in ds_sp.columns and fp_col != "fid_plus_mean"
            hdr = f"  {'Method':<18s} {'pFid+':>8s} {'pFid-':>8s} {'H-Fid':>8s}"
            if has_logit:
                hdr += f" {'logFid+':>9s} {'logFid-':>9s}"
            hdr += f" {'n':>5s}"
            print(hdr)
            print(f"  {'-'*70}")
            for _, row in ds_sp.sort_values("harmonic_mean", ascending=False).iterrows():
                line = (f"  {row['method']:<18s} "
                        f"{row[fp_col]:>8.4f} "
                        f"{row[fm_col]:>8.4f} "
                        f"{row['harmonic_mean']:>8.4f}")
                if has_logit:
                    line += (f" {row['fid_plus_mean']:>9.3f}"
                             f" {row['fid_minus_mean']:>9.3f}")
                line += f" {int(row['n']):>5d}"
                print(line)

    if not loc_table.empty:
        print(f"\n{'='*70}")
        print("LOCALIZATION COMPARISON (datasets with ground truth)")
        print(f"{'='*70}")
        print(f"  {'Method':<18s} {'Dataset':<12s} {'Loc':>8s} {'Prec':>8s} "
              f"{'Recall':>8s} {'F1':>8s}")
        print(f"  {'-'*60}")
        for _, row in loc_table.sort_values(
                ["dataset", "localization_mean"], ascending=[True, False]).iterrows():
            prec = row.get("gt_precision_mean", float("nan"))
            rec = row.get("gt_recall_mean", float("nan"))
            f1 = row.get("gt_f1_mean", float("nan"))
            print(f"  {row['method']:<18s} {row['dataset']:<12s} "
                  f"{row['localization_mean']:>8.3f} "
                  f"{prec:>8.3f} {rec:>8.3f} {f1:>8.3f}")

    print(f"\n{'='*70}")
    print("  Benchmark evaluation complete")
    print(f"{'='*70}")


# ═══════════════════════════════════════════════════════════════════════════
# §10  ENTRY POINT
# ═══════════════════════════════════════════════════════════════════════════


In [ ]:
if __name__ == "__main__":
    run_all_benchmarks()
